# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị

**Pipeline:** `truy vấn → CLIP → top-K ứng viên → điểm đồ thị (GNN + đường đi) → xếp hạng lại → giải thích → fidelity`

Điểm số cuối của một ứng viên:

$$\text{score} = \alpha \cdot \cos_{\text{CLIP}} + (1-\alpha)\big[\lambda \cdot s_{\text{GNN}} + (1-\lambda)\cdot s_{\text{path}}\big]$$

* `s_GNN` — GraphSAGE/GAT tự cài trên đồ thị dị thể (ảnh / lớp đối tượng / khái niệm).
  Mục 7.2 thay nhánh này bằng **SCENIR** (ICML 2025): Graph Autoencoder tự giám sát trên scene graph
  của từng ảnh — phương pháp mới cho cấp độ 2 (`CFG.gnn_type = "scenir"`).
* `s_path` — tổng **cộng tính** theo từng đường đi nối khái niệm truy vấn tới node ảnh.
  Vì cộng tính nên bỏ một đường đi làm điểm giảm đúng bằng phần đóng góp của nó ⇒ giải thích
  đến thẳng từ cơ chế xếp hạng, không phải hợp lý hoá sau khi có kết quả.
* `α`, `λ` chọn trên **validation**, không bao giờ trên test.

**Chạy trên Kaggle:** Settings → Accelerator → GPU P100 (hoặc T4 x2), Internet → On.
Notebook tự dò dữ liệu trong `/kaggle/input`; không thấy thì chạy chế độ `SYNTHETIC` để smoke-test toàn bộ mã.

In [1]:
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
print("Kaggle environment:", RUN_ON_KAGGLE)

Kaggle environment: True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cài đặt phụ thuộc. Trên Kaggle torch/numpy đã có sẵn; chỉ cần open_clip + nltk.
def pip_install(pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

if RUN_ON_KAGGLE:
    pip_install(["open_clip_torch", "nltk"])   # không ghim phiên bản: Kaggle đổi torch thường xuyên

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

torch 2.11.0+cu130 | device: cuda
GPU: NVIDIA L4 | 23.7 GB


## 1. Cấu hình

Mọi siêu tham số nằm ở một chỗ. `CFG.n_images` là núm vặn chính cho thời gian chạy:
4000 ảnh chạy khoảng 12-15 phút trên P100 (gồm cả trích đặc trưng CLIP).

In [4]:
class CFG:
    # --- dữ liệu ---
    dataset      = "flickr8k"
    n_images     = 8000
    min_obj_freq = 5           # lớp đối tượng hiếm hơn ngưỡng này bị loại — ĐẾM TRÊN TRAIN
    max_concepts = 4000

    # --- nhãn ảnh Flickr8k ---
    # "tagger": mọi ảnh gắn nhãn bằng CLIP zero-shot trên từ vựng rút từ caption TRAIN
    #           (không ảnh val/test nào được dùng caption của chính nó).
    # "oracle": nhãn rút từ caption 1..4 của chính ảnh (cách cũ) — chỉ để so sánh, KHÔNG báo cáo là kết quả chính.
    flickr_annot = "tagger"
    tag_k_grid   = [3, 5, 8, 10, 12]          # số nhãn/ảnh — chọn trên VAL theo F1
    tag_prompts  = ["a photo of a {}.", "a photo of {}.", "a photo containing a {}."]

    # --- chia dữ liệu (chia theo ẢNH, không theo caption) ---
    split_seed   = 20240101
    val_ratio    = 0.15
    test_ratio   = 0.20

    # --- CLIP ---
    clip_model   = "ViT-L-14"                 # trước: ViT-B-32. Tagger + baseline mạnh hơn rõ rệt
    clip_weights = "laion2b_s32b_b82k"
    clip_bs      = 128                        # ViT-L tốn bộ nhớ hơn; L4 24GB chạy 128 ổn

    # --- đồ thị ---
    max_path_len      = 3
    scene_min_count   = 2      # cạnh scene phải xuất hiện ≥ ngưỡng này trong caption TRAIN (trước đây = 1)
    scene_top_k       = 10     # mỗi lớp đối tượng giữ tối đa k cạnh scene mạnh nhất
    cooc_min_count    = 10
    cooc_top_k        = 10
    use_wordnet       = True
    isa_min_depth     = 4      # bỏ khái niệm quá chung (entity, object, whole, ...)
    use_idf           = True   # trọng số đường đi = IDF của node đích (tính trên TRAIN)
    drop_redundant_indirect = True   # bỏ đường đi gián tiếp tới khái niệm ảnh đã chứa trực tiếp

    # --- sửa ablation âm ---
    # (1) IDF: "taxonomy" = đối tượng đếm df qua P0, khái niệm qua P2 (isa thật, trước khi xáo).
    #     "all" = cách cũ: đếm qua MỌI mẫu ⇒ thêm quan hệ làm df phình, IDF của khớp trực tiếp
    #     tụt về ≈0 — đây là lý do chính khiến 'none' thua 'no_relations'.
    idf_from        = "taxonomy"
    # (2) Cạnh related: xếp hạng theo NPMI thay vì đếm thô (đếm thô ⇒ mọi lớp nối vào các hub
    #     phổ biến như person/man/floor). Trọng số cạnh = P(đích | nguồn) trên TRAIN.
    cooc_score      = "npmi"         # "npmi" | "count" (cũ)
    cooc_min_npmi   = 0.10
    edge_weighting  = True           # trọng số đường đi = idf(c) × Π P(đích|nguồn); False = đếm (cũ)
    # (3) GNN tách theo loại cạnh + cổng học được, khởi tạo gần như đóng ⇒ mô hình bắt đầu
    #     từ 'no_relations' và chỉ mở quan hệ khi nó thực sự giảm loss.
    gnn_typed       = True
    gate_init       = -1.5           # sigmoid(-1.5) ≈ 0.18 cho cạnh scene/related/isa; contains mở sẵn
    path_init_indirect = -1.0        # path_w khởi tạo cho P1..P5 (softplus(-1) ≈ 0.31; P0 = 0.69)
    scalar_lr_mult  = 10.0           # lr của cổng + path_w = lr × hệ số này
    # (4) Huấn luyện giống lúc suy luận: mẫu âm khó lấy từ CLIP top-K trên gallery TRAIN,
    #     và có z(s_clip) trong logit ⇒ đồ thị học phần CLIP còn thiếu, không học lại CLIP.
    hard_neg_k      = 31             # 0 = tắt (chỉ dùng âm trong batch như cũ)
    train_with_clip = True
    # Hệ số của z(s_clip) trong logit HUẤN LUYỆN có học không. AG (CLIP yếu, MRR 0.12): học được
    # thì tốt hơn (clip_w → 0.07). Flickr (CLIP mạnh): học làm clip_w → 0.4 và MRR giảm.
    # CHỌN THEO VAL MRR cho từng bộ dữ liệu, không theo test.
    learn_clip_w    = True

    # --- GNN ---
    gnn_type     = "sage"
    gnn_dim      = 256
    gnn_layers   = 2
    gnn_dropout  = 0.1
    lr           = 1e-3
    weight_decay = 1e-4
    epochs       = 20
    train_bs     = 128
    temperature  = 0.07

    # --- truy vấn / xếp hạng ---
    rerank_k     = 100

    # --- thí nghiệm ---
    seeds        = [0, 1, 2]
    alpha_grid   = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid  = [0.0, 0.25, 0.5, 0.75]     # bỏ 1.0: λ=1 ⇒ s_path có trọng số 0 ⇒ giải thích không trung thực với điểm cuối
    ablations    = ("none", "random_edges", "no_relations", "no_scene", "no_related", "no_concept")

    adaptive_fusion = True
    fusion_steps    = 300
    fusion_tau      = 0.2
    fusion_wd       = 1e-2

    # --- xAI ---
    n_explain_queries = 20
    top_m_paths       = 5

    # --- đường dẫn ---
    out_dir = Path("/kaggle/working") if RUN_ON_KAGGLE else Path("./work")

CFG.out_dir.mkdir(parents=True, exist_ok=True)
(CFG.out_dir / "cache").mkdir(exist_ok=True)
(CFG.out_dir / "results").mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 2. Nạp dữ liệu

Notebook dò tự động các tệp Visual Genome trong `/kaggle/input` theo tên
(`objects.json`, `relationships.json`, `image_data.json`, `region_descriptions.json`).
Không tìm thấy ⇒ tự sinh dữ liệu tổng hợp để toàn bộ mã vẫn chạy hết (smoke-test).

**Cấu trúc chuẩn hoá** dùng chung cho mọi nguồn:
```python
images = [{"image_id": int, "caption": str, "objects": [str], "relations": [(subj, pred, obj)]}]
```

In [5]:
# Lớp khái niệm dự phòng: dùng khi WordNet/nltk không tải được (Kaggle tắt Internet).
# Đây là một MMKG tí hon viết tay, đủ để tạo đường đi gián tiếp cho smoke-test và
# vẫn hoạt động trên Visual Genome vì các lớp đối tượng phổ biến đều có mặt.
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
}


def find_vg_files():
    """Dò các tệp Visual Genome trong /kaggle/input theo tên tệp."""
    wanted = ["objects.json", "relationships.json", "image_data.json",
              "region_descriptions.json", "attributes.json"]
    found = {}
    if not Path("/kaggle/input").exists():
        return found
    for name in wanted:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            found[name.replace(".json", "")] = sorted(hits, key=lambda p: -os.path.getsize(p))[0]
    return found


def _norm(s):
    """Chuẩn hoá chuỗi nhãn: chữ thường, bỏ ký tự lạ, gộp khoảng trắng."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_visual_genome(paths, n_images):
    """Nạp VG. Trả về danh sách dict đã chuẩn hoá."""
    print("Nạp Visual Genome từ:", {k: v for k, v in paths.items()})
    with open(paths["objects"]) as f:
        objects = json.load(f)
    obj_by_img, objid2name = {}, {}
    for rec in objects:
        iid = rec.get("image_id") or rec.get("id")
        names = []
        for o in rec.get("objects", []):
            nm = _norm((o.get("names") or [o.get("name", "")])[0])
            if nm:
                names.append(nm)
                objid2name[o.get("object_id")] = nm
        if names:
            obj_by_img[iid] = names
    del objects

    rel_by_img = defaultdict(list)
    if "relationships" in paths:
        with open(paths["relationships"]) as f:
            rels = json.load(f)
        for rec in rels:
            iid = rec.get("image_id") or rec.get("id")
            for r in rec.get("relationships", []):
                pred = _norm(r.get("predicate", ""))
                s = r.get("subject", {}); o = r.get("object", {})
                sn = _norm((s.get("names") or [s.get("name", "")])[0]) or objid2name.get(s.get("object_id"), "")
                on = _norm((o.get("names") or [o.get("name", "")])[0]) or objid2name.get(o.get("object_id"), "")
                if pred and sn and on:
                    rel_by_img[iid].append((sn, pred, on))
        del rels

    cap_by_img = {}
    if "region_descriptions" in paths:
        with open(paths["region_descriptions"]) as f:
            regions = json.load(f)
        for rec in regions:
            iid = rec.get("id") or rec.get("image_id")
            phrases = [r.get("phrase", "").strip() for r in rec.get("regions", [])]
            phrases = [p for p in phrases if 15 <= len(p) <= 160]
            if phrases:
                # ghép 3 vùng dài nhất thành một caption cấp ảnh
                phrases = sorted(phrases, key=len, reverse=True)[:3]
                cap_by_img[iid] = ". ".join(phrases)
        del regions

    url_by_img = {}
    if "image_data" in paths:
        with open(paths["image_data"]) as f:
            meta = json.load(f)
        for m in meta:
            url_by_img[m.get("image_id") or m.get("id")] = m.get("url", "")
        del meta

    ids = [i for i in obj_by_img if i in cap_by_img] or list(obj_by_img)
    ids = sorted(ids)[:n_images]
    out = []
    for iid in ids:
        out.append(dict(image_id=int(iid),
                        caption=cap_by_img.get(iid, " ".join(obj_by_img[iid][:8])),
                        objects=sorted(set(obj_by_img[iid])),
                        relations=list({tuple(r) for r in rel_by_img.get(iid, [])}),
                        url=url_by_img.get(iid, "")))
    print(f"  → {len(out)} ảnh, "
          f"{np.mean([len(r['objects']) for r in out]):.1f} đối tượng/ảnh, "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/ảnh")
    return out


def make_synthetic(n_images, seed=0):
    """Dữ liệu tổng hợp có cấu trúc quan hệ thật, dùng để smoke-test toàn bộ pipeline.

    Thiết kế có chủ ý: mỗi ảnh thuộc một 'chủ đề', chủ đề quyết định đối tượng và
    quan hệ, nên đồ thị MANG thông tin — nếu pipeline đúng thì điểm đồ thị phải
    cải thiện được kết quả so với đặc trưng nhiễu.
    """
    rng = random.Random(seed)
    themes = {
        "football":  (["ball", "field", "child", "goal", "grass", "shoe"],
                      [("child", "plays", "ball"), ("ball", "on", "grass"), ("child", "wears", "shoe")]),
        "kitchen":   (["pot", "stove", "woman", "plate", "spoon", "table"],
                      [("woman", "cooks with", "pot"), ("pot", "on", "stove"), ("plate", "on", "table")]),
        "beach":     (["sand", "umbrella", "wave", "man", "towel", "boat"],
                      [("man", "sits on", "towel"), ("umbrella", "on", "sand"), ("boat", "in", "wave")]),
        "street":    (["car", "sign", "road", "building", "light", "person"],
                      [("car", "on", "road"), ("sign", "near", "light"), ("person", "crosses", "road")]),
        "zoo":       (["elephant", "fence", "tree", "keeper", "water", "rock"],
                      [("elephant", "near", "fence"), ("tree", "behind", "fence"), ("keeper", "feeds", "elephant")]),
        "bedroom":   (["bed", "lamp", "pillow", "window", "curtain", "book"],
                      [("lamp", "next to", "bed"), ("pillow", "on", "bed"), ("curtain", "on", "window")]),
    }
    names = list(themes)
    out = []
    for i in range(n_images):
        th = names[i % len(names)]
        objs, rels = themes[th]
        k = rng.randint(3, len(objs))
        chosen = rng.sample(objs, k)
        r = [t for t in rels if t[0] in chosen and t[2] in chosen]
        # nhiễu: thỉnh thoảng mượn một đối tượng của chủ đề khác
        if rng.random() < 0.3:
            other = themes[rng.choice(names)][0]
            chosen.append(rng.choice(other))
        # 40% truy vấn nhắc tới KHÁI NIỆM (không có trong danh sách đối tượng) thay vì
        # gọi thẳng tên đối tượng ⇒ tạo ra nhóm truy vấn gián tiếp cần đường đi P2/P4/P5.
        if rng.random() < 0.4:
            concepts = sorted({c for o in chosen for c in FALLBACK_ISA.get(o, [])}
                              - set(chosen))
            if concepts:
                cap = (f"a photo showing {' and '.join(rng.sample(concepts, min(2, len(concepts))))}"
                       f" in a {th} scene")
            else:
                cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        else:
            cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        out.append(dict(image_id=i, caption=cap, objects=sorted(set(chosen)),
                        relations=r, url="", theme=th))
    return out


def load_dataset():
    mode = CFG.dataset
    if mode in ("auto", "vg"):
        paths = find_vg_files()
        if "objects" in paths:
            return load_visual_genome(paths, CFG.n_images), "visual_genome"
        if mode == "vg":
            raise FileNotFoundError("Không tìm thấy objects.json của Visual Genome trong /kaggle/input")
    print("⚠ Không tìm thấy dữ liệu công khai → chạy chế độ SYNTHETIC (smoke-test).")
    return make_synthetic(CFG.n_images), "synthetic"

## 3. Chia dữ liệu và phòng rò rỉ

Yêu cầu mục 6 của đề, cài đặt cụ thể:

| Quy định | Cách thực hiện trong mã |
|---|---|
| Test không dùng để huấn luyện / chọn siêu tham số | Chia **theo ảnh**; caption của test không vào tập huấn luyện |
| Cạnh suy ra từ thống kê phải xây từ train | `build_graph(..., stat_image_ids=train_ids)`; cạnh đồng xuất hiện chỉ đếm trên train |
| α và siêu tham số chọn trên validation | `tune_on_val()` quét lưới trên val, test chỉ chạy một lần |
| Nộp split cố định hoặc seed + mã tạo split | `split.json` được ghi ra kèm seed và hash |

Node ảnh của test **vẫn nằm trong đồ thị** (chúng là ứng viên cần xếp hạng), nhưng
cạnh của chúng chỉ đến từ annotation của chính bức ảnh, không từ thống kê toàn cục,
và không có gradient nào chảy qua chúng khi huấn luyện.

In [6]:
def make_split(records, seed=CFG.split_seed):
    ids = sorted(r["image_id"] for r in records)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_test = int(n * CFG.test_ratio)
    n_val  = int(n * CFG.val_ratio)
    split = dict(test=sorted(ids[:n_test]),
                 val=sorted(ids[n_test:n_test + n_val]),
                 train=sorted(ids[n_test + n_val:]),
                 seed=seed)
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split (seed={seed}, sha1={split['sha1']}): "
          f"train={len(split['train'])} val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])), "train/test chồng lấn!"
    assert not (set(split["val"]) & set(split["test"])), "val/test chồng lấn!"
    return split

## 4. Đặc trưng CLIP (đóng băng)

CLIP được **đóng băng hoàn toàn** và chỉ chạy một lần: đặc trưng lưu ra `.npy` rồi tái sử dụng
cho mọi thí nghiệm. Đây là baseline của Nhiệm vụ 1.

Thứ tự ưu tiên tìm ảnh: tệp ảnh trong `/kaggle/input` → tải theo URL → đặc trưng tổng hợp.

In [7]:
def index_local_images():
    """Lập chỉ mục mọi tệp ảnh trong /kaggle/input theo tên tệp (không đuôi)."""
    idx = {}
    roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    for root in roots:
        for p in root.rglob("*"):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                idx.setdefault(p.stem, str(p))
    return idx


def encode_with_clip(records, tag):
    """Trả về (img_feat, txt_feat) đã chuẩn hoá L2, shape [N, D]."""
    cache = CFG.out_dir / "cache" / f"clip_{tag}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache)
        print(f"Dùng lại đặc trưng CLIP từ cache: {cache.name}")
        return z["img"], z["txt"]

    try:
        import open_clip
        from PIL import Image
    except ImportError:
        open_clip = None

    img_index = index_local_images()
    have_images = sum(1 for r in records if str(r["image_id"]) in img_index)
    print(f"Ảnh tìm thấy cục bộ: {have_images}/{len(records)}")

    if open_clip is None or have_images < 0.5 * len(records):
        print("⚠ Không đủ ảnh thật hoặc thiếu open_clip → sinh đặc trưng tổng hợp "
              "(vẫn chạy hết pipeline, nhưng con số không dùng để báo cáo).")
        return synthetic_features(records)

    model, _, preprocess = open_clip.create_model_and_transforms(
        CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters():          # đóng băng tuyệt đối
        p.requires_grad_(False)

    # --- ảnh ---
    feats = []
    from torch.utils.data import Dataset, DataLoader

    class ImgDS(Dataset):
        def __len__(self): return len(records)
        def __getitem__(self, i):
            p = img_index.get(str(records[i]["image_id"]))
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    dl = DataLoader(ImgDS(), batch_size=CFG.clip_bs, num_workers=2)
    with torch.no_grad():
        for batch in dl:
            f = model.encode_image(batch.to(DEVICE))
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    img_feat = np.concatenate(feats)

    # --- văn bản ---
    feats = []
    caps = [r["caption"] for r in records]
    with torch.no_grad():
        for i in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[i:i + CFG.clip_bs]).to(DEVICE)
            f = model.encode_text(tok)
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    txt_feat = np.concatenate(feats)

    del model
    torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img_feat, txt=txt_feat)
    return img_feat, txt_feat


def synthetic_features(records, dim=512, seed=0):
    """Đặc trưng giả có tín hiệu thật: ảnh cùng chủ đề/đối tượng thì gần nhau."""
    rng = np.random.RandomState(seed)
    vocab = sorted({o for r in records for o in r["objects"]})
    basis = rng.randn(len(vocab), dim).astype(np.float32)
    v2i = {v: i for i, v in enumerate(vocab)}
    img, txt = [], []
    for r in records:
        idx = [v2i[o] for o in r["objects"]]
        core = basis[idx].mean(0) if idx else rng.randn(dim).astype(np.float32)
        img.append(core + 0.9 * rng.randn(dim).astype(np.float32))
        txt.append(core + 0.9 * rng.randn(dim).astype(np.float32))
    f = lambda x: (lambda a: a / np.linalg.norm(a, axis=1, keepdims=True))(np.stack(x))
    return f(img), f(txt)

## 5. Xây đồ thị

**Đỉnh:** ảnh · lớp đối tượng · khái niệm.
Đối tượng là **lớp** (một node `dog` dùng chung mọi ảnh) chứ không phải từng thể hiện —
nhờ vậy đồ thị nối được các ảnh với nhau và message passing mới có tác dụng.

**Cạnh:**

| Loại | Nguồn | Có phải thống kê? |
|---|---|---|
| `contains` ảnh → lớp đối tượng | annotation của chính ảnh đó | không → an toàn cho test |
| `scene` đối tượng → đối tượng | scene graph, gộp trên **train** | có → chỉ train |
| `related` đối tượng → đối tượng | đồng xuất hiện trên **train** | có → chỉ train |
| `isa` đối tượng → khái niệm | WordNet hypernym | không (tri thức ngoài) |

**Mẫu đường đi** dùng cho `s_path` (từ node ảnh tới node khái niệm của truy vấn):

| Mẫu | Đường đi | Dài | Loại |
|---|---|---|---|
| P0 | `img –contains→ c` | 1 | trực tiếp |
| P1 | `img –contains→ o –scene→ c` | 2 | gián tiếp |
| P2 | `img –contains→ o –isa→ c` | 2 | gián tiếp |
| P3 | `img –contains→ o –related→ c` | 2 | gián tiếp |
| P4 | `img –contains→ o –scene→ o₂ –isa→ c` | 3 | gián tiếp |
| P5 | `img –contains→ o –related→ o₂ –isa→ c` | 3 | gián tiếp |

`child plays football` với ảnh chỉ nhận diện được `ball`, `field` được bắt qua P1/P2/P4.

In [8]:
import scipy.sparse as sp

PATTERNS = ["P0_contains", "P1_scene", "P2_isa", "P3_related", "P4_scene_isa", "P5_related_isa"]
PATTERN_LEN = [1, 2, 2, 2, 3, 3]
PIDX = {p: i for i, p in enumerate(PATTERNS)}
PATTERN_DESC = {
    "P0_contains":  "ảnh –chứa→ {c}",
    "P1_scene":     "ảnh –chứa→ {o} –{rel}→ {c}",
    "P2_isa":       "ảnh –chứa→ {o} –là_một→ {c}",
    "P3_related":   "ảnh –chứa→ {o} –liên_quan→ {c}",
    "P4_scene_isa": "ảnh –chứa→ {o} –{rel}→ {o2} –là_một→ {c}",
    "P5_related_isa": "ảnh –chứa→ {o} –liên_quan→ {o2} –là_một→ {c}",
}

# ---------------------------------------------------------------------------------
# Lọc WordNet: chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC.
#   - nghĩa danh từ phổ biến nhất thuộc VISUAL_LEX (hoặc là môn thể thao/trò chơi),
#   - và từ đó được dùng như danh từ ít nhất ngang như động từ (loại stand, shake, jump, walk...).
# Loại được các node rác ở bản trước: pink, blue, beige, snowy, walking, shake, stand, bit...
# ---------------------------------------------------------------------------------
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
# khái niệm "cấp cơ bản" — luôn thêm nếu nằm trên đường hypernym (dog→animal, car→vehicle...)
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment"}

_WN = None


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            try:
                import nltk
                nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
                from nltk.corpus import wordnet as wn
                wn.synsets("dog"); _WN = wn
            except Exception:
                _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE = {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True          # không có WordNet → không lọc được
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":       # football, soccer, tennis... vẫn giữ
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def get_wordnet_hypernyms(words):
    """IsA: cha + ông (đủ sâu, không quá chung) + mọi tổ tiên thuộc BASIC_CATEGORIES.
    Chỉ dùng nghĩa phổ biến nhất ⇒ không còn 'bicycle là_một ride', 'beige là_một brown'."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        fb = {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
        print(f"⚠ WordNet không dùng được → FALLBACK_ISA ({len(fb)}/{len(words)} lớp có khái niệm cha).")
        return fb
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in s.hypernym_paths()[:2]:
            for depth in range(len(path) - 2, -1, -1):        # path[-1] chính là w
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3                   # cha hoặc ông
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                        or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    print(f"WordNet(+fallback): {len(out)}/{len(words)} lớp đối tượng có khái niệm cha")
    return out


class MultimodalGraph:
    """Đồ thị dị thể ảnh–đối tượng–khái niệm, kèm ma trận đường đi (đã nhân IDF) theo mẫu."""

    def __init__(self, records, train_ids, ablation="none", seed=0):
        """ablation: none | no_relations | random_edges | no_scene | no_related | no_concept"""
        self.records = records
        self.ablation = ablation
        self.n_img = len(records)
        self.img_pos = {r["image_id"]: i for i, r in enumerate(records)}
        train_set = set(train_ids)
        rng = np.random.RandomState(seed)

        # ---------- từ vựng lớp đối tượng: tần suất CHỈ trên TRAIN ----------
        freq = Counter(o for r in records if r["image_id"] in train_set for o in r["objects"])
        objs = sorted([o for o, c in freq.items() if c >= CFG.min_obj_freq])
        self.obj2id = {o: self.n_img + i for i, o in enumerate(objs)}
        self.n_obj = len(objs)

        # ---------- khái niệm (WordNet) ----------
        hyper = {} if ablation in ("no_relations", "no_concept") else get_wordnet_hypernyms(objs)
        # Khái niệm trùng tên một lớp đối tượng (vd 'dog' là cha của 'terrier') được nối
        # THẲNG vào node đối tượng đó — bản trước tạo 2 node 'dog' và truy vấn 'dog' chỉ
        # khớp node khái niệm, làm mất đường đi P0.
        concepts = sorted({c for v in hyper.values() for c in v
                           if c not in self.obj2id})[:CFG.max_concepts]
        base = self.n_img + self.n_obj
        self.con2id = {c: base + i for i, c in enumerate(concepts)}
        self.n_con = len(concepts)
        self.n_nodes = self.n_img + self.n_obj + self.n_con

        self.id2name = {}
        for o, i in self.obj2id.items(): self.id2name[i] = o
        for c, i in self.con2id.items(): self.id2name[i] = c

        # ---------- cạnh contains (nhãn riêng của mỗi ảnh) ----------
        rows, cols = [], []
        self.img_objects = []
        for i, r in enumerate(records):
            ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
            self.img_objects.append(ids)
            rows += [i] * len(ids); cols += ids
        A_con = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                              shape=(self.n_img, self.n_nodes))

        # ---------- cạnh scene: CHỈ train, có ngưỡng + top-k ----------
        scene = Counter(); pred_cnt = defaultdict(Counter)
        if ablation not in ("no_relations", "no_scene"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue                      # ← chốt chặn rò rỉ
                for s, p, o in r["relations"]:
                    if s in self.obj2id and o in self.obj2id and s != o:
                        key = (self.obj2id[s], self.obj2id[o])
                        scene[key] += 1; pred_cnt[key][p] += 1
            by_src = defaultdict(list)
            for (a, b), v in scene.items():
                if v >= CFG.scene_min_count:
                    by_src[a].append((v, b))
            scene = {(a, b): v for a, lst in by_src.items()
                     for v, b in sorted(lst, reverse=True)[:CFG.scene_top_k]}
        # vị từ hiển thị = vị từ PHỔ BIẾN NHẤT của cặp (bản trước lấy vị từ gặp đầu tiên)
        self.scene_pred = {k: pred_cnt[k].most_common(1)[0][0] for k in scene}
        # số ảnh TRAIN chứa mỗi lớp — mẫu số cho P(đích | nguồn)
        n_tr = max(sum(1 for r in records if r["image_id"] in train_set), 1)
        obj_df = Counter(self.obj2id[o] for r in records if r["image_id"] in train_set
                         for o in set(r["objects"]) if o in self.obj2id)
        scene_w = {(a, b): min(1.0, v / max(obj_df[a], 1)) for (a, b), v in scene.items()}
        A_scene = self._mat(scene_w)

        # ---------- cạnh related: đồng xuất hiện, CHỈ train ----------
        cooc = Counter()
        if ablation not in ("no_relations", "no_related"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue
                ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
                for a, b in itertools.combinations(ids, 2):
                    cooc[(a, b)] += 1; cooc[(b, a)] += 1
            cooc = {k: v for k, v in cooc.items() if v >= CFG.cooc_min_count}
            by_src = defaultdict(list)
            for (a, b), v in cooc.items():
                if CFG.cooc_score == "npmi":
                    # NPMI ∈ [-1, 1]: 0 = độc lập. Đếm thô ưu ái lớp phổ biến (hub) ⇒ mọi lớp
                    # đều nối tới person/man/floor và P3/P5 thành nhiễu phủ khắp gallery.
                    p_ab = v / n_tr
                    pmi = math.log(p_ab / ((obj_df[a] / n_tr) * (obj_df[b] / n_tr)))
                    s = pmi / max(-math.log(p_ab), 1e-6)
                    if s < CFG.cooc_min_npmi:
                        continue
                else:
                    s = v
                by_src[a].append((s, b))
            cooc = {(a, b): cooc[(a, b)] for a, lst in by_src.items()
                    for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]}
        cooc_w = {(a, b): v / max(obj_df[a], 1) for (a, b), v in cooc.items()}   # P(b | a)
        A_rel = self._mat(cooc_w)

        # ---------- cạnh isa (đích có thể là node khái niệm HOẶC node đối tượng) ----------
        isa = {}
        for o, cs in hyper.items():
            if o not in self.obj2id:
                continue
            for c in cs:
                t = self.obj2id.get(c, self.con2id.get(c))
                if t is not None and t != self.obj2id[o]:
                    isa[(self.obj2id[o], t)] = 1.0
        A_isa = self._mat(isa)
        A_isa_true = A_isa.copy()                 # cho IDF: không phụ thuộc ablation random_edges

        # ---------- ablation: cạnh ngẫu nhiên, GIỮ NGUYÊN số cạnh ----------
        if ablation == "random_edges":
            A_scene = self._randomize(A_scene, rng)
            A_rel   = self._randomize(A_rel, rng)
            A_isa   = self._randomize(A_isa, rng)
        if ablation == "no_relations":
            A_scene = A_rel = A_isa = self._mat({})

        self.scene_count = dict(scene)
        self.cooc_count = dict(cooc)
        for M in (A_scene, A_rel, A_isa):
            if CFG.edge_weighting:
                M.data[:] = np.clip(M.data, 0.0, 1.0)   # P(đích|nguồn); xáo cạnh có thể cộng dồn >1
            else:
                M.data[:] = 1.0                   # R_p[i,c] = SỐ đường đi thật sự (cách cũ)
        self.A_contains, self.A_scene, self.A_related, self.A_isa = A_con, A_scene, A_rel, A_isa

        # ---------- ma trận đếm đường đi theo mẫu ----------
        R = {
            "P0_contains":    A_con.tocsr(),
            "P1_scene":       (A_con @ A_scene).tocsr(),
            "P2_isa":         (A_con @ A_isa).tocsr(),
            "P3_related":     (A_con @ A_rel).tocsr(),
            "P4_scene_isa":   (A_con @ A_scene @ A_isa).tocsr(),
            "P5_related_isa": (A_con @ A_rel @ A_isa).tocsr(),
        }
        # Bỏ đường đi gián tiếp tới khái niệm mà ảnh ĐÃ chứa trực tiếp: chúng chỉ đếm lại
        # tín hiệu P0 (vd 'dog –pushing→ terrier –là_một→ dog' ở bản trước).
        if CFG.drop_redundant_indirect:
            direct = (A_con > 0).astype(np.float32)
            for p in PATTERNS[1:]:
                R[p] = (R[p] - R[p].multiply(direct)).tocsr()
                R[p].eliminate_zeros()

        # IDF của node đích, tính CHỈ trên ảnh TRAIN: khái niệm mà hầu như ảnh nào cũng
        # với tới (person, man...) không phân biệt được ứng viên ⇒ trọng số ≈ 0.
        tr_rows = np.array(sorted(self.img_pos[i] for i in train_set if i in self.img_pos))
        if CFG.idf_from == "taxonomy":
            # df(đối tượng) = số ảnh train CHỨA nó; df(khái niệm) = số ảnh train chứa một hạ vị
            # của nó (isa thật). Không đếm scene/related ⇒ IDF của khớp trực tiếp giống hệt
            # nhau ở mọi ablation, và quan hệ không còn "pha loãng" P0.
            df_con = np.asarray((A_con[tr_rows] > 0).sum(0)).ravel()
            df_isa = np.asarray(((A_con @ A_isa_true)[tr_rows] > 0).sum(0)).ravel()
            df = np.where(np.arange(self.n_nodes) < self.n_img + self.n_obj, df_con, df_isa)
        else:
            reach = None
            for p in PATTERNS:
                m = (R[p][tr_rows] > 0).astype(np.float32)
                reach = m if reach is None else reach + m
            df = np.asarray((reach > 0).sum(0)).ravel()
        idf = np.log((len(tr_rows) + 1.0) / (df + 1.0)).astype(np.float32)
        if not CFG.use_idf:
            idf = np.ones_like(idf)
        idf[:self.n_img] = 0.0
        self.idf = idf
        D = sp.diags(idf)
        # R[p][i, c] = Σ_{đường đi i→c theo mẫu p} idf(c). Liệt kê đường đi dùng ĐÚNG
        # trọng số này nên giải thích vẫn khớp điểm (self-test kiểm tra).
        self.R = {p: (R[p] @ D).tocsr() for p in PATTERNS}
        for p in PATTERNS:
            self.R[p].eliminate_zeros()

        self.name2id = dict(self.obj2id); self.name2id.update(self.con2id)
        self.edge_index, self.edge_type = self._build_edge_index()

    def _mat(self, d):
        if not d:
            return sp.csr_matrix((self.n_nodes, self.n_nodes), dtype=np.float32)
        keys = list(d.keys())
        r = np.fromiter((k[0] for k in keys), int, len(keys))
        c = np.fromiter((k[1] for k in keys), int, len(keys))
        v = np.fromiter(d.values(), np.float32, len(keys))
        return sp.csr_matrix((v, (r, c)), shape=(self.n_nodes, self.n_nodes))

    def _randomize(self, A, rng):
        """Giữ nguyên SỐ LƯỢNG cạnh, xáo trộn đích đến — đối chứng cho ablation."""
        A = A.tocoo()
        if A.nnz == 0:
            return A.tocsr()
        new_col = rng.randint(self.n_img, self.n_nodes, size=A.nnz)
        M = sp.csr_matrix((A.data, (A.row, new_col)), shape=A.shape)
        M.sum_duplicates()
        return M

    def _build_edge_index(self):
        parts, types = [], []
        for t, A in enumerate([self.A_contains.tocoo(), self.A_scene.tocoo(),
                               self.A_related.tocoo(), self.A_isa.tocoo()]):
            if A.nnz == 0:
                continue
            parts.append(np.stack([A.row, A.col])); types.append(np.full(A.nnz, t))
            parts.append(np.stack([A.col, A.row])); types.append(np.full(A.nnz, t + 4))
        if not parts:
            return (torch.zeros(2, 0, dtype=torch.long), torch.zeros(0, dtype=torch.long))
        ei = torch.tensor(np.concatenate(parts, 1), dtype=torch.long)
        et = torch.tensor(np.concatenate(types), dtype=torch.long)
        return ei, et

    def summary(self):
        return dict(nodes=self.n_nodes, images=self.n_img, object_classes=self.n_obj,
                    concepts=self.n_con, edges=int(self.edge_index.shape[1] // 2),
                    contains=int(self.A_contains.nnz), scene=int(self.A_scene.nnz),
                    related=int(self.A_related.nnz), isa=int(self.A_isa.nnz),
                    paths={p: int(self.R[p].nnz) for p in PATTERNS},
                    ablation=self.ablation)

## 6. Phân tích truy vấn → khái niệm trên đồ thị

Một truy vấn được quy về tập node (lớp đối tượng hoặc khái niệm) mà nó nhắc tới.
Đây là điểm neo để `s_path` tìm đường đi.

In [9]:
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next".split())


def parse_query(text, graph):
    """Trả về [(node_id, tên, n-gram khớp)] cho các khái niệm xuất hiện trong truy vấn."""
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):                                    # ưu tiên cụm 2 từ
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


def path_score_matrix(graph, concept_ids, cand_idx):
    """[len(cand), n_patterns] — số đường đi (đã log1p) theo từng mẫu, cho mỗi ứng viên."""
    out = np.zeros((len(cand_idx), len(PATTERNS)), np.float32)
    if not concept_ids:
        return out
    cols = np.array(sorted(set(concept_ids)))
    for p, name in enumerate(PATTERNS):
        sub = graph.R[name][cand_idx][:, cols]
        out[:, p] = np.asarray(sub.sum(1)).ravel()
    return np.log1p(out)

## 7. Mô hình

* **7.1 GNN cổ điển (cấp độ 1)**: GraphSAGE/GAT + PathScorer.
* **7.2 GNN của SCENIR (nhiệm vụ 2)**: thay nhánh `s_GNN` bằng encoder của SCENIR, bật bằng `CFG.gnn_type = "scenir"`.

### 7.1 GNN cổ điển (cấp độ 1)

GNN **tự cài bằng PyTorch thuần** (không cần PyTorch Geometric — tránh hẳn rủi ro cài đặt
trên Kaggle, và là thứ nhóm phải tự giải thích được khi vấn đáp).

* **GraphSAGE** — `h' = W₁h + W₂·mean(h_neighbours)`. Inductive: node ảnh test chưa từng
  tham gia huấn luyện vẫn có biểu diễn hợp lệ.
* **GAT** — hệ số chú ý từng cạnh, softmax theo node đích (dùng `index_reduce_` để ổn định số học).
* **PathScorer** — một trọng số học được cho mỗi mẫu đường đi, nhân hệ số suy giảm theo độ dài.
  Vì điểm là **tổng cộng tính** nên đóng góp của từng đường đi tách bạch được ⇒ fidelity đo được.

In [10]:
def scatter_mean(src, index, n):
    out = torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    cnt = torch.zeros(n, 1, device=src.device, dtype=src.dtype)
    cnt.index_add_(0, index, torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype))
    return out / cnt.clamp(min=1)


def scatter_softmax(logits, index, n):
    m = torch.full((n,), -1e30, device=logits.device, dtype=logits.dtype)
    m = m.index_reduce_(0, index, logits, "amax", include_self=True)
    e = (logits - m[index]).exp()
    s = torch.zeros(n, device=logits.device, dtype=logits.dtype).index_add_(0, index, e)
    return e / s[index].clamp(min=1e-16)


class SAGELayer(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.Linear(din, dout)

    def forward(self, x, edge_index, edge_weight=None):
        src, dst = edge_index
        msg = x[src] if edge_weight is None else x[src] * edge_weight.unsqueeze(-1)
        agg = scatter_mean(msg, dst, x.size(0))
        return self.self_w(x) + self.neigh_w(agg)


class TypedSAGELayer(nn.Module):
    """h' = W₀h + Σ_t σ(g_t)·W_t·mean_{u∈N_t(v)} h_u — mỗi loại cạnh một ma trận và một cổng.
    SAGELayer cũ gộp contains/scene/related/isa vào CHUNG một trung bình ⇒ cạnh nhiễu pha
    loãng tín hiệu ảnh–đối tượng, và mô hình không có cách nào tắt riêng một loại cạnh.
    Cổng các loại quan hệ khởi tạo gần đóng ⇒ điểm xuất phát ≈ no_relations."""

    N_TYPES = 8                                   # 4 loại × 2 chiều (xem _build_edge_index)

    def __init__(self, din, dout, gate_init=-1.5):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(self.N_TYPES)])
        g = torch.full((self.N_TYPES,), float(gate_init))
        g[0] = g[4] = 3.0                         # contains (hai chiều) mở sẵn
        self.gate = nn.Parameter(g)

    def forward(self, x, edge_index, edge_type):
        src, dst = edge_index
        out = self.self_w(x)
        for t in range(self.N_TYPES):
            m = edge_type == t
            if not bool(m.any()):
                continue
            agg = scatter_mean(x[src[m]], dst[m], x.size(0))
            out = out + torch.sigmoid(self.gate[t]) * self.neigh_w[t](agg)
        return out


class GATLayer(nn.Module):
    def __init__(self, din, dout, heads=4):
        super().__init__()
        assert dout % heads == 0
        self.h, self.dk = heads, dout // heads
        self.lin = nn.Linear(din, dout)
        self.att_src = nn.Parameter(torch.empty(heads, self.dk))
        self.att_dst = nn.Parameter(torch.empty(heads, self.dk))
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index, edge_weight=None):
        N = x.size(0)
        src, dst = edge_index
        z = self.lin(x).view(N, self.h, self.dk)
        a = (z * self.att_src).sum(-1)[src] + (z * self.att_dst).sum(-1)[dst]   # [E,H]
        a = F.leaky_relu(a, 0.2)
        if edge_weight is not None:
            a = a + edge_weight.log().clamp(min=-5).unsqueeze(-1)
        out = torch.zeros(N, self.h, self.dk, device=x.device, dtype=z.dtype)
        for hh in range(self.h):                      # softmax riêng từng head
            al = scatter_softmax(a[:, hh], dst, N)
            out[:, hh].index_add_(0, dst, z[src, hh] * al.unsqueeze(-1))
        return out.reshape(N, -1)


class GraphScorer(nn.Module):
    """Sinh biểu diễn node bằng GNN, chiếu CLIP-text vào cùng không gian, cộng điểm đường đi."""

    def __init__(self, n_nodes, clip_dim, img_feat, cfg=CFG, edge_type=None):
        super().__init__()
        self.n_img = img_feat.shape[0]
        self.typed = cfg.gnn_typed and cfg.gnn_type == "sage"
        # loại cạnh của đồ thị ĐẦY ĐỦ — dùng khi encode_graph được gọi không kèm edge_type
        self.register_buffer("full_et", (edge_type if edge_type is not None
                                         else torch.zeros(0, dtype=torch.long)).clone())
        # Node ẢNH: chiếu từ đặc trưng CLIP đóng băng ⇒ inductive, ảnh test chưa từng
        # tham gia huấn luyện vẫn có biểu diễn hợp lệ (không phải vector ngẫu nhiên).
        self.register_buffer("img_feat", img_feat.clone())
        self.img_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        # Node ĐỐI TƯỢNG / KHÁI NIỆM: embedding học được (chúng lặp lại giữa các ảnh).
        self.sym_emb = nn.Embedding(n_nodes - self.n_img, cfg.gnn_dim)
        nn.init.normal_(self.sym_emb.weight, std=0.02)
        if self.typed:
            self.layers = nn.ModuleList([TypedSAGELayer(cfg.gnn_dim, cfg.gnn_dim, cfg.gate_init)
                                         for _ in range(cfg.gnn_layers)])
        else:
            Layer = SAGELayer if cfg.gnn_type == "sage" else GATLayer
            self.layers = nn.ModuleList(
                [Layer(cfg.gnn_dim, cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(                     # CLIP-text → không gian đồ thị
            nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(),
            nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        # PathScorer: 1 trọng số / mẫu, softplus để luôn ≥0 (giải thích mới diễn giải được)
        # P0 khởi tạo 0 (softplus = 0.69), P1..P5 khởi tạo thấp: đường đi gián tiếp phải
        # "tự chứng minh" trên mẫu âm khó thì trọng số mới tăng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        # path_gain CỐ ĐỊNH (buffer, không học): sau znorm_per_row điểm bất biến theo tỉ lệ nên
        # gain không có tín hiệu thật — gradient chỉ đi qua eps=1e-6 của z-norm, rất nhỏ nhưng
        # cùng dấu, và Adam vẫn bước đủ lr ⇒ gain trôi về ~1e-7, nhánh path bị TẮT NGẦM và mọi
        # "đóng góp" trong giải thích hiện +0.0000 (đã gặp trên AG). Tắt nhánh path là việc của λ.
        self.register_buffer("path_gain", torch.tensor(1.0))
        # trọng số của z(s_clip) trong logit lúc HUẤN LUYỆN, học được (softplus, khởi tạo = 1):
        # trên AG CLIP rất yếu (MRR 0.12) nên mô hình cần hạ được trọng số này.
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        # Hệ số của z(s_path) trong logit HUẤN LUYỆN (softplus, khởi tạo = 1). Thiếu nó, nhánh path
        # có hệ số cứng = 1, nên khi nó gây hại mô hình tự tắt bằng đường thoái hoá: đợt 2 kéo
        # path_gain → 1e-7, đợt 3 dồn trọng số chuẩn hoá sang mẫu không có dữ liệu (P0 → 0.000).
        # Lúc suy luận vẫn trộn bằng λ chọn trên val; hệ số này chỉ dùng khi huấn luyện.
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        return torch.cat([self.img_proj(self.img_feat), self.sym_emb.weight], 0)

    def edge_gates(self):
        """σ(g_t) của từng lớp — in ra để biết mô hình có thực sự dùng quan hệ hay không."""
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers] if self.typed else []

    def encode_graph(self, edge_index, edge_type=None):
        h = self.node_features()
        if self.typed and edge_type is None:
            assert self.full_et.numel() == edge_index.size(1), \
                "encode_graph trên đồ thị con cần truyền edge_type tương ứng"
            edge_type = self.full_et
        for i, (l, n) in enumerate(zip(self.layers, self.norms)):
            h = l(h, edge_index, edge_type) if self.typed else l(h, edge_index)
            h = n(h)
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        """Trọng số hiệu dụng của mỗi mẫu: P0 NEO = 1, P_k = softplus(w_k) · decay^(len_k − 1).
        Neo P0 thay cho chuẩn hoá tổng = 1: tỉ lệ vẫn định danh được, nhưng trọng số không thể
        trôi sang mẫu vắng dữ liệu (no_relations seed 2 đợt 3: P0 = 0.000, nhánh path chết)."""
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)          # [B, d]
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])      # [B, K]

    def path_score(self, path_mat):
        """path_mat: [B, K, n_patterns] → [B, K]. Cộng tính theo mẫu ⇒ tách được đóng góp."""
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)

### 7.2 GNN của SCENIR thay cho GNN cổ điển

SCENIR (ICML 2025, [arXiv 2505.15867](https://arxiv.org/html/2505.15867v1)). Chỉ thay nhánh `s_GNN`; split, ứng viên CLIP,
`s_path`, chọn α/λ trên val, ablation và fidelity giữ nguyên. Bật bằng `CFG.gnn_type = "scenir"`.

* **Đầu vào:** mỗi ảnh một scene graph. Đỉnh = lớp đối tượng của ảnh + khái niệm IsA của chúng; cạnh = cạnh
  `scene`/`related` (xây chỉ từ train) và `isa` của `MultimodalGraph` nằm trong tập đỉnh đó. Đặc trưng đỉnh =
  CLIP-text của tên đỉnh.
* **Encoder:** SCENIR GAE — 3 lớp GIN với hai nhánh `mu`/`logstd` tách rời, feature decoder và edge decoder MLP,
  discriminator đối kháng. Huấn luyện tự giám sát trên scene graph của ảnh **train** rồi đóng băng.
* **Điểm:** `z_I = L2norm(Σ_v mu_v)`, `s_GNN = cos(q_proj(CLIP_text(q)), z_I)`. `q_proj` và trọng số đường đi học
  bằng đúng vòng InfoNCE của GNN cổ điển trong `train_graph_scorer`.

In [ ]:
from contextlib import contextmanager

# Siêu tham số SCENIR = config.json của repo gốc; chiều đặc trưng đỉnh lấy theo CLIP-text (512 thay vì 768).
for _k, _v in dict(scenir_latent=1000, scenir_layers=3, scenir_disc_hidden=32,
                   scenir_edge_hidden=256, scenir_edge_out=32, scenir_lr=1e-3,
                   scenir_lr_gamma=0.95, scenir_bs=64, scenir_disc_steps=5,
                   scenir_epochs=30, scenir_with_concepts=True,
                   scenir_proj_hidden=1024).items():
    setattr(CFG, _k, _v)


@contextmanager
def use_gnn(kind):
    """Tạm đổi CFG.gnn_type ('sage' | 'gat' | 'scenir'); khôi phục khi ra khỏi khối."""
    old = CFG.gnn_type
    CFG.gnn_type = kind
    try:
        yield
    finally:
        CFG.gnn_type = old


def image_scene_graphs(graph):
    """Hợp rời (disjoint union) scene graph của mọi ảnh, cache trên chính đồ thị.
    node_gid: id node toàn cục của từng đỉnh cục bộ; node_ptr/edge_ptr: đỉnh/cạnh của ảnh i nằm liền
    nhau trong [ptr[i], ptr[i+1]); edge_index vô hướng (mỗi cạnh lưu hai chiều)."""
    if hasattr(graph, "_scenir_sg"):
        return graph._scenir_sg
    A_obj = ((graph.A_scene + graph.A_related) > 0).tocsr()
    A_isa = graph.A_isa.tocsr()
    nbrs = lambda M, n: M.indices[M.indptr[n]:M.indptr[n + 1]]
    node_gid, node_ptr, edge_ptr, src, dst = [], [0], [0], [], []
    for i in range(graph.n_img):
        objs = [int(o) for o in graph.img_objects[i]]
        local = {o: k for k, o in enumerate(objs)}
        pairs = set()
        for o in objs:
            for t in nbrs(A_obj, o):
                t = int(t)
                if t in local and t != o:
                    pairs.add((min(local[o], local[t]), max(local[o], local[t])))
        if CFG.scenir_with_concepts:
            for o in objs:
                for c in nbrs(A_isa, o):
                    c = int(c)
                    if c == o:
                        continue
                    if c not in local:
                        local[c] = len(local)
                    pairs.add((min(local[o], local[c]), max(local[o], local[c])))
        base = len(node_gid)
        node_gid.extend(local)                 # dict giữ thứ tự chèn = thứ tự id cục bộ
        node_ptr.append(len(node_gid))
        for a, b in sorted(pairs):
            src += [base + a, base + b]; dst += [base + b, base + a]
        edge_ptr.append(len(src))
    sg = dict(node_gid=np.array(node_gid, np.int64), node_ptr=np.array(node_ptr, np.int64),
              edge_ptr=np.array(edge_ptr, np.int64),
              edge_index=np.array([src, dst], np.int64).reshape(2, -1))
    graph._scenir_sg = sg
    return sg


_SCENIR_NAME_EMB = {}


def _encode_node_names(names):
    """CLIP-text đóng băng (cùng encoder với truy vấn). Không có CLIP (synthetic/offline) thì dùng
    vector ngẫu nhiên cố định theo tên — chỉ để smoke-test."""
    try:
        return clip_text_features(names)
    except Exception as e:
        print(f"⚠ Không mã hoá được tên node bằng CLIP ({type(e).__name__}) → vector ngẫu nhiên theo tên.")
        out = []
        for n in names:
            v = np.random.RandomState(int(hashlib.md5(n.encode()).hexdigest()[:8], 16)).randn(512)
            out.append((v / np.linalg.norm(v)).astype(np.float32))
        return np.stack(out)


def node_name_features(graph):
    """[n_nodes, D] đặc trưng đỉnh; dòng của node ảnh để 0 (scene graph không chứa node ảnh)."""
    names = [graph.id2name[i] for i in range(graph.n_img, graph.n_nodes)]
    new = [n for n in dict.fromkeys(names) if n not in _SCENIR_NAME_EMB]
    if new:
        _SCENIR_NAME_EMB.update(zip(new, _encode_node_names(new)))
    dim = len(next(iter(_SCENIR_NAME_EMB.values()))) if _SCENIR_NAME_EMB else 512
    X = np.zeros((graph.n_nodes, dim), np.float32)
    if names:
        X[graph.n_img:] = np.stack([_SCENIR_NAME_EMB[n] for n in names])
    return torch.tensor(X)

In [ ]:
def scatter_sum(src, index, n):
    return torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype).index_add_(0, index, src)


class GINLayer(nn.Module):
    """GINConv(nn.Linear) của PyG với ε = 0: h'_v = W·(h_v + Σ_{u∈N(v)} h_u)."""

    def __init__(self, din, dout):
        super().__init__()
        self.lin = nn.Linear(din, dout)

    def forward(self, x, edge_index):
        src, dst = edge_index
        return self.lin(x + scatter_sum(x[src], dst, x.size(0)))


class ScenirEncoder(nn.Module):
    """MultiGINEncoder của SCENIR: hai nhánh GIN tách rời cho mu và logstd, không ReLU ở lớp cuối."""

    def __init__(self, din, dhid, dout, layers=3):
        super().__init__()
        dims = [din] + [dhid] * (layers - 1) + [dout]
        self.gnn_mu = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.gnn_logstd = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])

    @staticmethod
    def _run(layers, x, edge_index):
        for i, layer in enumerate(layers):
            x = layer(x, edge_index)
            if i < len(layers) - 1:
                x = F.relu(x)
        return x

    def forward(self, x, edge_index, mu_only=False):
        mu = self._run(self.gnn_mu, x, edge_index)
        return mu if mu_only else (mu, self._run(self.gnn_logstd, x, edge_index))


class ScenirMLP(nn.Module):
    """MLP 2 lớp — feature decoder và edge decoder của SCENIR."""

    def __init__(self, din, dhid, dout):
        super().__init__()
        self.layer_1, self.layer_2 = nn.Linear(din, dhid), nn.Linear(dhid, dout)

    def forward(self, x):
        return self.layer_2(F.relu(self.layer_1(x)))


class ScenirDiscriminator(nn.Module):
    def __init__(self, din, dhid):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din, dhid), nn.ReLU(),
                                 nn.Linear(dhid, dhid), nn.ReLU(), nn.Linear(dhid, 1))

    def forward(self, z):
        return self.net(z)


SCENIR_EPS, SCENIR_MAX_LOGSTD = 1e-15, 10


def scenir_negative_edges(edge_index, batch, n_samples):
    """Cặp (u, v) cùng một đồ thị trong batch, không tự nối, không trùng cạnh thật."""
    N = batch.numel()
    cnt = torch.bincount(batch)
    ptr = torch.cat([cnt.new_zeros(1), cnt.cumsum(0)[:-1]])
    u = torch.randint(0, N, (2 * n_samples,), device=batch.device)
    b = batch[u]
    off = torch.minimum((torch.rand(u.numel(), device=batch.device) * cnt[b]).long(), cnt[b] - 1)
    v = ptr[b] + off
    keep = (u != v) & ~torch.isin(u * N + v, edge_index[0] * N + edge_index[1])
    return torch.stack([u[keep], v[keep]])[:, :n_samples]


class ScenirGAE(nn.Module):
    """SCENIR = ARGVA (VGAE + điều chuẩn đối kháng) + feature decoder MLP + edge decoder MLP."""

    def __init__(self, din, cfg=CFG):
        super().__init__()
        L = cfg.scenir_latent
        self.encoder = ScenirEncoder(din, L, L, cfg.scenir_layers)
        self.discriminator = ScenirDiscriminator(L, cfg.scenir_disc_hidden)
        self.feature_decoder = ScenirMLP(L, din, din)
        self.edge_decoder = ScenirMLP(L, cfg.scenir_edge_hidden, cfg.scenir_edge_out)

    def encode(self, x, edge_index):
        mu, logstd = self.encoder(x, edge_index)
        logstd = logstd.clamp(max=SCENIR_MAX_LOGSTD)
        z = mu + torch.randn_like(logstd) * logstd.exp() if self.training else mu
        return z, mu, logstd

    @staticmethod
    def _inner(z, edge_index):
        return torch.sigmoid((z[edge_index[0]] * z[edge_index[1]]).sum(-1))

    @staticmethod
    def kl_loss(mu, logstd):
        return -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu ** 2 - logstd.exp() ** 2, dim=1))

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.discriminator(z)) + SCENIR_EPS).mean()

    def discriminator_loss(self, z):
        real = torch.sigmoid(self.discriminator(torch.randn_like(z)))
        fake = torch.sigmoid(self.discriminator(z.detach()))
        return -torch.log(real + SCENIR_EPS).mean() - torch.log(1 - fake + SCENIR_EPS).mean()

    def recon_loss(self, x, edge_index, batch):
        """recon_loss_batched của repo: MSE tái tạo đặc trưng + BCE cạnh dương/âm trên đầu ra edge decoder.
        Encode lại như bản gốc; trả về (loss, mu, logstd) để kl_loss dùng đúng lần encode này."""
        z, mu, logstd = self.encode(x, edge_index)
        loss = F.mse_loss(self.feature_decoder(z), x)
        if edge_index.size(1) > 0:
            z_e = self.edge_decoder(z)
            loss = loss - torch.log(self._inner(z_e, edge_index) + SCENIR_EPS).mean()
            neg = scenir_negative_edges(edge_index, batch, edge_index.size(1))
            if neg.size(1) > 0:
                loss = loss - torch.log(1 - self._inner(z_e, neg) + SCENIR_EPS).mean()
        return loss, mu, logstd

In [ ]:
def _sg_batch(sg, X, img_idx):
    """Gom scene graph của các ảnh img_idx thành một batch (x, edge_index, batch) trên DEVICE.
    X phải nằm sẵn trên DEVICE."""
    img_idx = np.asarray(img_idx)
    ranges = lambda p: np.concatenate([np.arange(p[i], p[i + 1]) for i in img_idx])
    nodes, edges = ranges(sg["node_ptr"]), ranges(sg["edge_ptr"])
    counts = sg["node_ptr"][img_idx + 1] - sg["node_ptr"][img_idx]
    remap = np.full(len(sg["node_gid"]), -1, np.int64)
    remap[nodes] = np.arange(len(nodes))
    ei = remap[sg["edge_index"][:, edges]]
    x = X[torch.from_numpy(sg["node_gid"][nodes]).to(X.device)]
    return (x, torch.from_numpy(ei).to(X.device),
            torch.from_numpy(np.repeat(np.arange(len(img_idx)), counts)).to(X.device))


def train_scenir_gae(sg, X, img_idx, seed=0, verbose=True):
    """Huấn luyện SCENIR GAE tự giám sát CHỈ trên scene graph của ảnh train (không truy vấn, không nhãn).
    loss = (1/6)·reg + 3·recon + (1/3)·(1/num_nodes)·kl; discriminator cập nhật disc_steps bước/lần."""
    set_seed(seed)
    # Giữ cả đồ thị không cạnh (recon_loss chịu được), nếu không ablation no_relations không còn gì để huấn luyện.
    n_nodes = np.diff(sg["node_ptr"])
    idx = np.array([i for i in img_idx if n_nodes[i] > 0], np.int64)
    assert len(idx) > 0, "Không ảnh train nào có scene graph — kiểm tra nhãn đối tượng."
    gae = ScenirGAE(X.shape[1]).to(DEVICE)
    disc_opt = torch.optim.AdamW(gae.discriminator.parameters(), lr=CFG.scenir_lr)
    enc_opt = torch.optim.AdamW(list(gae.encoder.parameters()) + list(gae.edge_decoder.parameters())
                                + list(gae.feature_decoder.parameters()), lr=CFG.scenir_lr)
    sched = torch.optim.lr_scheduler.ExponentialLR(enc_opt, gamma=CFG.scenir_lr_gamma)
    hist, t0 = [], time.time()
    for ep in range(CFG.scenir_epochs):
        gae.train()
        perm = np.random.permutation(idx)
        tot_e = tot_d = 0.0
        nb = 0
        for s in range(0, len(perm), CFG.scenir_bs):
            x, ei, b = _sg_batch(sg, X, perm[s:s + CFG.scenir_bs])
            enc_opt.zero_grad()
            z, _, _ = gae.encode(x, ei)
            d_acc = 0.0
            for _ in range(CFG.scenir_disc_steps):
                disc_opt.zero_grad()
                d_loss = gae.discriminator_loss(z)
                d_loss.backward()
                disc_opt.step()
                d_acc += d_loss.item()
            rec, mu, logstd = gae.recon_loss(x, ei, b)
            loss = ((1 / 6) * gae.reg_loss(z) + 3 * rec
                    + (1 / 3) * (1 / x.size(0)) * gae.kl_loss(mu, logstd))
            loss.backward()
            enc_opt.step()
            tot_e += loss.item(); tot_d += d_acc / CFG.scenir_disc_steps; nb += 1
        sched.step()
        hist.append((tot_e / max(nb, 1), tot_d / max(nb, 1)))
        if verbose and (ep % 5 == 0 or ep == CFG.scenir_epochs - 1):
            print(f"  [SCENIR GAE] epoch {ep:2d} | enc {hist[-1][0]:.4f} | disc {hist[-1][1]:.4f} "
                  f"| {len(idx)} đồ thị train | {time.time() - t0:.0f}s")
    gae.eval()
    for p in gae.parameters():
        p.requires_grad_(False)
    return gae, hist


@torch.no_grad()
def scenir_embed_images(gae, sg, X, chunk=1024):
    """z_I = Σ_v mu_v (eval ⇒ không lấy mẫu) cho MỌI ảnh; ảnh không có đỉnh nào → vector 0."""
    gae.eval()
    n_img = len(sg["node_ptr"]) - 1
    out = torch.zeros(n_img, CFG.scenir_latent)
    for s in range(0, n_img, chunk):
        idx = np.arange(s, min(s + chunk, n_img))
        x, ei, b = _sg_batch(sg, X, idx)
        if x.size(0) == 0:
            continue
        mu = gae.encoder(x, ei, mu_only=True)
        out[torch.from_numpy(idx)] = scatter_sum(mu, b, len(idx)).cpu()
    return out

In [ ]:
class ScenirGraphScorer(nn.Module):
    """Cùng giao diện với GraphScorer (encode_graph / gnn_score / path_score / pattern_weights), nên
    score_queries, tune_on_val, fidelity và demo phía sau dùng lại nguyên vẹn."""

    def __init__(self, gae, z_img, clip_dim, cfg=CFG):
        super().__init__()
        self.gae = gae
        self.register_buffer("z_img", F.normalize(z_img, dim=-1))      # sum-pool ⇒ bắt buộc L2
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.scenir_proj_hidden), nn.ReLU(),
                                    nn.Linear(cfg.scenir_proj_hidden, z_img.shape[1]))
        # Phần chấm đường đi khởi tạo giống GraphScorer (mục 7.1) để hai nhánh GNN so sánh công bằng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def encode_graph(self, edge_index=None, edge_type=None):
        """GAE đã đóng băng ⇒ nhúng ảnh tính một lần; edge_index/edge_type chỉ để khớp giao diện GraphScorer."""
        return self.z_img

    def edge_gates(self):
        return []                                # SCENIR không có cổng theo loại cạnh

    pattern_weights = GraphScorer.pattern_weights
    gnn_score = GraphScorer.gnn_score
    path_score = GraphScorer.path_score


def build_scenir_scorer(engine, train_img_idx, seed=0, verbose=True):
    """Dựng scene graph từng ảnh, huấn luyện GAE trên ảnh train, nhúng mọi ảnh."""
    g = engine.graph
    sg = image_scene_graphs(g)
    X = node_name_features(g).to(DEVICE)
    if verbose:
        nv, ne = np.diff(sg["node_ptr"]), np.diff(sg["edge_ptr"]) // 2
        print(f"  SCENIR scene graph/ảnh: {nv.mean():.1f} đỉnh, {ne.mean():.1f} cạnh | "
              f"ảnh không có cạnh: {(ne == 0).mean():.1%}")
    gae, _ = train_scenir_gae(sg, X, train_img_idx, seed=seed, verbose=verbose)
    return ScenirGraphScorer(gae, scenir_embed_images(gae, sg, X), engine.txt_feat.shape[1])

## 8. Bộ máy truy vấn: ứng viên, huấn luyện, xếp hạng lại

Để tính `s_path` cho mọi cặp (truy vấn, ảnh) mà không phải lặp Python, ta dựng
ma trận chỉ báo khái niệm `Q [n_query × n_node]` rồi nhân thưa một lần cho mỗi mẫu:
`S_p = R_p · Qᵀ` → `[n_img × n_query]`. Sau đó chỉ việc tra.

In [ ]:
def znorm_per_row(x, eps=1e-6):
    """Chuẩn hoá z theo từng truy vấn. Dùng CHUNG cho train và eval để α, λ chọn
    trên val vẫn đúng khi áp lên test."""
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


minmax_per_row = znorm_per_row      # giữ tên cũ cho tương thích


class RetrievalEngine:
    def __init__(self, records, graph, img_feat, txt_feat, queries):
        """queries: list[dict(text, gold_image_id, group)] — group ∈ {direct, indirect}"""
        self.records, self.graph = records, graph
        self.img_feat = torch.tensor(img_feat)
        self.txt_feat = torch.tensor(txt_feat)
        self.queries = queries
        self.q_concepts = [parse_query(q["text"], graph) for q in queries]
        self._build_S()

    def _build_S(self):
        g = self.graph
        rows, cols = [], []
        for j, hits in enumerate(self.q_concepts):
            for nid, _, _ in hits:
                rows.append(nid); cols.append(j)
        Q = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                          shape=(g.n_nodes, len(self.queries)))
        self.S = {p: (g.R[p] @ Q).tocsr() for p in PATTERNS}   # [n_img × n_query]

    def path_feats(self, img_idx, q_idx):
        """img_idx [B,K] hoặc [B]; q_idx [B] → [.., n_patterns]"""
        img_idx = np.asarray(img_idx); q_idx = np.asarray(q_idx)
        if img_idx.ndim == 1:
            img_idx = img_idx[:, None]
        B, K = img_idx.shape
        rr = img_idx.ravel(); cc = np.repeat(q_idx, K)
        out = np.zeros((B * K, len(PATTERNS)), np.float32)
        for p, name in enumerate(PATTERNS):
            out[:, p] = np.asarray(self.S[name][rr, cc]).ravel()
        return np.log1p(out).reshape(B, K, len(PATTERNS))   # log1p SAU khi cộng

    def clip_candidates(self, q_idx, gallery_idx, k):
        qf = self.txt_feat[q_idx]                       # [B, D]
        gf = self.img_feat[gallery_idx]                 # [G, D]
        sim = qf @ gf.T
        k = min(k, len(gallery_idx))
        top = sim.topk(k, dim=1)
        return gallery_idx[top.indices.numpy()], top.values.numpy()


def train_graph_scorer(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    """InfoNCE với mẫu âm trong batch. CLIP đóng băng; chỉ học GNN + trọng số đường đi."""
    set_seed(seed)
    g = engine.graph
    if CFG.gnn_type == "scenir":                 # mục 7.2: GAE tự giám sát trên ảnh train, rồi đóng băng
        model = build_scenir_scorer(engine, train_img_idx, seed=seed, verbose=verbose).to(DEVICE)
    else:
        model = GraphScorer(g.n_nodes, engine.txt_feat.shape[1],
                            engine.img_feat, edge_type=g.edge_type).to(DEVICE)
    # Cổng cạnh + trọng số mẫu đường đi chỉ là vài vô hướng: cho lr riêng (×CFG.scalar_lr_mult),
    # không weight decay — nếu không chúng gần như đứng yên ở giá trị khởi tạo.
    # Tham số đã đóng băng (GAE của SCENIR) không đưa vào optimizer.
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n, p in params
              if n.endswith("gate") or n in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW(
        [dict(params=[p for _, p in params if id(p) not in sid]),
         dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
        lr=CFG.lr, weight_decay=CFG.weight_decay)
    # INDUCTIVE: khi huấn luyện bỏ mọi cạnh chạm node ảnh KHÔNG thuộc train (val/test).
    # Node ảnh val/test thành node cô lập ⇒ không gửi thông điệp, không nhận gradient,
    # không ảnh hưởng tới biểu diễn đối tượng/khái niệm học được. Lúc đánh giá
    # (score_queries) mới dùng đồ thị đầy đủ — GraphSAGE inductive nên hợp lệ.
    keep_img = np.zeros(g.n_img, bool)
    keep_img[np.asarray(train_img_idx)] = True
    src, dst = g.edge_index.numpy()
    is_bad = lambda v: (v < g.n_img) & ~keep_img[np.minimum(v, g.n_img - 1)]
    mask = ~(is_bad(src) | is_bad(dst))
    tmask = torch.from_numpy(mask)
    ei = g.edge_index[:, tmask].to(DEVICE)
    et = g.edge_type[tmask].to(DEVICE)
    if verbose:
        print(f"  cạnh dùng khi huấn luyện: {int(mask.sum())}/{len(mask)} "
              f"(đã bỏ {int((~mask).sum())} cạnh chạm ảnh val/test)")
    hist = []
    tr_q, tr_img = np.asarray(train_q_idx), np.asarray(train_img_idx)
    img_dev = engine.img_feat.to(DEVICE)

    # Mẫu âm KHÓ: CLIP top-K trong gallery TRAIN (bỏ ảnh đúng). Lúc test đồ thị chỉ xếp lại
    # top-100 của CLIP, nên phải học phân biệt ĐÚNG những ứng viên đó — với âm ngẫu nhiên
    # trong batch, đường đi gián tiếp "dễ thắng" và học trọng số không dùng được ở test.
    hard = None
    if CFG.hard_neg_k > 0:
        cand_hn, _ = engine.clip_candidates(tr_q, tr_img, CFG.hard_neg_k + 1)
        hard = np.stack([np.concatenate([row[row != gold], row[:1]])[:CFG.hard_neg_k]
                         for row, gold in zip(cand_hn, tr_img)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.tensor(cand_np, device=DEVICE)
        pm = torch.tensor(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:      # CLIP không có gradient; chỉ trọng số clip_w học được
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, img_dev[cand]))
        return z / CFG.temperature

    for ep in range(CFG.epochs):
        model.train()
        order = np.random.permutation(len(train_q_idx))
        tot, nb = 0.0, 0
        for s in range(0, len(order), CFG.train_bs):
            sel = order[s:s + CFG.train_bs]
            if len(sel) < 8:
                continue
            qi = tr_q[sel]
            gi = tr_img[sel]                              # ảnh đúng của từng truy vấn
            qf = engine.txt_feat[qi].to(DEVICE)
            h = model.encode_graph(ei, et)

            # (a) âm trong batch như cũ: [B, B], đáp án trên đường chéo
            loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf),
                                   torch.arange(len(sel), device=DEVICE))
            # (b) âm khó từ CLIP: [B, 1+K], đáp án ở cột 0
            if hard is not None:
                cand_h = np.concatenate([gi[:, None], hard[sel]], 1)
                loss = loss + F.cross_entropy(logits_for(cand_h, qi, h, qf),
                                              torch.zeros(len(sel), dtype=torch.long, device=DEVICE))

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 3 == 0 or ep == CFG.epochs - 1):
            w = model.pattern_weights().detach().cpu().numpy()
            gates = model.edge_gates()
            # Cổng của LỚP 0: ở lớp cuối chỉ node ảnh được dùng, mà ảnh chỉ nhận cạnh contains
            # ⇒ cổng scene/related/isa lớp cuối không có gradient (luôn đứng ở giá trị khởi tạo).
            gs = (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4])) if gates else ""
            print(f"  epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.3f}" for x in w) + gs +
                  f" | clip_w {F.softplus(model.clip_logit).item():.2f}")
    return model, hist


@torch.no_grad()
def score_queries(model, engine, q_idx, gallery_idx, k=None):
    """Trả về (cand_idx, s_clip, s_gnn, s_path) — mọi điểm đã chuẩn hoá z theo hàng. Dùng đồ thị ĐẦY ĐỦ."""
    k = k or CFG.rerank_k
    model.eval()
    ei = engine.graph.edge_index.to(DEVICE)
    h = model.encode_graph(ei)
    cand, s_clip = engine.clip_candidates(q_idx, gallery_idx, k)
    pm = torch.tensor(engine.path_feats(cand, q_idx)).to(DEVICE)
    qf = engine.txt_feat[q_idx].to(DEVICE)
    s_gnn = model.gnn_score(h, qf, torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    return cand, znorm_per_row(s_clip), znorm_per_row(s_gnn), znorm_per_row(s_path)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)

## 9. Chỉ số đánh giá

In [12]:
def retrieval_metrics(cand, scores, gold_pos, ks=(1, 5, 10)):
    """gold_pos: chỉ số ảnh đúng (trong không gian toàn cục) cho từng truy vấn."""
    order = np.argsort(-scores, axis=1)
    ranked = np.take_along_axis(cand, order, axis=1)
    hit = ranked == np.asarray(gold_pos)[:, None]
    has = hit.any(1)
    rank = np.where(has, hit.argmax(1) + 1, 10**9)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float(np.where(has, 1.0 / rank, 0.0).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    return out


def fmt_metrics(runs):
    """runs: list[dict] từ nhiều seed → chuỗi 'mean ± std'."""
    keys = runs[0].keys()
    return {k: f"{np.mean([r[k] for r in runs]):.4f} ± {np.std([r[k] for r in runs]):.4f}"
            for k in keys}

## 10. Bộ truy vấn đánh giá: trực tiếp vs gián tiếp

Đây là bằng chứng chính cho câu hỏi "tri thức quan hệ giúp ở đâu".

* **Trực tiếp** — truy vấn nhắc tới lớp đối tượng có sẵn trong annotation của ảnh đúng
  (tồn tại đường đi P0). CLIP thường đã làm tốt nhóm này.
* **Gián tiếp** — truy vấn nhắc tới khái niệm **không** nằm trong danh sách đối tượng của ảnh,
  nhưng nối được qua quan hệ (P1–P5). Đây là nhóm đồ thị phải chứng minh giá trị.

Cách phân nhóm này tự động và kiểm tra được, tránh việc gán nhãn cảm tính.

In [13]:
def build_query_set(records, graph, img_ids, seed=0, verbose=True):
    pos = graph.img_pos
    queries = []
    for iid in img_ids:
        i = pos[iid]
        r = records[i]
        hits = parse_query(r["caption"], graph)
        own = set(graph.img_objects[i])
        hit_ids = {n for n, _, _ in hits}
        direct = bool(hit_ids & own)
        indirect_ids = hit_ids - own
        reach = 0
        if indirect_ids:
            cols = np.array(sorted(indirect_ids))
            for p in PATTERNS[1:]:
                reach += int(graph.R[p][i][:, cols].nnz)
        group = "indirect" if (indirect_ids and reach > 0 and not direct) else \
                ("direct" if direct else "other")
        queries.append(dict(text=r["caption"], gold=iid, group=group,
                            n_concepts=len(hits), n_indirect=len(indirect_ids)))
    if verbose:
        print(f"Nhóm truy vấn: {dict(Counter(q['group'] for q in queries))}")
    return queries


def export_eval_queries(queries, graph, path, n=20, allowed_ids=None):
    """Xuất bộ ≥20 truy vấn đánh giá của cấp độ 2. allowed_ids = id ảnh TEST (bắt buộc)."""
    assert allowed_ids is not None, "Truyền allowed_ids=set(split['test']) — chỉ lấy truy vấn của tập test"
    allowed_ids = set(allowed_ids)
    pool = [q for q in queries if q["gold"] in allowed_ids]
    ind = [q for q in pool if q["group"] == "indirect"]
    dir_ = [q for q in pool if q["group"] == "direct"]
    rng = random.Random(0)
    sel = rng.sample(ind, min(n // 2, len(ind))) + rng.sample(dir_, min(n // 2, len(dir_)))
    payload = dict(
        n=len(sel),
        construction=("Truy vấn lấy từ caption của ảnh trong tập test. Nhóm 'indirect' là "
                      "truy vấn có khái niệm KHÔNG nằm trong annotation đối tượng của ảnh "
                      "đúng nhưng nối được qua đường đi P1-P5; nhóm 'direct' có ít nhất một "
                      "khái niệm trùng annotation (đường đi P0). Phân nhóm tự động, kiểm tra "
                      "lại được bằng hàm build_query_set."),
        usage="CHỈ dùng để đánh giá. Không tinh chỉnh mô hình trên bộ này.",
        queries=[{k: q[k] for k in ("text", "gold", "group", "n_concepts", "n_indirect")}
                 for q in sel])
    with open(path, "w") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Đã ghi {len(sel)} truy vấn đánh giá → {path}")
    return sel

## 11. Chọn siêu tham số trên VALIDATION

Quét lưới `(α, λ)` trên tập val. Test chưa hề được chạm vào ở bước này.

In [14]:
def tune_on_val(model, engine, val_q, val_gallery, verbose=True):
    cand, sc, sg, sp_ = score_queries(model, engine, val_q["q_idx"], val_gallery)
    best = (-1, None)
    grid = []
    for a in CFG.alpha_grid:
        for l in CFG.lambda_grid:
            m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), val_q["gold_pos"])
            grid.append(dict(alpha=a, lam=l, **m))
            if m["MRR"] > best[0]:
                best = (m["MRR"], (a, l))
    if verbose:
        print(f"  Tốt nhất trên val: α={best[1][0]}, λ={best[1][1]} (MRR={best[0]:.4f})")
    return best[1], grid

## 12. Giải thích và đo fidelity (Nhiệm vụ 3)

Giải thích = danh sách **đường đi cụ thể** đóng góp nhiều nhất vào `s_path` của ứng viên đó.
Phần đóng góp không phải suy đoán: nó được đo bằng cách **bỏ đúng đường đi đó rồi tính lại điểm**.

| Chỉ số | Định nghĩa | Kỳ vọng |
|---|---|---|
| fidelity− (cần thiết) | bỏ đường đi giải thích → Δ điểm, Δ thứ hạng | điểm giảm rõ, hạng tụt |
| fidelity+ (đủ) | **chỉ giữ** đường đi giải thích → Δ điểm | điểm gần như không đổi |
| sparsity | 1 − |đường đi giải thích| / |toàn bộ đường đi| | càng cao càng gọn |
| validity | đường đi có thật và nối đúng khái niệm truy vấn ↔ ảnh | 100% |
| **đối chứng** | bỏ **ngẫu nhiên** cùng số đường đi | phải ÍT ảnh hưởng hơn |

Dòng đối chứng là dòng quan trọng nhất: nếu bỏ đường đi giải thích không tệ hơn bỏ ngẫu nhiên
thì giải thích vô nghĩa. Đây gần như chắc chắn là câu hỏi vấn đáp.

In [15]:
def enumerate_paths(graph, img_pos_idx, concept_ids, cap=100000):
    """Liệt kê đường đi CỤ THỂ từ node ảnh tới các khái niệm truy vấn.
    Mỗi đường đi mang trọng số w = idf(node đích) × Π trọng số cạnh (P(đích|nguồn) nếu
    CFG.edge_weighting) — đúng trọng số trong graph.R, nên tổng theo mẫu khớp tuyệt đối
    với điểm tính bằng ma trận."""
    target = set(concept_ids)
    objs = graph.img_objects[img_pos_idx]
    own = set(objs)
    A_s, A_r, A_i = graph.A_scene, graph.A_related, graph.A_isa
    idf = graph.idf
    out = []

    def nbrs(M, n):
        sl = slice(M.indptr[n], M.indptr[n + 1])
        return zip(M.indices[sl], M.data[sl])

    def add(pattern, nodes, rel=None, ew=1.0):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return                                  # cùng quy tắc với ma trận R
        w = float(idf[c]) * float(ew)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=nodes, rel=rel, w=w))

    for o in objs:
        if o in target:
            add("P0_contains", [o])
        for o2, w1 in nbrs(A_s, o):
            rel = graph.scene_pred.get((o, o2), "quan hệ")
            if o2 in target:
                add("P1_scene", [o, o2], rel, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P4_scene_isa", [o, o2, c], rel, w1 * w2)
        for c, w1 in nbrs(A_i, o):
            if c in target:
                add("P2_isa", [o, c], None, w1)
        for o2, w1 in nbrs(A_r, o):
            if o2 in target:
                add("P3_related", [o, o2], None, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], None, w1 * w2)
        if len(out) > cap:
            break
    return out[:cap]


def paths_to_counts(paths):
    """Đặc trưng mẫu = log1p(Σ trọng số đường đi của mẫu) — giống hệt RetrievalEngine.path_feats."""
    v = np.zeros(len(PATTERNS), np.float64)
    for p in paths:
        v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_text(graph, p):
    n = [graph.id2name.get(x, f"#{x}") for x in p["nodes"]]
    t = PATTERN_DESC[p["pattern"]]
    d = dict(c=n[-1], o=n[0], rel=p.get("rel") or "quan_hệ")
    if len(n) >= 3:
        d["o2"] = n[1]
    return t.format(**d)


@torch.no_grad()
def explain_and_score(model, graph, img_idx, concept_ids, top_m=CFG.top_m_paths):
    """Trả về (giải thích, điểm đầy đủ, chi tiết fidelity) cho MỘT cặp (truy vấn, ảnh)."""
    paths = enumerate_paths(graph, img_idx, concept_ids)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts(ps) * w).sum())

    full = score(paths)
    # đóng góp biên của từng đường đi: bỏ đúng nó ra rồi tính lại
    contrib = []
    for i, p in enumerate(paths):
        rest = paths[:i] + paths[i + 1:]
        contrib.append(full - score(rest))
    order = np.argsort(-np.array(contrib)) if contrib else np.array([], int)
    expl_idx = list(order[:top_m])
    expl = [dict(text=path_text(graph, paths[i]), pattern=paths[i]["pattern"],
                 contribution=float(contrib[i])) for i in expl_idx]

    keep = [paths[i] for i in expl_idx]
    drop = [p for i, p in enumerate(paths) if i not in set(expl_idx)]
    # Đối chứng: bỏ NGẪU NHIÊN cùng số đường đi, lấy trung bình 10 lần bốc.
    # Bốc một lần thôi thì với truy vấn ít đường đi nó hay trùng đúng tập giải thích
    # và đối chứng mất ý nghĩa.
    rng = np.random.RandomState(0)
    rnd_scores = []
    for _ in range(10):
        ridx = set(rng.choice(len(paths), size=min(len(expl_idx), len(paths)),
                              replace=False).tolist()) if paths else set()
        rnd_scores.append(score([p for i, p in enumerate(paths) if i not in ridx]))
    rnd_score = float(np.mean(rnd_scores)) if rnd_scores else full

    detail = dict(
        n_paths=len(paths),
        score_full=full,
        score_without_explanation=score(drop),      # fidelity−
        score_only_explanation=score(keep),         # fidelity+
        score_without_random=rnd_score,             # ĐỐI CHỨNG (trung bình 10 lần)
        sparsity=1 - len(expl_idx) / max(len(paths), 1),
        validity=float(all(p["nodes"][-1] in set(concept_ids) for p in keep)) if keep else 0.0,
    )
    return expl, full, detail

## 13. Chạy thí nghiệm chính

Một hàm `run_experiment` cho cả 3 cấu hình ablation, mỗi cấu hình 3 seed.

In [16]:
_GROUP_CACHE = {}
PER_QUERY = {}          # (ablation, seed) → MRR từng truy vấn test; khoá "<ab>+adaptive" cho bản thích nghi
ART_BY_ABL = {}         # ablation → mô hình + đồ thị của seed đầu
FUSION_LOG = []         # nhật ký kết hợp thích nghi theo (ablation, seed)


def reciprocal_ranks(cand, scores, gold_pos):
    order = np.argsort(-scores, axis=1)
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold_pos)[:, None]
    return np.where(hit.any(1), 1.0 / (hit.argmax(1) + 1), 0.0)


def prepare(records, split, img_feat, txt_feat, ablation="none", seed=0):
    graph = MultimodalGraph(records, split["train"], ablation=ablation, seed=seed)
    # Nhóm trực tiếp/gián tiếp chốt MỘT LẦN trên đồ thị đầy đủ rồi dùng lại cho mọi ablation.
    if "groups" not in _GROUP_CACHE:
        base = graph if ablation == "none" else MultimodalGraph(records, split["train"])
        _GROUP_CACHE["groups"] = {q["gold"]: q["group"] for q in
                                  build_query_set(records, base, [r["image_id"] for r in records])}
    queries = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    for q in queries:
        q["group"] = _GROUP_CACHE["groups"].get(q["gold"], q["group"])
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, queries)
    pos = graph.img_pos
    idx = lambda ids: np.array([pos[i] for i in ids])
    parts = {}
    for name in ("train", "val", "test"):
        ids = split[name]
        parts[name] = dict(q_idx=idx(ids), gold_pos=idx(ids), img_idx=idx(ids),
                           groups=[queries[pos[i]]["group"] for i in ids])
    return graph, eng, parts


def evaluate_split(model, engine, part, alpha, lam, by_group=True):
    gallery = part["img_idx"]
    cand, sc, sg, sp_ = score_queries(model, engine, part["q_idx"], gallery)
    res = {}
    full = combine(sc, sg, sp_, alpha, lam)
    res["CLIP"] = retrieval_metrics(cand, sc, part["gold_pos"])
    res["CLIP+Graph"] = retrieval_metrics(cand, full, part["gold_pos"])
    if by_group:
        g = np.array(part["groups"])
        for grp in ("direct", "indirect"):
            m = g == grp
            if m.sum() >= 5:
                res[f"CLIP [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], sc[m], np.array(part["gold_pos"])[m])
                res[f"CLIP+Graph [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], full[m], np.array(part["gold_pos"])[m])
    return res, (cand, sc, sg, sp_)


# ============================ KẾT HỢP ĐIỂM THÍCH NGHI ============================
FUSION_FEATS = ["has_concept", "log_n_concepts", "path_coverage", "clip_margin"]


def fusion_features(engine, q_idx, cand, sc):
    """Đặc trưng của TRUY VẤN, không dùng nhãn đúng: có khái niệm không, bao nhiêu khái niệm,
    tỉ lệ ứng viên top-K có ≥1 đường đi, độ chênh top1–top2 của CLIP (CLIP tự tin thì ít cần đồ thị)."""
    ncon = np.array([len(engine.q_concepts[q]) for q in q_idx], np.float32)
    cov = (engine.path_feats(cand, q_idx).sum(-1) > 0).mean(1)
    srt = -np.sort(-sc, axis=1)
    return np.stack([(ncon > 0).astype(np.float32), np.log1p(ncon), cov,
                     srt[:, 0] - srt[:, 1]], 1).astype(np.float32)


def _fuse(a, l, sc, sg, sp):
    return a[:, None] * sc + (1 - a[:, None]) * (l[:, None] * sg + (1 - l[:, None]) * sp)


class AdaptiveFusion(nn.Module):
    def __init__(self, d, a0, l0, mu, sd):
        super().__init__()
        self.wa, self.wl = nn.Linear(d, 1), nn.Linear(d, 1)
        for lin, v in ((self.wa, a0), (self.wl, l0)):
            v = min(max(float(v), 0.02), 0.98)
            nn.init.zeros_(lin.weight); nn.init.constant_(lin.bias, math.log(v / (1 - v)))
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(sd, dtype=torch.float32))

    def gates(self, Xn):
        return torch.sigmoid(self.wa(Xn)).squeeze(-1), torch.sigmoid(self.wl(Xn)).squeeze(-1)

    @torch.no_grad()
    def score(self, X, sc, sg, sp):
        a, l = self.gates((torch.tensor(X) - self.mu) / self.sd)
        a, l = a.numpy(), l.numpy()
        return _fuse(a, l, sc, sg, sp), a, l


def fit_fusion(X, sc, sg, sp, cand, gold, a0, l0):
    """Listwise CE trên top-K ứng viên của VAL. Khởi tạo w=0 ⇒ xuất phát đúng bằng (α, λ) toàn cục."""
    gold = np.asarray(gold); hit = cand == gold[:, None]; keep = hit.any(1)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    m = AdaptiveFusion(X.shape[1], a0, l0, mu, sd)
    m.wa.bias.requires_grad_(False); m.wl.bias.requires_grad_(False)   # α(q) chỉ lệch khỏi α toàn cục theo đặc trưng
    if keep.sum() < 10:
        return m
    t = lambda a: torch.tensor(a[keep], dtype=torch.float32)
    Xn, SC, SG, SP = t((X - mu) / sd), t(sc), t(sg), t(sp)
    y = torch.tensor(hit[keep].argmax(1))
    opt = torch.optim.Adam([m.wa.weight, m.wl.weight], lr=0.05)
    for _ in range(CFG.fusion_steps):
        a, l = m.gates(Xn)
        loss = F.cross_entropy(_fuse(a, l, SC, SG, SP) / CFG.fusion_tau, y) \
            + CFG.fusion_wd * (m.wa.weight.pow(2).sum() + m.wl.weight.pow(2).sum())
        opt.zero_grad(); loss.backward(); opt.step()
    return m


def cv_fusion(X, sc, sg, sp, cand, gold, a0, l0, k=5, seed=0):
    """MRR ngoài-fold trên VAL: thích nghi vs (α, λ) toàn cục. Chống overfit khi chỉ có ~100–1000 truy vấn val."""
    gold = np.asarray(gold); n = len(X)
    idx = np.random.RandomState(seed).permutation(n)
    rr_a, rr_g = np.zeros(n), np.zeros(n)
    for f in np.array_split(idx, k):
        tr = np.setdiff1d(idx, f)
        m = fit_fusion(X[tr], sc[tr], sg[tr], sp[tr], cand[tr], gold[tr], a0, l0)
        rr_a[f] = reciprocal_ranks(cand[f], m.score(X[f], sc[f], sg[f], sp[f])[0], gold[f])
        rr_g[f] = reciprocal_ranks(cand[f], combine(sc[f], sg[f], sp[f], a0, l0), gold[f])
    return float(rr_a.mean()), float(rr_g.mean())


def adaptive_rows(fus, engine, part, raw):
    cand, sc, sg, sp_ = raw
    X = fusion_features(engine, part["q_idx"], cand, sc)
    s, a, l = fus.score(X, sc, sg, sp_)
    gold = np.asarray(part["gold_pos"])
    res = {"CLIP+Graph (adaptive)": retrieval_metrics(cand, s, gold)}
    g = np.array(part["groups"])
    for grp in ("direct", "indirect"):
        m = g == grp
        if m.sum() >= 5:
            res[f"CLIP+Graph (adaptive) [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], s[m], gold[m])
    return res, s, a, l, X


def _adaptive_step(model, eng, parts, alpha, lam, test_raw):
    pv = parts["val"]
    vc, vsc, vsg, vsp = score_queries(model, eng, pv["q_idx"], pv["img_idx"])
    Xv = fusion_features(eng, pv["q_idx"], vc, vsc)
    cv_a, cv_g = cv_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    fus = fit_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    ad_res, s_ad, a_q, l_q, Xt = adaptive_rows(fus, eng, parts["test"], test_raw)
    has = Xt[:, 0] > 0
    flog = dict(alpha=alpha, lam=lam, cv_global=cv_g, cv_adaptive=cv_a,
                alpha_q_with_concept=float(a_q[has].mean()) if has.any() else float("nan"),
                alpha_q_no_concept=float(a_q[~has].mean()) if (~has).any() else float("nan"),
                w_alpha=dict(zip(FUSION_FEATS, fus.wa.weight.detach().numpy().ravel().round(3).tolist())))
    return fus, flog, ad_res, s_ad


def run_experiment(records, split, img_feat, txt_feat, ablations=("none",), seeds=None):
    seeds = seeds or CFG.seeds
    _GROUP_CACHE.clear(); PER_QUERY.clear(); ART_BY_ABL.clear(); FUSION_LOG.clear()
    all_res = defaultdict(list)
    artefacts = {}
    for ab in ablations:
        print(f"\n{'='*66}\nABLATION = {ab}\n{'='*66}")
        for si, seed in enumerate(seeds):
            graph, eng, parts = prepare(records, split, img_feat, txt_feat, ab, seed)
            if si == 0:
                print("Đồ thị:", graph.summary())
            model, hist = train_graph_scorer(eng, parts["train"]["q_idx"],
                                             parts["train"]["img_idx"], seed=seed,
                                             verbose=(si == 0))
            (alpha, lam), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"],
                                             verbose=(si == 0))
            res, raw = evaluate_split(model, eng, parts["test"], alpha, lam)
            cand, sc, sg, sp_ = raw
            gold_t, groups_t = parts["test"]["gold_pos"], np.array(parts["test"]["groups"])
            rr_clip = reciprocal_ranks(cand, sc, gold_t)
            PER_QUERY[(ab, seed)] = dict(rr=reciprocal_ranks(cand, combine(sc, sg, sp_, alpha, lam), gold_t),
                                         rr_clip=rr_clip, groups=groups_t, lam=lam)
            fus, flog = None, None
            if CFG.adaptive_fusion:
                fus, flog, ad_res, s_ad = _adaptive_step(model, eng, parts, alpha, lam, raw)
                res.update(ad_res)
                FUSION_LOG.append(dict(ablation=ab, seed=seed, **flog))
                PER_QUERY[(f"{ab}+adaptive", seed)] = dict(rr=reciprocal_ranks(cand, s_ad, gold_t),
                                                           rr_clip=rr_clip, groups=groups_t, lam=lam)
            for k, v in res.items():
                all_res[f"{ab} | {k}"].append(v)
            if si == 0:
                ART_BY_ABL[ab] = dict(graph=graph, engine=eng, model=model, parts=parts, alpha=alpha,
                                      lam=lam, grid=grid, raw=raw, hist=hist, fusion=fus)
            if ab == "none" and si == 0:
                artefacts = ART_BY_ABL[ab]
            msg = (f"  seed {seed}: R@1={res['CLIP+Graph']['R@1']:.4f} "
                   f"MRR={res['CLIP+Graph']['MRR']:.4f} (α={alpha}, λ={lam})")
            if flog:
                msg += (f" | adaptive MRR={res['CLIP+Graph (adaptive)']['MRR']:.4f} "
                        f"[CV val {flog['cv_adaptive']:.4f} vs {flog['cv_global']:.4f}; "
                        f"α(q) có/không khái niệm {flog['alpha_q_with_concept']:.2f}/{flog['alpha_q_no_concept']:.2f}]")
            print(msg)
    return all_res, artefacts


def results_table(all_res):
    import pandas as pd
    rows = []
    for k, runs in all_res.items():
        ab, name = k.split(" | ", 1)
        rows.append(dict(ablation=ab, model=name, **fmt_metrics(runs)))
    return pd.DataFrame(rows)


def final_model_table(all_res, fusion_log):
    """MÔ HÌNH CUỐI = (cấu trúc đồ thị, global|adaptive) có MRR VAL cao nhất (adaptive tính ngoài-fold).
    Test chỉ được ĐỌC cho cấu hình đã chọn. Đây là con số chính đưa vào báo cáo."""
    import pandas as pd
    fl = pd.DataFrame(fusion_log)
    fl = fl[fl.ablation == "none"]          # ablation là phân tích, không phải ứng viên mô hình cuối
    sel = pd.concat([fl.groupby("ablation").cv_global.mean().rename("val_MRR").reset_index().assign(mode="global"),
                     fl.groupby("ablation").cv_adaptive.mean().rename("val_MRR").reset_index().assign(mode="adaptive")]
                    ).sort_values("val_MRR", ascending=False).reset_index(drop=True)
    best = sel.iloc[0]
    tab = results_table(all_res)
    pat = r"^CLIP\+Graph( \[|$)" if best["mode"] == "global" else r"^CLIP\+Graph \(adaptive\)"
    test = tab[(tab.ablation == best.ablation) & (tab.model.str.match(pat) | tab.model.str.match(r"^CLIP( \[|$)"))]
    print(f"→ Mô hình cuối (chọn trên VAL): đồ thị='{best.ablation}', kết hợp={best['mode']} "
          f"(VAL MRR {best.val_MRR:.4f})")
    return sel, test

## 14. Phân tích 5 thành công / 5 thất bại + fidelity trên ≥10 truy vấn

In [17]:
def analyse_cases(art, records, n_success=5, n_fail=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, (cand, sc, sg, sp_) = art["parts"]["test"], art["raw"]
    full = combine(sc, sg, sp_, art["alpha"], art["lam"])
    gold = np.array(part["gold_pos"])

    def rank_of(S):
        o = np.argsort(-S, 1); r = np.take_along_axis(cand, o, 1)
        hit = r == gold[:, None]
        return np.where(hit.any(1), hit.argmax(1) + 1, 10**9)

    r_clip, r_full = rank_of(sc), rank_of(full)
    delta = r_clip - r_full                                   # >0 = đồ thị giúp
    succ = [i for i in np.argsort(-delta) if r_full[i] <= 10][:n_success]
    fail = [i for i in np.argsort(delta) if r_clip[i] < r_full[i]][:n_fail]

    cases = []
    for kind, idxs in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idxs:
            qi = part["q_idx"][i]
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            expl, s, det = explain_and_score(model, graph, int(gold[i]), cids)
            cases.append(dict(kind=kind, query=eng.queries[qi]["text"][:110],
                              group=eng.queries[qi]["group"],
                              rank_clip=int(r_clip[i]), rank_graph=int(r_full[i]),
                              explanation=expl, detail=det))
    return cases


def fidelity_study(art, n_queries=CFG.n_explain_queries, top_m_paths_min=CFG.top_m_paths):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part = art["parts"]["test"]
    rows = []
    for i in range(len(part["q_idx"])):
        qi = part["q_idx"][i]
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        expl, s, det = explain_and_score(model, graph, int(part["gold_pos"][i]), cids)
        if det["n_paths"] == 0:
            continue
        if det["n_paths"] <= top_m_paths_min:   # quá ít đường đi ⇒ giải thích = toàn bộ, đối chứng vô nghĩa
            continue
        rows.append(dict(group=eng.queries[qi]["group"], **det,
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"]))
        if len(rows) >= n_queries:
            break
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không có truy vấn nào có đường đi — kiểm tra lại bước dựng đồ thị.")
        return df, {}
    summary = dict(
        n_queries=len(df),
        mean_paths=float(df.n_paths.mean()),
        fidelity_minus=float(df.drop_explanation.mean()),
        fidelity_minus_random_control=float(df.drop_random.mean()),
        fidelity_plus_gap=float((df.score_full - df.score_only_explanation).mean()),
        sparsity=float(df.sparsity.mean()),
        validity=float(df.validity.mean()),
    )
    summary["necessity_ratio"] = (summary["fidelity_minus"] /
                                  max(summary["fidelity_minus_random_control"], 1e-9))
    return df, summary

In [18]:
@torch.no_grad()
def final_rank_fidelity(art, n_queries=50, top_m=CFG.top_m_paths, n_rand=10):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, alpha, lam = art["parts"]["test"], art["alpha"], art["lam"]
    model.eval()
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    cand_all, clip_all = eng.clip_candidates(part["q_idx"], part["img_idx"], CFG.rerank_k)
    rng = np.random.RandomState(0)
    z = znorm_per_row
    sc = lambda ps: gain * float((paths_to_counts(ps) * w).sum())
    rows = []

    for i, qi in enumerate(part["q_idx"]):
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        gold = int(part["gold_pos"][i])
        cand = cand_all[i]
        hit = np.where(cand == gold)[0]
        if not cids or len(hit) == 0:
            continue
        paths = enumerate_paths(graph, gold, cids)
        if len(paths) <= top_m:          # ít đường đi quá thì đối chứng ngẫu nhiên vô nghĩa
            continue
        j = int(hit[0])
        pm = eng.path_feats(cand[None], np.array([qi]))[0]
        sp_raw = gain * (pm * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[qi]].to(DEVICE),
                                torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_clip = clip_all[i][None]

        def rank_with(kept):
            sp = sp_raw.copy()
            sp[j] = sc(kept)
            fin = combine(z(s_clip), z(s_gnn), z(sp[None]), alpha, lam)[0]
            return int((fin > fin[j]).sum()) + 1

        full = sc(paths)
        contrib = np.array([full - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib)[:top_m].tolist())

        r_rand = np.mean([
            rank_with([p for k, p in enumerate(paths)
                       if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
            for _ in range(n_rand)])
        rows.append(dict(group=eng.queries[qi]["group"], n_paths=len(paths),
                         rank_full=rank_with(paths),
                         rank_drop_expl=rank_with([p for k, p in enumerate(paths) if k not in top]),
                         rank_drop_rand=float(r_rand),
                         rank_only_expl=rank_with([paths[k] for k in top])))
        if len(rows) >= n_queries:
            break

    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không truy vấn nào có > top_m đường đi.")
        return df, {}
    summ = dict(n_queries=len(df),
                rank_full=df.rank_full.mean(),
                rank_after_drop_explanation=df.rank_drop_expl.mean(),
                rank_after_drop_random=df.rank_drop_rand.mean(),
                rank_with_only_explanation=df.rank_only_expl.mean(),
                frac_rank_worse_when_drop_expl=float((df.rank_drop_expl > df.rank_full).mean()),
                frac_rank_worse_when_drop_rand=float((df.rank_drop_rand > df.rank_full).mean()))
    return df, summ

## 15. Demo: nhập truy vấn → top-k + giải thích

In [19]:
@torch.no_grad()
def demo_query(text, art, k=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    gallery = art["parts"]["test"]["img_idx"]
    hits = parse_query(text, graph)
    print(f"\nTruy vấn: {text!r}")
    print(f"Khái niệm nhận ra: {[h[1] for h in hits] or '(không có)'}")
    cids = [n for n, _, _ in hits]

    # mã hoá truy vấn tự do: dùng CLIP nếu có, nếu không lấy caption gần nhất làm proxy
    q_emb = encode_free_text(text, eng)
    sim = (q_emb @ eng.img_feat[gallery].T).numpy()[0]
    top = np.argsort(-sim)[:CFG.rerank_k]
    cand = gallery[top]
    pm = torch.tensor(np.stack([_path_feats_free(graph, cand, cids)])).to(DEVICE)
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    s_gnn = model.gnn_score(h, q_emb.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    s = combine(znorm_per_row(sim[top][None]), znorm_per_row(s_gnn),
                znorm_per_row(s_path), art["alpha"], art["lam"])[0]
    order = np.argsort(-s)[:k]
    for rank, j in enumerate(order, 1):
        img_i = int(cand[j])
        rec = eng.records[img_i]
        expl, _, det = explain_and_score(model, graph, img_i, cids)
        print(f"\n  #{rank}  image_id={rec['image_id']}  score={s[j]:.4f}")
        print(f"      caption: {rec['caption'][:90]}")
        print(f"      đối tượng: {rec['objects'][:8]}")
        if expl:
            print(f"      GIẢI THÍCH ({det['n_paths']} đường đi, hiển thị {len(expl)}):")
            for e in expl:
                print(f"        • {e['text']}   [đóng góp {e['contribution']:+.4f}]")
        else:
            print("      (không có đường đi đồ thị — xếp hạng chỉ dựa trên CLIP)")


def _proxy_text(text, eng):
    toks = set(_norm(text).split())
    best, bi = -1, 0
    for i, q in enumerate(eng.queries):
        ov = len(toks & set(_norm(q["text"]).split()))
        if ov > best:
            best, bi = ov, i
    return eng.txt_feat[bi:bi + 1]


def encode_free_text(text, eng):
    """CLIP thật chỉ dùng khi đặc trưng ảnh cũng là CLIP thật; synthetic thì dùng proxy."""
    if dataset_name == "synthetic":
        return _proxy_text(text, eng)
    try:
        import open_clip
        if not hasattr(encode_free_text, "_m"):
            m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model,
                                                            pretrained=CFG.clip_weights)
            encode_free_text._m = m.to(DEVICE).eval()
            encode_free_text._t = open_clip.get_tokenizer(CFG.clip_model)
        with torch.no_grad():
            f = encode_free_text._m.encode_text(encode_free_text._t([text]).to(DEVICE))
        return F.normalize(f, dim=-1).float().cpu()
    except Exception:
        return _proxy_text(text, eng)

def _path_feats_free(graph, cand, cids):
    out = np.zeros((len(cand), len(PATTERNS)), np.float32)
    if not cids:
        return out
    cols = np.array(sorted(set(cids)))
    for p, name in enumerate(PATTERNS):
        out[:, p] = np.asarray(graph.R[name][cand][:, cols].sum(1)).ravel()
    return np.log1p(out)

## 16. Self-test — chạy trước khi tin bất kỳ con số nào

Ba kiểm tra này chính là ba câu hỏi vấn đáp dễ bị hỏi nhất. Chạy được và in ✓ thì
nhóm có bằng chứng ngay tại chỗ.

In [20]:
def self_test(records, split, img_feat, txt_feat):
    import copy
    ok = []

    # (1) RÒ RỈ — cấy một quan hệ chỉ tồn tại ở ảnh TEST, đồ thị train không được có
    rec2 = copy.deepcopy(records)
    pos = {r["image_id"]: i for i, r in enumerate(rec2)}
    A, B = "zzsecretobja", "zzsecretobjb"
    for iid in split["test"][:40]:
        r = rec2[pos[iid]]
        r["objects"] = sorted(set(r["objects"]) | {A, B})
        r["relations"] = list(r["relations"]) + [(A, "juggles", B)]
    gt = MultimodalGraph(rec2, split["train"])
    ga = MultimodalGraph(rec2, split["train"] + split["test"])
    e = lambda G, M: (M[G.obj2id[A], G.obj2id[B]] != 0) if A in G.obj2id else False
    c1 = (not e(gt, gt.A_scene)) and e(ga, ga.A_scene)
    c2 = (not e(gt, gt.A_related)) and e(ga, ga.A_related)
    ok.append(("Cạnh scene + đồng xuất hiện chỉ xây từ train", c1 and c2))

    # (2) TRUNG THỰC — điểm đường đi tính bằng ma trận phải khớp liệt kê từng đường đi
    graph = MultimodalGraph(records, split["train"])
    qs = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, qs)
    bad = n = 0
    for qi in range(min(300, len(qs))):
        cids = [x for x, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        n += 1
        m = eng.path_feats(np.array([[qi]]), np.array([qi]))[0, 0]
        v = paths_to_counts(enumerate_paths(graph, qi, cids))
        bad += int(not np.allclose(m, v, atol=1e-5))
    ok.append((f"Điểm đường đi khớp giải thích liệt kê ({n - bad}/{n} cặp)", bad == 0))

    # (3) SPLIT — không chồng lấn và tái lập được
    s2 = make_split(records, seed=CFG.split_seed)
    ok.append(("Split tái lập được từ seed", s2["sha1"] == split["sha1"]))

    print()
    for name, passed in ok:
        print(f"  {'✓' if passed else '✗ THẤT BẠI'}  {name}")
    assert all(p for _, p in ok), "Self-test không đạt — dừng lại và sửa trước khi chạy tiếp."
    return True

---
# THỰC THI

Từ đây trở xuống là phần chạy thật. Muốn chạy nhanh để thử, giảm `CFG.n_images`.

In [21]:
# ============ FLICKR8K: nạp dữ liệu, rút đối tượng/quan hệ, CLIP, split ============
import nltk
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    try: nltk.download(_pkg, quiet=True)
    except Exception: pass
from nltk.stem import WordNetLemmatizer

_LEM = WordNetLemmatizer(); _LEM_CACHE = {}
try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()          # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
except Exception as e:
    print("⚠ Không tải được POS tagger → dùng cách rút danh từ thô:", type(e).__name__)
    _TAGGER = None

def _lemma(w):
    r = _LEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "n")
        except Exception: r = w[:-1] if (w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[w] = r
    return r

NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())


def extract_objects_relations(captions):
    """Danh từ NHÌN THẤY ĐƯỢC (lọc WordNet) → lớp đối tượng. Quan hệ = (danh từ, động từ/giới từ,
    danh từ kế tiếp). Heuristic — nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    objs, rels = set(), set()
    for cap in captions:
        toks = re.findall(r"[A-Za-z]+", cap)
        if not toks:
            continue
        if _TAGGER is not None:
            tags = _TAGGER.tag(toks)
        else:
            tags = [(t, "NN" if len(t) > 2 and t.lower() not in STOP else "DT") for t in toks]
        last_n, preds = None, []
        for w, t in tags:
            wl = w.lower()
            if t in ("NN", "NNS") and len(wl) > 2 and wl not in NOT_OBJECTS and wl not in STOP:
                n = _lemma(wl)
                if n in NOT_OBJECTS or not is_visual_noun(n):
                    continue                     # tính từ/động từ bị tag nhầm NN: pink, walking, stand...
                objs.add(n)
                if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                    rels.add((last_n, " ".join(preds), n))
                last_n, preds = n, []
            elif (t.startswith("VB") or t in ("IN", "TO")) and wl not in _SKIP_P and last_n is not None:
                preds.append(wl)
    return objs, rels


def load_flickr8k():
    """Chỉ rút nhãn-từ-caption (cap_objects/cap_relations). objects/relations THẬT dùng cho
    đồ thị được gán sau bởi apply_flickr_annotation (tagger hoặc oracle)."""
    from datasets import load_dataset as hf_load
    dsd = hf_load("jxie/flickr8k")
    records, parts, iid = [], [], 0
    for hf_name, tag in (("train", "train"), ("validation", "val"), ("test", "test")):
        d = dsd[hf_name]; parts.append(d)
        for row in d.remove_columns("image"):
            allc = [str(row[f"caption_{k}"]).strip() for k in range(5)]
            objs, rels = extract_objects_relations(allc[1:])   # caption 1..4; caption_0 là truy vấn
            records.append(dict(image_id=iid, caption=allc[0], captions=allc,
                                cap_objects=sorted(objs), cap_relations=sorted(rels),
                                objects=[], relations=[], url="", split=tag))
            iid += 1
    print(f"Flickr8k: {len(records)} ảnh | {np.mean([len(r['cap_objects']) for r in records]):.1f} danh từ/ảnh | "
          f"{np.mean([len(r['cap_relations']) for r in records]):.1f} quan hệ/ảnh (rút từ caption)")
    for r in records[:3]:
        print("  •", r["caption"], "→", r["cap_objects"][:8], r["cap_relations"][:3])
    return records, parts


def encode_flickr8k(records, parts):
    cache = CFG.out_dir / "cache" / f"clip_flickr8k_{CFG.clip_model}_{CFG.clip_weights}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    img, txt = [], []
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), 128):
                ims = d[s:s + 128]["image"]
                x = torch.stack([preprocess(im.convert("RGB")) for im in ims]).to(DEVICE)
                img.append(F.normalize(model.encode_image(x), dim=-1).float().cpu().numpy())
        caps = [r["caption"] for r in records]
        for s in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[s:s + CFG.clip_bs]).to(DEVICE)
            txt.append(F.normalize(model.encode_text(tok), dim=-1).float().cpu().numpy())
    del model; torch.cuda.empty_cache()
    img, txt = np.concatenate(img), np.concatenate(txt)
    assert len(img) == len(records) == len(txt)
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


# =====================================================================================
# NHÃN ẢNH BẰNG CLIP ZERO-SHOT (thay cho nhãn rút từ caption của chính ảnh)
#   - Từ vựng: danh từ nhìn thấy được trong caption TRAIN, tần suất ≥ min_obj_freq.
#   - Điểm: cos(ảnh, prompt-ensemble); chuẩn hoá theo từng từ bằng mean/std trên ảnh TRAIN.
#   - (cách chuẩn hoá, k) chọn trên VAL theo F1 so với danh từ trong caption val.
#   - Caption của ảnh val/test KHÔNG bao giờ vào đồ thị; caption test chỉ dùng để BÁO CÁO P/R/F1.
# =====================================================================================
@torch.no_grad()
def clip_text_features(texts, bs=512):
    import open_clip
    if not hasattr(clip_text_features, "_m"):
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        clip_text_features._m = m.to(DEVICE).eval()
        clip_text_features._t = open_clip.get_tokenizer(CFG.clip_model)
    m, tok = clip_text_features._m, clip_text_features._t
    out = [F.normalize(m.encode_text(tok(texts[s:s + bs]).to(DEVICE)), dim=-1).float().cpu().numpy()
           for s in range(0, len(texts), bs)]
    return np.concatenate(out)


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1); R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def clip_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vset = set(vocab)

    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = img_feat @ T.T                                          # [N, V]
    mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên train
    scores = {"raw": S, "z": (S - mu) / sd, "raw+z": S / S[tr].std() + (S - mu) / sd}

    def tags(Sc, idx, k):
        top = np.argpartition(-Sc[idx], k, axis=1)[:, :k]
        return [{vocab[j] for j in row} for row in top]
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]

    grid, best = [], None
    for mode, Sc in scores.items():
        for k in CFG.tag_k_grid:
            P, R, F1 = _prf(tags(Sc, va, k), truth(va))
            grid.append(dict(norm=mode, k=k, val_P=P, val_R=R, val_F1=F1))
            if best is None or F1 > best[0]:
                best = (F1, mode, k)
    _, mode, k = best
    all_tags = tags(scores[mode], np.arange(len(records)), k)
    P, R, F1 = _prf([all_tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm=mode, k=k, test_P=P, test_R=R, test_F1=F1, grid=grid)
    return all_tags, info


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    """Gán records[i]['objects'] / ['relations'] dùng cho đồ thị.
    relations chỉ giữ cho ảnh TRAIN (dùng để xây cạnh scene thống kê)."""
    mode = mode or CFG.flickr_annot
    tr = set(split["train"])
    info = dict(annotation=mode)
    if mode == "tagger":
        global TAG_INFO
        if not all("tags" in r for r in records) or "TAG_INFO" not in globals():
            tags, TAG_INFO = clip_tagger(records, split, img_feat, text_encoder)
            for r, t in zip(records, tags):
                r["tags"] = sorted(t)
        info.update(TAG_INFO)
    for r in records:
        r["objects"] = list(r["tags"]) if mode == "tagger" else list(r["cap_objects"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    msg = f"Nhãn ảnh = {mode}"
    if mode == "tagger":
        msg += (f" | vocab={info['vocab_size']} | chuẩn hoá={info['norm']}, k={info['k']} (chọn trên val)"
                f" | TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    print(msg)
    return info


# Split chính thức của HF; bản ngẫu nhiên cũ giữ lại cho trường hợp synthetic.
if "_make_split_random" not in globals():
    _make_split_random = make_split

def make_split(records, seed=CFG.split_seed):
    if not all("split" in r for r in records):
        return _make_split_random(records, seed)
    split = {k: sorted(r["image_id"] for r in records if r["split"] == k) for k in ("train", "val", "test")}
    split["seed"] = "official-hf-split"
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split chính thức HF (sha1={split['sha1']}): train={len(split['train'])} "
          f"val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])) and not (set(split["val"]) & set(split["test"]))
    return split


# Truy vấn Flickr có số nhiều ("dogs") → phải lemmatize như lúc rút đối tượng.
def parse_query(text, graph):
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits

In [22]:
# records, dataset_name = load_dataset()
# split = make_split(records)
# img_feat, txt_feat = encode_with_clip(records, dataset_name)
# print(f"\nDữ liệu: {dataset_name} | {len(records)} ảnh | đặc trưng CLIP {img_feat.shape}")

In [23]:
import pandas as pd
records, hf_parts = load_flickr8k()
dataset_name = "flickr8k"
split = make_split(records)
img_feat, txt_feat = encode_flickr8k(records, hf_parts)
annot_info = apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)
print(f"\nDữ liệu: {dataset_name} | {len(records)} ảnh | đặc trưng CLIP {img_feat.shape}")

if CFG.flickr_annot == "tagger":
    print("\nChọn (chuẩn hoá, k) của tagger trên VALIDATION:")
    print(pd.DataFrame(annot_info["grid"]).pivot_table(index="k", columns="norm",
                                                         values="val_F1").round(3).to_string())
    for r in records[-3:]:                       # vài ảnh TEST: nhãn tagger vs danh từ trong caption
        print(f"  • {r['caption'][:70]}\n      tagger : {r['objects']}\n      caption: {r['cap_objects']}")
with open(CFG.out_dir / "results" / "annotation_info.json", "w") as f:
    json.dump({k: v for k, v in annot_info.items()}, f, indent=2)

README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-7025a2b59(…): reconstructing file:   0%|          |  0.00B /  138MB            

... (6 outputs left out)

Flickr8k: 8000 ảnh | 6.1 danh từ/ảnh | 4.8 quan hệ/ảnh (rút từ caption)
  • A black dog is running after a white dog in the snow . → ['body', 'dog', 'ground', 'snow', 'water'] [('body', 'of', 'water'), ('dog', 'chase across', 'ground'), ('dog', 'play in', 'snow')]
  • A little baby plays croquet . → ['ball', 'boy', 'car', 'child', 'croquet', 'croquette', 'girl', 'hammer'] [('ball', 'beside', 'car'), ('boy', 'playing with', 'croquet'), ('car', 'with', 'ball')]
  • A brown dog in the snow has something hot pink in its mouth . → ['dog', 'hat', 'mouth', 'shirt', 'snow'] [('dog', 'carrying in', 'mouth'), ('dog', 'holding', 'shirt'), ('dog', 'in holding', 'hat')]
Split chính thức HF (sha1=005202c69674): train=6000 val=1000 test=1000


open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

Nhãn ảnh = tagger | vocab=685 | chuẩn hoá=raw+z, k=8 (chọn trên val) | TEST P=0.215 R=0.299 F1=0.250

Dữ liệu: flickr8k | 8000 ảnh | đặc trưng CLIP (8000, 768)

Chọn (chuẩn hoá, k) của tagger trên VALIDATION:
norm    raw  raw+z      z
k                        
3     0.171  0.204  0.215
5     0.203  0.242  0.237
8     0.221  0.248  0.232
10    0.217  0.245  0.224
12    0.212  0.239  0.217
  • A basketball player performing a lay-up .
      tagger : ['athlete', 'basket', 'basketball', 'hoop', 'opponent', 'player', 'teammate', 'wheeler']
      caption: ['ball', 'basketball', 'block', 'boy', 'jersey', 'man', 'player', 'team']
  • A man who has a gray beard and gray hair laughs while wearing a purple
      tagger : ['artist', 'beard', 'cheek', 'grey', 'guy', 'hair', 'male', 'man']
      caption: ['beard', 'crowd', 'hair', 'man', 'people', 'shirt']
  • A big dog stands on his hand leg as tennis balls are thrown his direct
      tagger : ['ball', 'boxer', 'dog', 'greyhound', 'jumping', 'snowb

In [24]:
self_test(records, split, img_feat, txt_feat)

WordNet(+fallback): 628/679 lớp đối tượng có khái niệm cha
WordNet(+fallback): 628/681 lớp đối tượng có khái niệm cha
WordNet(+fallback): 628/679 lớp đối tượng có khái niệm cha
Split chính thức HF (sha1=005202c69674): train=6000 val=1000 test=1000

  ✓  Cạnh scene + đồng xuất hiện chỉ xây từ train
  ✓  Điểm đường đi khớp giải thích liệt kê (296/296 cặp)
  ✓  Split tái lập được từ seed


True

In [25]:
# Chọn CFG.learn_clip_w trên VALIDATION (1 seed, cấu hình 'none'). Không nhìn test.
def pick_learn_clip_w(records, split, img_feat, txt_feat, seed=0):
    rows = []
    for flag in (False, True):
        CFG.learn_clip_w = flag
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(records, split, img_feat, txt_feat, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        rows.append(dict(learn_clip_w=flag, val_MRR=max(r["MRR"] for r in grid), alpha=a, lam=l,
                         clip_w=round(F.softplus(model.clip_logit).item(), 3),
                         path_w=round(F.softplus(model.path_logit).item(), 3)))
    df_ = pd.DataFrame(rows); display(df_)
    CFG.learn_clip_w = bool(df_.loc[df_.val_MRR.idxmax(), "learn_clip_w"])
    print("→ CFG.learn_clip_w =", CFG.learn_clip_w, "(chọn theo VAL MRR)")
    return df_

clipw_flickr = pick_learn_clip_w(records, split, img_feat, txt_feat)

WordNet(+fallback): 628/679 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 6012, 'indirect': 1553, 'other': 435}
WordNet(+fallback): 628/679 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 6012, 'indirect': 1553, 'other': 435}


,learn_clip_w,val_MRR,alpha,lam,clip_w,path_w
0,False,0.804763,0.7,0.75,0.546,1.0
1,True,0.805526,0.7,0.75,0.546,1.0


→ CFG.learn_clip_w = True (chọn theo VAL MRR)


In [26]:
# Thí nghiệm chính: 5 cấu hình ablation × 3 seed, nhãn ảnh = CFG.flickr_annot
all_res, art = run_experiment(records, split, img_feat, txt_feat, ablations=CFG.ablations)
df = results_table(all_res)
df.to_csv(CFG.out_dir / "results" / f"main_results_{CFG.flickr_annot}.csv", index=False)
df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
display(df)


ABLATION = none
WordNet(+fallback): 628/679 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 6012, 'indirect': 1553, 'other': 435}
Đồ thị: {'nodes': 9108, 'images': 8000, 'object_classes': 679, 'concepts': 429, 'edges': 70121, 'contains': 63974, 'scene': 1485, 'related': 3389, 'isa': 1273, 'paths': {'P0_contains': 63974, 'P1_scene': 142732, 'P2_isa': 94399, 'P3_related': 238476, 'P4_scene_isa': 238984, 'P5_related_isa': 365250}, 'ablation': 'none'}
  cạnh dùng khi huấn luyện: 108258/140242 (đã bỏ 31984 cạnh chạm ảnh val/test)
  epoch  0 | loss 27.4332 | w_path 1.000 0.331 0.279 0.328 0.207 0.223 | gate L0[scene,rel,isa] 0.19 0.20 0.22 | clip_w 0.86
  epoch  3 | loss 18.7836 | w_path 1.000 0.742 0.239 0.571 0.472 0.401 | gate L0[scene,rel,isa] 0.20 0.20 0.21 | clip_w 0.73
  epoch  6 | loss 14.8845 | w_path 1.000 0.865 0.199 0.588 0.692 0.458 | gate L0[scene,rel,isa] 0.22 0.21 0.22 | clip_w 0.69
  epoch  9 | loss 11.3585 | w_path 1.000 0.910 0.173 0.591 0.823 0.482 | gate L0[sce

,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.9690 ± 0.0000,0.7996 ± 0.0000,0.8394 ± 0.0000
1,none,CLIP+Graph,0.7170 ± 0.0033,0.9160 ± 0.0022,0.9690 ± 0.0016,0.8056 ± 0.0030,0.8438 ± 0.0026
2,none,CLIP [direct] (n=745),0.7248 ± 0.0000,0.9195 ± 0.0000,0.9664 ± 0.0000,0.8140 ± 0.0000,0.8498 ± 0.0000
3,none,CLIP+Graph [direct] (n=745),0.7432 ± 0.0033,0.9235 ± 0.0019,0.9718 ± 0.0011,0.8247 ± 0.0030,0.8593 ± 0.0025
4,none,CLIP [indirect] (n=200),0.6450 ± 0.0000,0.8900 ± 0.0000,0.9700 ± 0.0000,0.7517 ± 0.0000,0.8028 ± 0.0000
5,none,CLIP+Graph [indirect] (n=200),0.6283 ± 0.0062,0.8900 ± 0.0071,0.9550 ± 0.0041,0.7395 ± 0.0049,0.7893 ± 0.0048
6,none,CLIP+Graph (adaptive),0.7103 ± 0.0079,0.9123 ± 0.0009,0.9647 ± 0.0017,0.8005 ± 0.0052,0.8387 ± 0.0043
7,none,CLIP+Graph (adaptive) [direct] (n=745),0.7369 ± 0.0097,0.9239 ± 0.0025,0.9691 ± 0.0011,0.8206 ± 0.0063,0.8554 ± 0.0047
8,none,CLIP+Graph (adaptive) [indirect] (n=200),0.6233 ± 0.0062,0.8717 ± 0.0062,0.9433 ± 0.0085,0.7331 ± 0.0058,0.7811 ± 0.0063
9,random_edges,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.9690 ± 0.0000,0.7996 ± 0.0000,0.8394 ± 0.0000


In [27]:
# =====================================================================================
# ABLATION THEO GIẢI THÍCH: giải thích có đúng với tri thức ĐỘC LẬP với đồ thị không?
#
# Với mỗi ablation (seed đầu), giải thích của ẢNH ĐÚNG trên mọi truy vấn test = top-m đường đi
# QUAN HỆ (P1..P5, bỏ P0) theo đóng góp biên — đúng định nghĩa explain_and_score. Mỗi CẠNH
# trong đường đi được chấm bằng nguồn KHÔNG dùng để dựng đồ thị:
#   • scene / related  : NPMI của hai đầu cạnh trên tập nhãn tham chiếu của ảnh VAL+TEST
#                        (Flickr: danh từ trong caption 1..4 — caption 0 là truy vấn nên bỏ;
#                         AG: annotation của video). Đồ thị chỉ thống kê trên TRAIN ⇒ độc lập.
#                        Cạnh "hợp lý" ⇔ NPMI > 0 (hai thứ hay đi cùng nhau hơn ngẫu nhiên).
#   • isa              : đúng ⇔ đích nằm trong bao đóng hypernym WordNet (hoặc FALLBACK_ISA).
# Chỉ số:
#   edge_ok    % cạnh hợp lý            path_ok  % đường đi có MỌI cạnh hợp lý
#   grounded   % đường đi mà mọi node ĐỐI TƯỢNG trung gian có trong nhãn tham chiếu của ảnh đó
#   rand_path_ok  path_ok khi chọn NGẪU NHIÊN cùng số đường đi quan hệ của chính ảnh đó
#                 ⇒ (path_ok − rand_path_ok) = cơ chế chọn giải thích có ưu tiên đường đi đúng không
# Không rò rỉ: nhãn tham chiếu chỉ dùng để CHẤM, không huấn luyện / chọn siêu tham số.
# Chạy NGAY sau cell thí nghiệm chính (cần ART_BY_ABL của cell 28 bản đợt 4).
# =====================================================================================
from functools import lru_cache


def reference_sets(records, ids):
    key = "cap_objects" if "cap_objects" in records[0] else "objects"
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    return key, {i: set(records[pos[i]][key]) for i in ids}


def make_npmi(docs):
    n = len(docs)
    c1 = Counter(x for d in docs for x in d)
    c2 = Counter(p for d in docs for p in itertools.combinations(sorted(d), 2))
    def npmi(a, b):
        ab = c2.get(tuple(sorted((a, b))), 0)
        if ab == 0 or c1[a] == 0 or c1[b] == 0:
            return -1.0
        p_ab = ab / n
        return math.log(p_ab / ((c1[a] / n) * (c1[b] / n))) / max(-math.log(p_ab), 1e-9)
    return npmi


@lru_cache(maxsize=None)
def _isa_ok(a, b):
    if b in FALLBACK_ISA.get(a, []):
        return True
    wn = _wordnet()
    if wn is None:
        return False
    for s in wn.synsets(a.replace(" ", "_"), pos="n")[:3]:
        for h in s.closure(lambda x: x.hypernyms()):
            if b in {l.name().replace("_", " ").lower() for l in h.lemmas()}:
                return True
    return False


_EDGE_KIND = {"P1_scene": ["scene"], "P2_isa": ["isa"], "P3_related": ["related"],
              "P4_scene_isa": ["scene", "isa"], "P5_related_isa": ["related", "isa"]}


def _edges(p):
    nodes = p["nodes"]                      # [o, c] hoặc [o, o2, c]
    return list(zip(_EDGE_KIND[p["pattern"]], nodes[:-1], nodes[1:]))


def _rel_contrib(paths, model):
    """Đóng góp biên của từng đường đi = đúng công thức explain_and_score, nhưng O(P)."""
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    pidx = np.array([PIDX[p["pattern"]] for p in paths])
    pw = np.array([p["w"] for p in paths])
    tot = np.bincount(pidx, weights=pw, minlength=len(PATTERNS))
    return gain * w[pidx] * (np.log1p(tot[pidx]) - np.log1p(np.maximum(tot[pidx] - pw, 0)))


def explanation_grounding(art_by_abl, records, split, top_m=CFG.top_m_paths, n_rand=10, seed=0):
    key, ref = reference_sets(records, split["val"] + split["test"])
    npmi = make_npmi(list(ref.values()))
    rng = np.random.RandomState(seed)
    rows = []
    for ab, art in art_by_abl.items():
        g, eng, model, part = art["graph"], art["engine"], art["model"], art["parts"]["test"]
        name = lambda n: g.id2name.get(n, "")
        is_obj = lambda n: g.n_img <= n < g.n_img + g.n_obj
        edge_cache = {}

        def edge_ok(kind, a, b):
            k = (kind, a, b)
            if k not in edge_cache:
                edge_cache[k] = (_isa_ok(name(a), name(b)) if kind == "isa"
                                 else npmi(name(a), name(b)) > 0)
            return edge_cache[k]

        def path_ok(p):
            return all(edge_ok(*e) for e in _edges(p))

        for i, qi in enumerate(part["q_idx"]):
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            if not cids:
                continue
            gold = int(part["gold_pos"][i])
            rel = [p for p in enumerate_paths(g, gold, cids) if p["pattern"] != "P0_contains"]
            if not rel:
                continue
            top = [rel[j] for j in np.argsort(-_rel_contrib(rel, model))[:top_m]]
            truth = ref.get(records[gold]["image_id"], set())
            k = len(top)
            rand_ok = np.mean([np.mean([path_ok(rel[j]) for j in rng.choice(len(rel), k, replace=False)])
                               for _ in range(n_rand)])
            edges = [e for p in top for e in _edges(p)]
            rows.append(dict(
                ablation=ab, group=part["groups"][i], n_rel_paths=len(rel), n_expl=k,
                edge_ok=np.mean([edge_ok(*e) for e in edges]),
                path_ok=np.mean([path_ok(p) for p in top]),
                rand_path_ok=rand_ok,
                grounded=np.mean([all(name(n) in truth for n in p["nodes"][:-1] if is_obj(n))
                                  for p in top]),
                mean_npmi=np.mean([npmi(name(a), name(b)) for kd, a, b in edges if kd != "isa"] or [np.nan])))
    df = pd.DataFrame(rows)
    print(f"Nhãn tham chiếu: '{key}' của {len(ref)} ảnh VAL+TEST | top_m={top_m}")
    return df


def _boot_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = x[np.random.RandomState(seed).randint(0, len(x), (n_boot, len(x)))].mean(1)
    return f"{x.mean():.3f} [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]"


def grounding_table(df):
    out = []
    for grp in ("all", "direct", "indirect"):
        for ab in df.ablation.unique():
            d = df[(df.ablation == ab) & ((df.group == grp) | (grp == "all"))]
            if len(d) < 5:
                continue
            out.append(dict(ablation=ab, group=grp, n_queries=len(d),
                            rel_paths_per_q=round(d.n_rel_paths.mean(), 1),
                            edge_ok=_boot_ci(d.edge_ok), path_ok=_boot_ci(d.path_ok),
                            rand_path_ok=_boot_ci(d.rand_path_ok),
                            selection_lift=_boot_ci(d.path_ok - d.rand_path_ok),
                            grounded=_boot_ci(d.grounded), mean_npmi=round(d.mean_npmi.mean(), 3)))
    return pd.DataFrame(out)


# ------------------------------ Flickr8k ------------------------------
ART_FLICKR = dict(ART_BY_ABL)                     # giữ lại trước khi cell oracle / AG ghi đè
gdf = explanation_grounding(ART_FLICKR, records, split)
gtab = grounding_table(gdf)
gdf.to_csv(CFG.out_dir / "results" / "explanation_grounding_per_query.csv", index=False)
gtab.to_csv(CFG.out_dir / "results" / "explanation_grounding.csv", index=False)
display(gtab)

Nhãn tham chiếu: 'cap_objects' của 2000 ảnh VAL+TEST | top_m=5


,ablation,group,n_queries,rel_paths_per_q,edge_ok,path_ok,rand_path_ok,selection_lift,grounded,mean_npmi
0,none,all,638,5.8,"0.739 [0.718, 0.761]","0.690 [0.665, 0.714]","0.649 [0.625, 0.674]","0.040 [0.031, 0.050]","0.279 [0.255, 0.304]",-0.080
1,random_edges,all,278,1.4,"0.267 [0.221, 0.316]","0.249 [0.203, 0.297]","0.249 [0.203, 0.297]","-0.000 [-0.000, -0.000]","0.143 [0.105, 0.184]",-0.566
2,no_scene,all,477,5.3,"0.705 [0.677, 0.732]","0.621 [0.587, 0.653]","0.568 [0.534, 0.601]","0.053 [0.041, 0.067]","0.143 [0.121, 0.166]",-0.206
3,no_related,all,515,2.6,"0.834 [0.808, 0.859]","0.826 [0.799, 0.852]","0.826 [0.799, 0.851]","0.001 [-0.001, 0.003]","0.381 [0.349, 0.414]",0.111
4,no_concept,all,587,3.6,"0.751 [0.725, 0.775]","0.751 [0.725, 0.775]","0.747 [0.722, 0.772]","0.004 [-0.000, 0.008]","0.326 [0.299, 0.352]",0.025
5,none,direct,438,5.4,"0.744 [0.716, 0.770]","0.699 [0.668, 0.729]","0.668 [0.637, 0.698]","0.031 [0.021, 0.042]","0.330 [0.298, 0.361]",-0.063
6,random_edges,direct,186,1.4,"0.253 [0.199, 0.310]","0.233 [0.178, 0.290]","0.233 [0.178, 0.290]","-0.000 [-0.000, -0.000]","0.180 [0.129, 0.233]",-0.571
7,no_scene,direct,301,5.0,"0.700 [0.666, 0.734]","0.614 [0.571, 0.654]","0.568 [0.526, 0.610]","0.045 [0.031, 0.060]","0.158 [0.128, 0.188]",-0.213
8,no_related,direct,361,2.6,"0.837 [0.807, 0.865]","0.829 [0.799, 0.858]","0.829 [0.799, 0.858]","0.000 [-0.001, 0.003]","0.442 [0.405, 0.480]",0.117
9,no_concept,direct,402,3.4,"0.756 [0.724, 0.785]","0.756 [0.724, 0.785]","0.755 [0.724, 0.785]","0.001 [-0.004, 0.005]","0.382 [0.349, 0.416]",0.025


In [28]:
# Kiểm định CẶP giữa các ablation trên cùng truy vấn test (trung bình 3 seed / truy vấn).
# Chênh MRR 0.000x với std 0.002-0.003 thì KHÔNG kết luận được gì — cần CI cho hiệu số.
# Chạy NGAY sau cell thí nghiệm chính (cell oracle / AG sẽ ghi đè PER_QUERY).
def paired_test(pq, a="none", b="no_relations", n_boot=10000, seed=0):
    seeds = sorted({s for ab, s in pq if ab == a} & {s for ab, s in pq if ab == b})
    ra = np.mean([pq[(a, s)]["rr"] for s in seeds], 0)
    rb = np.mean([pq[(b, s)]["rr"] for s in seeds], 0)
    groups = pq[(a, seeds[0])]["groups"]
    rng = np.random.RandomState(seed)
    rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(ra), bool) if grp == "all" else groups == grp
        d = (ra - rb)[m]
        if len(d) < 5:
            continue
        boot = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(group=grp, n=int(m.sum()), MRR_a=ra[m].mean(), MRR_b=rb[m].mean(),
                         diff=d.mean(), ci_lo=lo, ci_hi=hi,
                         p_a_not_better=float((boot <= 0).mean()),
                         a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum())))
    return pd.DataFrame(rows).assign(a=a, b=b)


PQ_MAIN = dict(PER_QUERY)            # giữ lại trước khi cell sau ghi đè
sig = pd.concat([paired_test(PQ_MAIN, "none", b)
                 for b in ("no_relations", "random_edges", "no_scene", "no_related", "no_concept")
                 if any(k[0] == b for k in PQ_MAIN)])
sig.to_csv(CFG.out_dir / "results" / "paired_ablation_test.csv", index=False)
print("λ chọn trên val cho 'none':", [v["lam"] for k, v in PQ_MAIN.items() if k[0] == "none"],
      "(λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)")
display(sig.round(4))

λ chọn trên val cho 'none': [0.75, 0.75, 0.75] (λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1000,0.8056,0.8014,0.0042,-0.0012,0.0096,0.0609,127,133,none,no_relations
1,direct,745,0.8247,0.8199,0.0048,-0.0014,0.0111,0.0640,88,90,none,no_relations
2,indirect,200,0.7395,0.7364,0.0032,-0.0091,0.0155,0.3142,32,35,none,no_relations
0,all,1000,0.8056,0.8019,0.0037,-0.0018,0.0092,0.0987,143,132,none,random_edges
1,direct,745,0.8247,0.8188,0.0059,-0.0003,0.0123,0.0321,101,92,none,random_edges
2,indirect,200,0.7395,0.7441,-0.0046,-0.0179,0.0087,0.7501,33,34,none,random_edges
0,all,1000,0.8056,0.8056,0.0000,-0.0050,0.0050,0.5031,129,126,none,no_scene
1,direct,745,0.8247,0.8218,0.0029,-0.0027,0.0086,0.1507,97,81,none,no_scene
2,indirect,200,0.7395,0.7478,-0.0082,-0.0219,0.0047,0.8900,30,36,none,no_scene
0,all,1000,0.8056,0.8025,0.0031,-0.0020,0.0082,0.1133,122,120,none,no_related


In [29]:
# Mô hình cuối của Flickr8k chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_FLICKR = list(FUSION_LOG)
sel_f, final_f = final_model_table(all_res, FUSION_FLICKR)
display(sel_f.round(4)); display(final_f)
display(pd.DataFrame(FUSION_FLICKR).round(3))
sig_ad = paired_test(dict(PER_QUERY), "none+adaptive", "none")
display(sig_ad.round(4))
for name, obj in (("final_model_val_selection.csv", sel_f), ("final_model_test.csv", final_f),
                  ("fusion_log.csv", pd.DataFrame(FUSION_FLICKR)), ("paired_adaptive_vs_global.csv", sig_ad)):
    obj.to_csv(CFG.out_dir / "results" / name, index=False)

→ Mô hình cuối (chọn trên VAL): đồ thị='none', kết hợp=global (VAL MRR 0.8061)


,ablation,val_MRR,mode
0,none,0.8061,global
1,none,0.8000,adaptive


,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.9690 ± 0.0000,0.7996 ± 0.0000,0.8394 ± 0.0000
1,none,CLIP+Graph,0.7170 ± 0.0033,0.9160 ± 0.0022,0.9690 ± 0.0016,0.8056 ± 0.0030,0.8438 ± 0.0026
2,none,CLIP [direct] (n=745),0.7248 ± 0.0000,0.9195 ± 0.0000,0.9664 ± 0.0000,0.8140 ± 0.0000,0.8498 ± 0.0000
3,none,CLIP+Graph [direct] (n=745),0.7432 ± 0.0033,0.9235 ± 0.0019,0.9718 ± 0.0011,0.8247 ± 0.0030,0.8593 ± 0.0025
4,none,CLIP [indirect] (n=200),0.6450 ± 0.0000,0.8900 ± 0.0000,0.9700 ± 0.0000,0.7517 ± 0.0000,0.8028 ± 0.0000
5,none,CLIP+Graph [indirect] (n=200),0.6283 ± 0.0062,0.8900 ± 0.0071,0.9550 ± 0.0041,0.7395 ± 0.0049,0.7893 ± 0.0048


,ablation,seed,alpha,lam,cv_global,cv_adaptive,alpha_q_with_concept,alpha_q_no_concept,w_alpha
0,none,0,0.7,0.75,0.807,0.801,0.681,0.991,"{'has_concept': -0.1899999976158142, 'log_n_co..."
1,none,1,0.8,0.75,0.807,0.801,0.741,0.992,"{'has_concept': -0.19200000166893005, 'log_n_c..."
2,none,2,0.7,0.75,0.805,0.798,0.680,0.988,"{'has_concept': -0.17900000512599945, 'log_n_c..."
3,random_edges,0,0.7,0.75,0.804,0.800,0.675,0.991,"{'has_concept': -0.1889999955892563, 'log_n_co..."
4,random_edges,1,0.7,0.50,0.805,0.800,0.678,0.992,"{'has_concept': -0.20399999618530273, 'log_n_c..."
5,random_edges,2,0.7,0.75,0.805,0.799,0.679,0.991,"{'has_concept': -0.1889999955892563, 'log_n_co..."
6,no_relations,0,0.7,0.75,0.807,0.804,0.672,0.986,"{'has_concept': -0.24799999594688416, 'log_n_c..."
7,no_relations,1,0.7,0.50,0.806,0.802,0.671,0.995,"{'has_concept': -0.296999990940094, 'log_n_con..."
8,no_relations,2,0.8,0.50,0.806,0.799,0.736,0.995,"{'has_concept': -0.2800000011920929, 'log_n_co..."
9,no_scene,0,0.7,0.75,0.808,0.800,0.675,0.994,"{'has_concept': -0.20000000298023224, 'log_n_c..."


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1000,0.8005,0.8056,-0.0051,-0.0093,-0.0010,0.9940,93,121,none+adaptive,none
1,direct,745,0.8206,0.8247,-0.0042,-0.0087,0.0003,0.9654,68,76,none+adaptive,none
2,indirect,200,0.7331,0.7395,-0.0065,-0.0179,0.0055,0.8612,25,36,none+adaptive,none


In [30]:
# So sánh với nhãn ORACLE (rút từ caption 1..4 của chính ảnh — cách của bản trước).
# Chỉ để định lượng "rò rỉ" của cách cũ: chênh lệch oracle − tagger chính là phần điểm
# đến từ việc ứng viên mang sẵn mô tả văn bản của chính nó. KHÔNG báo cáo là kết quả chính.
RUN_ORACLE = True
if RUN_ORACLE:
    apply_flickr_annotation(records, split, img_feat, "oracle")
    oracle_res, _ = run_experiment(records, split, img_feat, txt_feat,
                                   ablations=("none", "no_relations"))
    df_oracle = results_table(oracle_res)
    df_oracle.to_csv(CFG.out_dir / "results" / "main_results_oracle.csv", index=False)
    apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)   # KHÔI PHỤC nhãn chính
    cmp_annot = pd.concat([df[df.ablation.isin(["none", "no_relations"])].assign(annotation=CFG.flickr_annot),
                           df_oracle.assign(annotation="oracle")])
    cmp_annot = cmp_annot[cmp_annot.model.isin(["CLIP", "CLIP+Graph"])]
    cmp_annot.to_csv(CFG.out_dir / "results" / "annotation_comparison.csv", index=False)
    display(cmp_annot[["annotation", "ablation", "model", "R@1", "R@5", "MRR"]])

Nhãn ảnh = oracle

ABLATION = none
WordNet(+fallback): 632/685 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 7492, 'indirect': 386, 'other': 122}
Đồ thị: {'nodes': 9112, 'images': 8000, 'object_classes': 685, 'concepts': 427, 'edges': 49992, 'contains': 46037, 'scene': 1501, 'related': 1173, 'isa': 1281, 'paths': {'P0_contains': 46037, 'P1_scene': 212107, 'P2_isa': 77043, 'P3_related': 220401, 'P4_scene_isa': 368530, 'P5_related_isa': 334531}, 'ablation': 'none'}
  cạnh dùng khi huấn luyện: 76972/99984 (đã bỏ 23012 cạnh chạm ảnh val/test)
  epoch  0 | loss 19.5372 | w_path 1.000 0.202 0.228 0.281 0.146 0.164 | gate L0[scene,rel,isa] 0.18 0.17 0.16 | clip_w 0.76
  epoch  3 | loss 14.6039 | w_path 1.000 0.123 0.232 0.374 0.159 0.250 | gate L0[scene,rel,isa] 0.20 0.14 0.13 | clip_w 0.59
  epoch  6 | loss 11.6167 | w_path 1.000 0.086 0.221 0.417 0.165 0.270 | gate L0[scene,rel,isa] 0.19 0.14 0.13 | clip_w 0.55
  epoch  9 | loss 9.0026 | w_path 1.000 0.065 0.190 0.431 0.155 0.266

,annotation,ablation,model,R@1,R@5,MRR
0,tagger,none,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.7996 ± 0.0000
1,tagger,none,CLIP+Graph,0.7170 ± 0.0033,0.9160 ± 0.0022,0.8056 ± 0.0030
18,tagger,no_relations,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.7996 ± 0.0000
19,tagger,no_relations,CLIP+Graph,0.7090 ± 0.0008,0.9200 ± 0.0014,0.8014 ± 0.0015
0,oracle,none,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.7996 ± 0.0000
1,oracle,none,CLIP+Graph,0.7280 ± 0.0016,0.9260 ± 0.0024,0.8159 ± 0.0006
9,oracle,no_relations,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.7996 ± 0.0000
10,oracle,no_relations,CLIP+Graph,0.7293 ± 0.0005,0.9283 ± 0.0009,0.8160 ± 0.0005


In [31]:
# Fidelity đo trên THỨ HẠNG của điểm cuối (CLIP + GNN + path)
rdf, rsumm = final_rank_fidelity(art)
rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
for k, v in rsumm.items():
    print(f"  {k:36s} {v:.3f}" if isinstance(v, float) else f"  {k:36s} {v}")
with open(CFG.out_dir / "results" / "xai_final_rank_summary.json", "w") as f:
    json.dump(rsumm, f, indent=2)

  n_queries                            50
  rank_full                            2.540
  rank_after_drop_explanation          3.040
  rank_after_drop_random               2.880
  rank_with_only_explanation           2.700
  frac_rank_worse_when_drop_expl       0.300
  frac_rank_worse_when_drop_rand       0.320


**Cách đọc bảng.**

1. `none / CLIP` vs `none / CLIP+Graph` — đồ thị có giúp không (Nhiệm vụ 1).
2. `none` vs `random_edges` / `no_relations` — cải thiện có đến từ **cấu trúc quan hệ** không.
   Chỉ kết luận "quan hệ có ích" khi `none` > `random_edges` **và** `none` > `no_relations`
   vượt quá độ lệch chuẩn giữa các seed.
3. `no_scene` / `no_concept` — loại cạnh nào (scene graph rút từ caption hay IsA WordNet) đóng góp.
4. Dòng `[indirect]` — nhóm duy nhất mà đường đi P1–P5 có thể giúp; P0 không tồn tại cho nhóm này.

Nếu `no_relations` vẫn ≥ `none`, kết luận trung thực là: lợi ích đến từ nhãn đối tượng (P0)
và GNN, còn quan hệ trong đồ thị hiện tại chưa giúp — kèm phân tích lý do ở mục case study.

In [32]:
# Độ nhạy theo α (yêu cầu ở cột Mở rộng của Nhiệm vụ 2)
import pandas as pd
gdf = pd.DataFrame(art["grid"])
piv = gdf.pivot_table(index="alpha", columns="lam", values="MRR")
print("MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:")
print(piv.round(4).to_string())
piv.to_csv(CFG.out_dir / "results" / "alpha_sensitivity.csv")

MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:
lam      0.00    0.25    0.50    0.75
alpha                                
0.0    0.2004  0.2425  0.2865  0.1964
0.2    0.4182  0.5315  0.6227  0.5453
0.4    0.6165  0.7071  0.7650  0.7460
0.5    0.6925  0.7539  0.7860  0.7884
0.6    0.7425  0.7776  0.7942  0.8002
0.7    0.7689  0.7913  0.7970  0.8065
0.8    0.7844  0.7908  0.8006  0.8044
0.9    0.7915  0.7953  0.7993  0.8029
1.0    0.7973  0.7973  0.7973  0.7973


In [33]:
# Phân tích 5 thành công + 5 thất bại
cases = analyse_cases(art, records)
for c in cases:
    print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → CLIP+Graph {c['rank_graph']}")
    print(f"  truy vấn: {c['query']}")
    for e in c["explanation"]:
        print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    d = c["detail"]
    print(f"  fidelity−: {d['score_full'] - d['score_without_explanation']:+.4f} | "
          f"đối chứng ngẫu nhiên: {d['score_full'] - d['score_without_random']:+.4f} | "
          f"sparsity {d['sparsity']:.2f}")
with open(CFG.out_dir / "results" / "case_analysis.json", "w") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)


[THÀNH CÔNG] nhóm=indirect | hạng CLIP 18 → CLIP+Graph 9
  truy vấn: A boy looks suspiciously at someone nearby .
    • ảnh –chứa→ young –liên_quan→ boy   [+0.4841]
    • ảnh –chứa→ youth –liên_quan→ boy   [+0.2521]
  fidelity−: +1.0197 | đối chứng ngẫu nhiên: +1.0197 | sparsity 0.00

[THÀNH CÔNG] nhóm=direct | hạng CLIP 15 → CLIP+Graph 6
  truy vấn: A child plays on playground equipment .
    • ảnh –chứa→ child   [+1.7176]
    • ảnh –chứa→ girl –for→ camera –là_một→ equipment   [+0.6502]
    • ảnh –chứa→ child –liên_quan→ playground   [+0.1720]
    • ảnh –chứa→ kid –liên_quan→ playground   [+0.1584]
    • ảnh –chứa→ toddler –liên_quan→ playground   [+0.1572]
  fidelity−: +3.0045 | đối chứng ngẫu nhiên: +2.1224 | sparsity 0.50

[THÀNH CÔNG] nhóm=direct | hạng CLIP 11 → CLIP+Graph 2
  truy vấn: a lone skateboarder jumping a huge skateboarding ramp .
    • ảnh –chứa→ ramp   [+0.7764]
    • ảnh –chứa→ skateboarding   [+0.7294]
    • ảnh –chứa→ skateboarding –liên_quan→ skateboarder   [+0

In [34]:
# Fidelity / validity / sparsity trên ≥10 truy vấn (cấp độ 3)
fdf, summ = fidelity_study(art)
fdf.to_csv(CFG.out_dir / "results" / "fidelity_per_query.csv", index=False)
print("Tổng hợp xAI:")
for k, v in summ.items():
    print(f"  {k:34s} {v:.4f}" if isinstance(v, float) else f"  {k:34s} {v}")
print("\nnecessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên")
print("cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.")
with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
    json.dump(summ, f, indent=2)

Tổng hợp xAI:
  n_queries                          20
  mean_paths                         9.1500
  fidelity_minus                     2.5580
  fidelity_minus_random_control      1.7263
  fidelity_plus_gap                  0.4871
  sparsity                           0.4018
  validity                           1.0000
  necessity_ratio                    1.4818

necessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên
cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.


In [35]:
# Bộ ≥20 truy vấn đánh giá (cấp độ 2) — CHỈ lấy truy vấn của ảnh test
eval_qs = export_eval_queries(art["engine"].queries, art["graph"],
                              CFG.out_dir / "results" / "eval_queries_20.json", n=20,
                              allowed_ids=split["test"])

Đã ghi 20 truy vấn đánh giá → /kaggle/working/results/eval_queries_20.json


In [36]:
# Demo
demo_query("a child playing football on the grass", art, k=3)
demo_query("a person cooking food in a kitchen", art, k=3)


Truy vấn: 'a child playing football on the grass'
Khái niệm nhận ra: ['child', 'football', 'grass']

  #1  image_id=7180  score=2.5811
      caption: A boy soccer player running down the field .
      đối tượng: ['athlete', 'boy', 'defender', 'football', 'footballer', 'player', 'referee', 'soccer']
      GIẢI THÍCH (6 đường đi, hiển thị 5):
        • ảnh –chứa→ football   [đóng góp +1.9904]
        • ảnh –chứa→ boy –liên_quan→ child   [đóng góp +0.8510]
        • ảnh –chứa→ boy –liên_quan→ toddler –là_một→ child   [đóng góp +0.4232]
        • ảnh –chứa→ football –in→ grass   [đóng góp +0.1211]
        • ảnh –chứa→ soccer –on→ grass   [đóng góp +0.1065]

  #2  image_id=7547  score=2.1849
      caption: A group of young boys playing soccer .
      đối tượng: ['field', 'football', 'footballer', 'formation', 'opponent', 'referee', 'soccer', 'teammate']
      GIẢI THÍCH (5 đường đi, hiển thị 5):
        • ảnh –chứa→ football   [đóng góp +1.9904]
        • ảnh –chứa→ field –of→ grass   [đón

---
## Nhiệm vụ 2 — GNN của SCENIR thay GNN cổ điển (Flickr8k)

Cùng `run_experiment`, cùng split, nhãn tagger và ứng viên CLIP; chỉ đổi `CFG.gnn_type = "scenir"` (mục 7.2).

In [ ]:
with use_gnn("scenir"):
    sc_res, sc_art = run_experiment(records, split, img_feat, txt_feat,
                                    ablations=("none", "random_edges", "no_relations"))
sc_df = results_table(sc_res)
sc_df.to_csv(CFG.out_dir / "results" / f"main_results_scenir_{CFG.flickr_annot}.csv", index=False)
display(sc_df)

In [ ]:
# GNN cổ điển vs SCENIR, tách nhóm trực tiếp / gián tiếp
cmp_gnn = pd.concat([df.assign(gnn=CFG.gnn_type), sc_df.assign(gnn="scenir")])
cmp_gnn = cmp_gnn[cmp_gnn.ablation.isin(["none", "random_edges", "no_relations"])]
cmp_gnn.to_csv(CFG.out_dir / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

In [37]:
# Gói kết quả để nộp
print("\nTệp đã ghi:")
for p in sorted((CFG.out_dir / "results").glob("*")) + [CFG.out_dir / "split.json"]:
    print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")


Tệp đã ghi:
  /kaggle/working/results/alpha_sensitivity.csv  (0.7 KB)
  /kaggle/working/results/annotation_comparison.csv  (0.9 KB)
  /kaggle/working/results/annotation_info.json  (2.3 KB)
  /kaggle/working/results/case_analysis.json  (10.1 KB)
  /kaggle/working/results/eval_queries_20.json  (4.1 KB)
  /kaggle/working/results/explanation_grounding.csv  (2.3 KB)
  /kaggle/working/results/explanation_grounding_per_query.csv  (152.4 KB)
  /kaggle/working/results/fidelity_final_rank.csv  (1.1 KB)
  /kaggle/working/results/fidelity_per_query.csv  (2.9 KB)
  /kaggle/working/results/final_model_test.csv  (0.7 KB)
  /kaggle/working/results/final_model_val_selection.csv  (0.1 KB)
  /kaggle/working/results/fusion_log.csv  (4.5 KB)
  /kaggle/working/results/main_results.csv  (6.4 KB)
  /kaggle/working/results/main_results_oracle.csv  (2.1 KB)
  /kaggle/working/results/main_results_tagger.csv  (6.4 KB)
  /kaggle/working/results/paired_ablation_test.csv  (2.2 KB)
  /kaggle/working/results/paired_a

# Action Genome

In [40]:
import shutil, os
DRIVE_AG = "/content/drive/MyDrive/DLDT/dataset/action-genome"
DATA_PATH = "/content/ag_local"

_nfiles = lambda d: sum(len(fs) for _, _, fs in os.walk(d)) if Path(d).exists() else 0
n_src, n_dst = _nfiles(DRIVE_AG), _nfiles(DATA_PATH)
if n_dst < n_src:
    print(f"Bản cục bộ thiếu ({n_dst}/{n_src} tệp) → chép từ Drive")
    shutil.copytree(DRIVE_AG, DATA_PATH, dirs_exist_ok=True)
    n_dst = _nfiles(DATA_PATH)
assert n_dst >= n_src, f"Chép chưa đủ: {n_dst}/{n_src} tệp — chạy lại cell này"
print(f"AG cục bộ OK: {n_dst} tệp")

Bản cục bộ thiếu (9504/38239 tệp) → chép từ Drive
AG cục bộ OK: 38239 tệp


In [41]:
# =====================================================================================
# CẤP ĐỘ 2 — DATASET THỨ HAI: ACTION GENOME (text-to-video)
# Chạy toàn bộ pipeline trên AG mà KHÔNG ghi đè kết quả Flickr8k.
# =====================================================================================
import os
import nltk
from contextlib import contextmanager
from nltk.stem import WordNetLemmatizer
import pandas as pd
for _p in ("wordnet", "omw-1.4"):
    try: nltk.download(_p, quiet=True)
    except Exception: pass

_AG_LEM = WordNetLemmatizer(); _AG_LEM_CACHE = {}

def _ag_lemma(w, pos="n"):
    k = (w, pos)
    if k not in _AG_LEM_CACHE:
        try: _AG_LEM_CACHE[k] = _AG_LEM.lemmatize(w, pos)
        except Exception: _AG_LEM_CACHE[k] = w
    return _AG_LEM_CACHE[k]

# ---- Từ vựng AG: phần lấy từ dữ liệu vs phần viết tay (ghi rõ trong báo cáo) ----
_ag_hits = glob.glob(f"{DATA_PATH}/**/ag_records.json", recursive=True)
assert _ag_hits, "Chưa Add Input output của notebook ag-download"
_ag_payload = json.load(open(_ag_hits[0]))

# (1) Quan hệ người–vật → động từ gốc. Ánh xạ thủ công 1-1 từ tên lớp AG; đã kiểm tra bên dưới.
#     Bỏ: quan hệ không gian, quan hệ phủ định/mơ hồ, covered_by (ngược chiều ngữ nghĩa).
AG_PRED2VERB = {"looking_at": "look", "holding": "hold", "carrying": "carry",
                "have_it_on_the_back": "carry", "drinking_from": "drink", "eating": "eat",
                "sitting_on": "sit", "standing_on": "stand", "lying_on": "lie", "leaning_on": "lean",
                "touching": "touch", "wearing": "wear", "wiping": "wipe", "writing_on": "write",
                "twisting": "twist"}
_pred_seen = Counter(p for r in _ag_payload["records"] for _, p, _ in r["triples"])
print("Predicate có trong dữ liệu nhưng KHÔNG dùng:",
      {p: n for p, n in _pred_seen.items() if p not in AG_PRED2VERB})
_bad = [p for p in AG_PRED2VERB if p not in _pred_seen]
assert not _bad, f"Khóa không tồn tại trong annotation AG: {_bad}"

# (2) Alias danh từ: SINH TỰ ĐỘNG từ tên lớp nhiều tên của AG (vd 'cup/glass/bottle')
_ag_classes = {c for r in _ag_payload["records"] for c in r["objects"]}
AG_NOUN_ALIAS = {_ag_lemma(alt.strip().lower()): _ag_lemma(c.split("/")[0].strip().lower())
                 for c in _ag_classes if "/" in c for alt in c.split("/")[1:]}
print("Alias danh từ sinh từ dữ liệu:", AG_NOUN_ALIAS)

# (3) Alias động từ: VIẾT TAY, căn cứ là tần suất động từ trong mô tả của tập TRAIN
AG_VERB_ALIAS = {"watch": "look", "grab": "hold"}
_AG_VERBS = set(AG_PRED2VERB.values())
_train_verbs = Counter(
    _ag_lemma(t, "v") for r in _ag_payload["records"] if r["split"] == "train"
    for t in _norm(r["caption"]).split()
    if t not in STOP and _ag_lemma(t, "v") != t)        # xấp xỉ: token có dạng biến tố động từ
print("Động từ phổ biến trên TRAIN chưa được ánh xạ:",
      [(v, n) for v, n in _train_verbs.most_common(40)
       if v not in _AG_VERBS and v not in AG_VERB_ALIAS][:20])


def ag_canon(cls):
    """'cup/glass/bottle' → 'cup'; 'groceries' → 'grocery' (khớp lemma phía truy vấn)."""
    return _ag_lemma(cls.split("/")[0].strip().lower().replace("_", " "))


def load_action_genome():
    root = Path(_ag_hits[0]).parent
    out = []
    for i, r in enumerate(_ag_payload["records"]):
        objs = {ag_canon(c) for c in r["objects"]} - {"person"}   # video nào cũng có người
        comps, rels = set(), set()
        for kind, pred, cls in r["triples"]:
            v, o = AG_PRED2VERB.get(pred), ag_canon(cls)
            if v is None or o == "person":
                continue
            c = f"{v} {o}"                       # node ghép, vd 'hold cup'
            comps.add(c)
            rels.add((o, "afford", c))           # cup –afford→ hold cup (gộp CHỈ trên train)
        out.append(dict(image_id=i, video_id=r["video_id"], split=r["split"],
                        caption=r["caption"], captions=r["descriptions"],
                        objects=sorted(objs | comps), relations=sorted(rels), url="",
                        frames=[str(root / "ag_frames" / f) for f in r["frames"]]))
    print(f"Action Genome: {len(out)} video | "
          f"{np.mean([len(r['objects']) for r in out]):.1f} node/video | "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/video | "
          f"{np.mean([len(r['frames']) for r in out]):.1f} frame/video")
    for r in out[:3]:
        print("  •", r["caption"][:80], "→", r["objects"][:8])
    return out


def encode_action_genome(recs):
    """CLIP từng frame → mean-pool theo thời gian → L2. Văn bản = mô tả Charades."""
    cache = CFG.out_dir / "cache" / f"clip_ag_{CFG.clip_model}_{CFG.clip_weights}_{len(recs)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model,
                                                                 pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    items = [(i, p) for i, r in enumerate(recs) for p in r["frames"]]
    missing = [p for _, p in items if not os.path.exists(p)]
    assert len(missing) <= 0.01 * len(items), \
        f"{len(missing)}/{len(items)} frame không tồn tại (vd {missing[:2]}) — kiểm tra DATA_PATH"
    class FrameDS(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, k):
            try: im = Image.open(items[k][1]).convert("RGB")
            except Exception: im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    feats = []
    with torch.no_grad():
        from tqdm.auto import tqdm
        log(f"CLIP: mã hoá {len(items)} frame của {len(recs)} video (batch {CFG.clip_bs})")
        for x in tqdm(DataLoader(FrameDS(), batch_size=CFG.clip_bs, num_workers=2),
                      desc="CLIP frame", unit="batch"):
            feats.append(F.normalize(model.encode_image(x.to(DEVICE)), dim=-1).float().cpu().numpy())
        feats = np.concatenate(feats)
        img = np.zeros((len(recs), feats.shape[1]), np.float32)
        np.add.at(img, np.array([i for i, _ in items]), feats)
        img /= np.linalg.norm(img, axis=1, keepdims=True).clip(1e-8)
        caps = [r["caption"] for r in recs]
        # Mô tả Charades thường dài hơn 77 token ⇒ open_clip cắt cụt. Tách theo câu, mã hoá từng câu,
        # lấy trung bình. Giữ cả bản cắt cụt; chọn bản nào bằng MRR CLIP trên VAL (không nhìn test).
        pieces, owner = [], []
        for i, t in enumerate(caps):
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip().split()) >= 2] or [t]
            pieces += sents; owner += [i] * len(sents)
        enc = lambda xs: np.concatenate([F.normalize(model.encode_text(tokenizer(xs[s:s + CFG.clip_bs]).to(DEVICE)),
                                                     dim=-1).float().cpu().numpy()
                                         for s in range(0, len(xs), CFG.clip_bs)])
        txt_trunc = enc(caps)
        E = enc(pieces); txt_chunk = np.zeros_like(txt_trunc)
        np.add.at(txt_chunk, np.array(owner), E)
        txt_chunk /= np.linalg.norm(txt_chunk, axis=1, keepdims=True)
        va = np.array([i for i, r in enumerate(recs) if r["split"] == "val"])
        mrr = lambda T: reciprocal_ranks(np.tile(va, (len(va), 1)), T[va] @ img[va].T, va).mean()
        m_t, m_c = mrr(txt_trunc), mrr(txt_chunk)
        assert max(m_t, m_c) > 0.02, \
            f"CLIP VAL MRR {max(m_t, m_c):.4f} ≈ ngẫu nhiên — frame hỏng hoặc ảnh/văn bản lệch thứ tự"
        txt = txt_chunk if m_c > m_t else txt_trunc
        log(f"Văn bản AG — VAL MRR CLIP: cắt cụt {m_t:.4f} | theo câu {m_c:.4f} → dùng "
            f"{'theo câu' if m_c > m_t else 'cắt cụt'}")
    log(f"CLIP xong: ảnh {img.shape}, văn bản {txt.shape} → cache {cache.name}")
    del model; torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


def ag_make_split(recs, seed=None):
    """Split lấy từ ag_records.json: train/test gốc của AG, val tách bằng seed cố định."""
    sp_ = {k: sorted(r["image_id"] for r in recs if r["split"] == k) for k in ("train", "val", "test")}
    sp_["seed"] = "ag-official-train-test + val seed 20240101"
    payload = json.dumps({k: sp_[k] for k in ("train", "val", "test")}, sort_keys=True)
    sp_["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(sp_, f)
    print(f"Split AG (sha1={sp_['sha1']}): train={len(sp_['train'])} "
          f"val={len(sp_['val'])} test={len(sp_['test'])}")
    assert not (set(sp_["train"]) & set(sp_["test"])) and not (set(sp_["val"]) & set(sp_["test"]))
    return sp_


def ag_parse_query(text, graph):
    """'A person is drinking from a glass' → ['drink', 'cup'] → khớp 'drink cup' và 'cup'."""
    toks = []
    for t in _norm(text).split():
        if t in STOP or len(t) < 2:
            continue
        v = _ag_lemma(t, "v"); v = AG_VERB_ALIAS.get(v, v)
        if v in _AG_VERBS:
            toks.append(v); continue
        n = _ag_lemma(t, "n"); toks.append(AG_NOUN_ALIAS.get(n, n))
    cands = [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


@contextmanager
def use_dataset(out_dir, **overrides):
    """Tạm thay hàm toàn cục + thư mục output; khôi phục nguyên trạng khi ra khỏi khối."""
    g = globals()
    saved = {k: g.get(k) for k in overrides}
    saved_out = CFG.out_dir
    g.update(overrides)
    CFG.out_dir = Path(out_dir)
    for sub in ("cache", "results"):
        (CFG.out_dir / sub).mkdir(parents=True, exist_ok=True)
    _GROUP_CACHE.clear()
    try:
        yield
    finally:
        for k, v in saved.items():
            if v is None: g.pop(k, None)
            else: g[k] = v
        CFG.out_dir = saved_out
        _GROUP_CACHE.clear()


# (4) IsA cho node ghép 'động từ vật' → khái niệm hành động 'động từ' (vd 'hold cup' –là_một→ 'hold').
#     WordNet chỉ phủ 31/202 lớp AG nên tầng khái niệm gần như rỗng; đây là tri thức NGOÀI
#     (cấu trúc tên lớp), không phải thống kê ⇒ an toàn cho test. Truy vấn "someone holding
#     something" giờ có đường đi P2/P4/P5 tới 'hold' thay vì không khớp node nào.
#     Ghi vào FALLBACK_ISA toàn cục để mọi cell AG sau (semantic linking) dùng cùng một đồ thị;
#     tên 'động từ vật' không bao giờ xuất hiện trong Flickr8k nên không ảnh hưởng phần trước.
AG_ACTION_ISA = {f"{v} {ag_canon(cls)}": [v]
                 for r in _ag_payload["records"] for _, pred, cls in r["triples"]
                 for v in [AG_PRED2VERB.get(pred)] if v is not None and ag_canon(cls) != "person"}
FALLBACK_ISA.update(AG_ACTION_ISA)
print(f"IsA hành động: {len(AG_ACTION_ISA)} node ghép → {len(set(AG_PRED2VERB.values()))} khái niệm động từ")

# ------------------------------------ LOG TIẾN ĐỘ ------------------------------------
# Mọi dòng có [mm:ss từ đầu cell | +ss so với dòng trước] và flush ngay (Colab hay giữ buffer).
# Các hàm nặng của pipeline được BỌC lại qua use_dataset(...) — chỉ có hiệu lực trong khối
# with, tự khôi phục khi ra khỏi khối, không sửa cell nào khác.
_T0 = _TL = time.time()


def log(msg):
    global _TL
    now = time.time()
    m, s_ = divmod(int(now - _T0), 60)
    print(f"[{m:02d}:{s_:02d} | +{now - _TL:5.1f}s] {msg}", flush=True)
    _TL = now


@contextmanager
def step(title):
    log(f"▶ {title}")
    t = time.time()
    yield
    log(f"✔ {title} ({time.time() - t:.1f}s)")


_N_ABL = len(CFG.ablations); _N_SEED = len(CFG.seeds)
_PROG = dict(abl=None, i_abl=0, i_seed=0)
_orig = dict(prepare=prepare, train_graph_scorer=train_graph_scorer, tune_on_val=tune_on_val,
             evaluate_split=evaluate_split, self_test=self_test)


def _prepare_logged(records, split, img_feat, txt_feat, ablation="none", seed=0):
    if ablation != _PROG["abl"]:
        _PROG.update(abl=ablation, i_abl=_PROG["i_abl"] + 1, i_seed=0)
    _PROG["i_seed"] += 1
    tag = f"[ablation {_PROG['i_abl']}/{_N_ABL} '{ablation}' | seed {_PROG['i_seed']}/{_N_SEED}]"
    with step(f"{tag} dựng đồ thị + bộ truy vấn"):
        out = _orig["prepare"](records, split, img_feat, txt_feat, ablation, seed)
    g = out[0]
    log(f"    nodes={g.n_nodes} | scene={g.A_scene.nnz} related={g.A_related.nnz} isa={g.A_isa.nnz}"
        f" | đường đi P1..P5={sum(g.R[p].nnz for p in PATTERNS[1:])}")
    return out


def _train_logged(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    with step(f"  huấn luyện GNN seed={seed} ({CFG.epochs} epoch, {len(train_q_idx)} truy vấn)"):
        model, hist = _orig["train_graph_scorer"](engine, train_q_idx, train_img_idx,
                                                  seed=seed, verbose=verbose)
    gates = model.edge_gates()
    log(f"    loss {hist[0]:.3f} → {hist[-1]:.3f} | clip_w {F.softplus(model.clip_logit).item():.2f}"
        f" | path_w {F.softplus(model.path_logit).item():.2f}"
        f" | w_path " +
        " ".join(f"{x:.3f}" for x in model.pattern_weights().detach().cpu().numpy()) +
        (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4]) if gates else ""))
    return model, hist


def _tune_logged(model, engine, val_q, val_gallery, verbose=True):
    with step(f"  quét (α, λ) trên VAL ({len(CFG.alpha_grid) * len(CFG.lambda_grid)} cặp)"):
        (a, l), grid = _orig["tune_on_val"](model, engine, val_q, val_gallery, verbose=False)
    best = max(grid, key=lambda r: r["MRR"])
    log(f"    chọn α={a}, λ={l} | VAL MRR={best['MRR']:.4f} (CLIP thuần: "
        f"{next(r['MRR'] for r in grid if r['alpha'] == 1.0):.4f})"
        + ("  ⚠ λ=1 ⇒ s_path không tham gia điểm cuối" if l == 1.0 else ""))
    return (a, l), grid


def _eval_logged(model, engine, part, alpha, lam, by_group=True):
    res, raw = _orig["evaluate_split"](model, engine, part, alpha, lam, by_group)
    log(f"    TEST: CLIP MRR={res['CLIP']['MRR']:.4f} → CLIP+Graph MRR={res['CLIP+Graph']['MRR']:.4f}")
    return res, raw


def _self_test_logged(*a, **k):
    with step("self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu"):
        return _orig["self_test"](*a, **k)


# ------------------------------------ CHẠY ------------------------------------
AG_OUT = Path("/kaggle/working/output/action_genome")
log(f"Bắt đầu Action Genome | ablations={CFG.ablations} | seeds={CFG.seeds} | epochs={CFG.epochs}"
    f" | device={DEVICE}")
with use_dataset(AG_OUT, parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged,
                 self_test=_self_test_logged):
    R = CFG.out_dir / "results"

    with step("1/9 nạp annotation AG"):
        ag_records = load_action_genome()
    with step("2/9 split"):
        ag_split = make_split(ag_records)
    with step("3/9 đặc trưng CLIP (dùng cache nếu có)"):
        ag_img, ag_txt = encode_action_genome(ag_records)
    with step("4/9 self-test"):
        self_test(ag_records, ag_split, ag_img, ag_txt)
    if "pick_learn_clip_w" in globals():
        with step("4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0)"):
            clipw_ag = pick_learn_clip_w(ag_records, ag_split, ag_img, ag_txt)

    with step(f"5/9 thí nghiệm chính: {_N_ABL} ablation × {_N_SEED} seed = {_N_ABL * _N_SEED} lần huấn luyện"):
        _PROG.update(abl=None, i_abl=0, i_seed=0)
        ag_all_res, ag_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                            ablations=CFG.ablations)
    ag_df = results_table(ag_all_res)
    import pickle
    with open(R / "per_query_all_ablations.pkl", "wb") as f:      # cho kiểm định cặp về sau
        pickle.dump(dict(PER_QUERY), f)
    ag_df.to_csv(R / "main_results.csv", index=False)
    display(ag_df)
    PQ_AG = dict(PER_QUERY) if "PER_QUERY" in globals() else {}   # cho cell kiểm định cặp

    ag_piv = pd.DataFrame(ag_art["grid"]).pivot_table(index="alpha", columns="lam", values="MRR")
    ag_piv.to_csv(R / "alpha_sensitivity.csv")
    print("\nMRR trên VALIDATION theo (α, λ):\n", ag_piv.round(4).to_string())

    with step("6/9 phân tích 5 thành công / 5 thất bại"):
        ag_cases = analyse_cases(ag_art, ag_records)
    for c in ag_cases:
        print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → "
              f"CLIP+Graph {c['rank_graph']}\n  truy vấn: {c['query']}")
        for e in c["explanation"]:
            print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    with open(R / "case_analysis.json", "w") as f:
        json.dump(ag_cases, f, ensure_ascii=False, indent=2)

    with step("7/9 fidelity trên điểm đường đi"):
        ag_fdf, ag_summ = fidelity_study(ag_art)
    ag_fdf.to_csv(R / "fidelity_per_query.csv", index=False)
    with open(R / "xai_summary.json", "w") as f:
        json.dump(ag_summ, f, indent=2)
    print("\nxAI (điểm đường đi):", {k: round(v, 4) if isinstance(v, float) else v
                                     for k, v in ag_summ.items()})

    if "final_rank_fidelity" in globals():
        with step("8/9 fidelity trên thứ hạng điểm cuối"):
            ag_rdf, ag_rsumm = final_rank_fidelity(ag_art)
        ag_rdf.to_csv(R / "fidelity_final_rank.csv", index=False)
        with open(R / "xai_final_rank_summary.json", "w") as f:
            json.dump(ag_rsumm, f, indent=2)
        print("xAI (thứ hạng điểm cuối):", {k: round(v, 3) if isinstance(v, float) else v
                                           for k, v in ag_rsumm.items()})

    with step("9/9 xuất bộ truy vấn đánh giá + demo"):
        export_eval_queries(ag_art["engine"].queries, ag_art["graph"],
                            R / "eval_queries_20.json", n=20, allowed_ids=ag_split["test"])
        demo_query("a person drinking from a glass while sitting on the couch", ag_art, k=3)
        demo_query("someone watching tv and holding a phone", ag_art, k=3)

# ------------------- Bảng tổng hợp cấp độ 2: hai bộ dữ liệu -------------------
if "df" in globals():
    both = pd.concat([df.assign(dataset="flickr8k"), ag_df.assign(dataset="action_genome")])
    both = both[both.ablation == "none"][["dataset", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]]
    both.to_csv(Path("/kaggle/working/results") / "level2_two_datasets.csv", index=False)
    display(both)
log("XONG toàn bộ cell Action Genome")

Predicate có trong dữ liệu nhưng KHÔNG dùng: {'behind': 9032, 'beneath': 7499, 'in_front_of': 28163, 'on_the_side_of': 19271, 'in': 3362, 'above': 2225, 'covered_by': 670}
Alias danh từ sinh từ dữ liệu: {'couch': 'sofa', 'glass': 'cup', 'bottle': 'cup', 'notebook': 'paper', 'cabinet': 'closet', 'camera': 'phone'}
Động từ phổ biến trên TRAIN chưa được ánh xạ: [('walk', 1962), ('take', 1224), ('put', 1073), ('pick', 828), ('open', 813), ('shoe', 467), ('clothe', 455), ('close', 409), ('get', 364), ('sneeze', 363), ('throw', 361), ('turn', 356), ('stave', 288), ('begin', 281), ('place', 257), ('start', 254), ('play', 253), ('leave', 249), ('laugh', 221), ('pour', 215)]
IsA hành động: 219 node ghép → 14 khái niệm động từ
[00:00 | +  0.0s] Bắt đầu Action Genome | ablations=('none', 'random_edges', 'no_relations', 'no_scene', 'no_related', 'no_concept') | seeds=[0, 1, 2] | epochs=20 | device=cuda
[00:00 | +  0.0s] ▶ 1/9 nạp annotation AG
Action Genome: 9587 video | 11.3 node/video | 7.9 quan

CLIP frame:   0%|          | 0/299 [00:00<?, ?batch/s]

[14:18 | +851.9s] Văn bản AG — VAL MRR CLIP: cắt cụt 0.1741 | theo câu 0.1612 → dùng cắt cụt
[14:18 | +  0.0s] CLIP xong: ảnh (9587, 768), văn bản (9587, 768) → cache clip_ag_ViT-L-14_laion2b_s32b_b82k_9587.npz
[14:21 | +  2.8s] ✔ 3/9 đặc trưng CLIP (dùng cache nếu có) (860.4s)
[14:21 | +  0.0s] ▶ 4/9 self-test
[14:21 | +  0.0s] ▶ self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
WordNet(+fallback): 208/214 lớp đối tượng có khái niệm cha
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Split AG (sha1=1f1f4e811772): train=6611 val=1166 test=1810

  ✓  Cạnh scene + đồng xuất hiện chỉ xây từ train
  ✓  Điểm đường đi khớp giải thích liệt kê (296/296 cặp)
  ✓  Split tái lập được từ seed
[14:43 | + 22.1s] ✔ self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu (22.1s)
[14:43 | +  0.0s] ✔ 4/9 self-test (22.1s)
[14:43 | +  0.0s] ▶ 4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'non

,learn_clip_w,val_MRR,alpha,lam,clip_w,path_w
0,False,0.454031,0.2,0.5,0.069,1.0
1,True,0.455391,0.2,0.5,0.069,1.0


→ CFG.learn_clip_w = True (chọn theo VAL MRR)
[27:11 | +  0.0s] ✔ 4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0) (748.3s)
[27:11 | +  0.0s] ▶ 5/9 thí nghiệm chính: 6 ablation × 3 seed = 18 lần huấn luyện

ABLATION = none
[27:12 | +  0.0s] ▶ [ablation 1/6 'none' | seed 1/3] dựng đồ thị + bộ truy vấn
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 346, 'indirect': 457}
[27:27 | + 15.7s] ✔ [ablation 1/6 'none' | seed 1/3] dựng đồ thị + bộ truy vấn (15.7s)
[27:27 | +  0.0s]     nodes=9847 | scene=167 related=1615 isa=221 | đường đi P1..P5=640036
Đồ thị: {'nodes': 9847, 'images': 9587, 'object_classes': 202, 'concepts': 58, 'edges': 110163, 'contains': 108160, 'scene': 167, 'related': 1615, 'isa': 221, 'paths': {'P0_contains': 108160, 'P1_scene': 91495, 'P2_isa': 88992, 'P3_related': 216593, 'P4_scene_isa': 75966, 'P5_related_isa': 166990}, 'ablation': 'none'}
[27:27 | +  0.0s] ▶   huấn luyện GNN seed=0 (20 epoch, 6611 t

,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000
1,none,CLIP+Graph,0.2827 ± 0.0032,0.4967 ± 0.0055,0.5737 ± 0.0053,0.3775 ± 0.0032,0.4208 ± 0.0040
2,none,CLIP [direct] (n=1626),0.1064 ± 0.0000,0.2380 ± 0.0000,0.3155 ± 0.0000,0.1730 ± 0.0000,0.1977 ± 0.0000
3,none,CLIP+Graph [direct] (n=1626),0.2985 ± 0.0020,0.5168 ± 0.0054,0.5910 ± 0.0053,0.3945 ± 0.0029,0.4385 ± 0.0037
4,none,CLIP [indirect] (n=100),0.0800 ± 0.0000,0.1900 ± 0.0000,0.2400 ± 0.0000,0.1433 ± 0.0000,0.1570 ± 0.0000
5,none,CLIP+Graph [indirect] (n=100),0.1267 ± 0.0205,0.2733 ± 0.0047,0.3833 ± 0.0125,0.2042 ± 0.0099,0.2375 ± 0.0106
6,none,CLIP+Graph (adaptive),0.2891 ± 0.0019,0.4994 ± 0.0054,0.5770 ± 0.0026,0.3839 ± 0.0022,0.4265 ± 0.0024
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3040 ± 0.0015,0.5166 ± 0.0053,0.5918 ± 0.0054,0.3987 ± 0.0025,0.4419 ± 0.0035
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1467 ± 0.0094,0.3133 ± 0.0047,0.3867 ± 0.0236,0.2295 ± 0.0027,0.2580 ± 0.0077
9,random_edges,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000



MRR trên VALIDATION theo (α, λ):
 lam      0.00    0.25    0.50    0.75
alpha                                
0.0    0.2916  0.4173  0.4366  0.3530
0.2    0.3385  0.4208  0.4538  0.4223
0.4    0.2784  0.3108  0.3402  0.3424
0.5    0.2582  0.2749  0.2914  0.2985
0.6    0.2367  0.2454  0.2555  0.2578
0.7    0.2161  0.2212  0.2267  0.2286
0.8    0.2013  0.2032  0.2057  0.2095
0.9    0.1862  0.1877  0.1881  0.1881
1.0    0.1728  0.1728  0.1728  0.1728
[99:24 | +  0.0s] ▶ 6/9 phân tích 5 thành công / 5 thất bại
[99:24 | +  0.0s] ✔ 6/9 phân tích 5 thành công / 5 thất bại (0.0s)

[THÀNH CÔNG] nhóm=direct | hạng CLIP 99 → CLIP+Graph 1
  truy vấn: A person opens the door to a room, walks in, and closes the door. The person grabs a duster and a pillow. The 
    • ảnh –chứa→ pillow   [+0.8351]
    • ảnh –chứa→ door   [+0.5743]
    • ảnh –chứa→ look broom –liên_quan→ hold broom –là_một→ hold   [+0.0129]
    • ảnh –chứa→ broom –liên_quan→ hold broom –là_một→ hold   [+0.0128]
    • ảnh –chứa→ broom

,dataset,model,R@1,R@5,R@10,MRR,nDCG@10
0,flickr8k,CLIP,0.7060 ± 0.0000,0.9130 ± 0.0000,0.9690 ± 0.0000,0.7996 ± 0.0000,0.8394 ± 0.0000
1,flickr8k,CLIP+Graph,0.7170 ± 0.0033,0.9160 ± 0.0022,0.9690 ± 0.0016,0.8056 ± 0.0030,0.8438 ± 0.0026
2,flickr8k,CLIP [direct] (n=745),0.7248 ± 0.0000,0.9195 ± 0.0000,0.9664 ± 0.0000,0.8140 ± 0.0000,0.8498 ± 0.0000
3,flickr8k,CLIP+Graph [direct] (n=745),0.7432 ± 0.0033,0.9235 ± 0.0019,0.9718 ± 0.0011,0.8247 ± 0.0030,0.8593 ± 0.0025
4,flickr8k,CLIP [indirect] (n=200),0.6450 ± 0.0000,0.8900 ± 0.0000,0.9700 ± 0.0000,0.7517 ± 0.0000,0.8028 ± 0.0000
5,flickr8k,CLIP+Graph [indirect] (n=200),0.6283 ± 0.0062,0.8900 ± 0.0071,0.9550 ± 0.0041,0.7395 ± 0.0049,0.7893 ± 0.0048
6,flickr8k,CLIP+Graph (adaptive),0.7103 ± 0.0079,0.9123 ± 0.0009,0.9647 ± 0.0017,0.8005 ± 0.0052,0.8387 ± 0.0043
7,flickr8k,CLIP+Graph (adaptive) [direct] (n=745),0.7369 ± 0.0097,0.9239 ± 0.0025,0.9691 ± 0.0011,0.8206 ± 0.0063,0.8554 ± 0.0047
8,flickr8k,CLIP+Graph (adaptive) [indirect] (n=200),0.6233 ± 0.0062,0.8717 ± 0.0062,0.9433 ± 0.0085,0.7331 ± 0.0058,0.7811 ± 0.0063
0,action_genome,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000


[99:25 | +  0.0s] XONG toàn bộ cell Action Genome


In [42]:
# Mô hình cuối của Action Genome chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_AG = list(FUSION_LOG)
sel_ag, final_ag = final_model_table(ag_all_res, FUSION_AG)
display(sel_ag.round(4)); display(final_ag)
display(pd.DataFrame(FUSION_AG).round(3))
sig_ad_ag = paired_test(PQ_AG, "none+adaptive", "none")
display(sig_ad_ag.round(4))
R_AG = AG_OUT / "results"
for name, obj in (("final_model_val_selection.csv", sel_ag), ("final_model_test.csv", final_ag),
                  ("fusion_log.csv", pd.DataFrame(FUSION_AG)), ("paired_adaptive_vs_global.csv", sig_ad_ag)):
    obj.to_csv(R_AG / name, index=False)

→ Mô hình cuối (chọn trên VAL): đồ thị='none', kết hợp=adaptive (VAL MRR 0.4618)


,ablation,val_MRR,mode
0,none,0.4618,adaptive
1,none,0.4514,global


,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000
2,none,CLIP [direct] (n=1626),0.1064 ± 0.0000,0.2380 ± 0.0000,0.3155 ± 0.0000,0.1730 ± 0.0000,0.1977 ± 0.0000
4,none,CLIP [indirect] (n=100),0.0800 ± 0.0000,0.1900 ± 0.0000,0.2400 ± 0.0000,0.1433 ± 0.0000,0.1570 ± 0.0000
6,none,CLIP+Graph (adaptive),0.2891 ± 0.0019,0.4994 ± 0.0054,0.5770 ± 0.0026,0.3839 ± 0.0022,0.4265 ± 0.0024
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3040 ± 0.0015,0.5166 ± 0.0053,0.5918 ± 0.0054,0.3987 ± 0.0025,0.4419 ± 0.0035
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1467 ± 0.0094,0.3133 ± 0.0047,0.3867 ± 0.0236,0.2295 ± 0.0027,0.2580 ± 0.0077


,ablation,seed,alpha,lam,cv_global,cv_adaptive,alpha_q_with_concept,alpha_q_no_concept,w_alpha
0,none,0,0.2,0.5,0.454,0.467,0.200,0.232,"{'has_concept': 0.0430000014603138, 'log_n_con..."
1,none,1,0.2,0.5,0.444,0.458,0.200,0.226,"{'has_concept': 0.0560000017285347, 'log_n_con..."
2,none,2,0.2,0.5,0.456,0.460,0.199,0.225,"{'has_concept': 0.06199999898672104, 'log_n_co..."
3,random_edges,0,0.2,0.5,0.446,0.458,0.201,0.229,"{'has_concept': 0.04100000113248825, 'log_n_co..."
4,random_edges,1,0.2,0.5,0.439,0.449,0.200,0.229,"{'has_concept': 0.08100000023841858, 'log_n_co..."
5,random_edges,2,0.2,0.5,0.445,0.457,0.199,0.219,"{'has_concept': 0.07599999755620956, 'log_n_co..."
6,no_relations,0,0.2,0.5,0.457,0.466,0.202,0.216,"{'has_concept': 0.07000000029802322, 'log_n_co..."
7,no_relations,1,0.2,0.5,0.459,0.466,0.200,0.213,"{'has_concept': 0.07100000232458115, 'log_n_co..."
8,no_relations,2,0.2,0.5,0.450,0.462,0.201,0.226,"{'has_concept': 0.03500000014901161, 'log_n_co..."
9,no_scene,0,0.2,0.5,0.443,0.447,0.201,0.233,"{'has_concept': 0.039000000804662704, 'log_n_c..."


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1810,0.3839,0.3775,0.0063,0.0036,0.0091,0.0000,316,233,none+adaptive,none
1,direct,1626,0.3987,0.3945,0.0042,0.0017,0.0067,0.0004,254,202,none+adaptive,none
2,indirect,100,0.2295,0.2042,0.0252,0.0084,0.0448,0.0004,29,16,none+adaptive,none


In [43]:
# =====================================================================================
# LIÊN KẾT TRUY VẤN → NODE BẰNG NGỮ NGHĨA (CLIP-text), KHÔNG DÙNG TỪ ĐIỂN VIẾT TAY
# τ chọn trên VALIDATION. So sánh: dictionary (3 danh sách) | exact (khớp chuỗi) | semantic@τ
# =====================================================================================
import pandas as pd

class SemanticLinker:
    """CLIP-text đóng băng; cache embedding theo chuỗi để mọi lần dựng đồ thị dùng lại."""
    def __init__(self):
        import open_clip
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        self.m = m.to(DEVICE).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.tok = open_clip.get_tokenizer(CFG.clip_model)
        self.cache = {}
        self.dim = self.embed(["x"]).shape[1]

    @torch.no_grad()
    def embed(self, texts):
        new = [t for t in dict.fromkeys(texts) if t not in self.cache]
        for s in range(0, len(new), 1024):
            b = new[s:s + 1024]
            f = F.normalize(self.m.encode_text(self.tok(b).to(DEVICE)), dim=-1).float().cpu().numpy()
            self.cache.update(zip(b, f))
        if not texts:
            return np.zeros((0, self.dim), np.float32)
        return np.stack([self.cache[t] for t in texts])


if "LINKER" not in globals():
    LINKER = SemanticLinker()


def query_ngrams(text):
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    return [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]


def _node_index(graph):
    """Embedding tên node, gắn vào chính đồ thị (mỗi ablation có tập node khác nhau)."""
    if not hasattr(graph, "_sem"):
        names = sorted(graph.name2id)
        graph._sem = (names, np.array([graph.name2id[n] for n in names]), LINKER.embed(names))
    return graph._sem


def make_semantic_parse(tau):
    """Mỗi n-gram nối tới node gần nhất nếu cosine ≥ tau. Trả về (node_id, tên node, n-gram)."""
    def parse(text, graph):
        grams = query_ngrams(text)
        if not grams:
            return []
        names, ids, E = _node_index(graph)
        S = LINKER.embed(grams) @ E.T
        best = S.argmax(1)
        sc = S[np.arange(len(grams)), best]
        hits, seen = [], set()
        for g in np.argsort(-sc):
            if sc[g] < tau:
                break
            nid = int(ids[best[g]])
            if nid not in seen:
                seen.add(nid); hits.append((nid, names[best[g]], grams[g]))
        return hits
    parse.tau = tau
    return parse


# Mã hóa trước mọi n-gram của toàn bộ truy vấn (theo batch) để vòng thí nghiệm chạy nhanh
LINKER.embed(sorted({g for r in ag_records for g in query_ngrams(r["caption"])}))
_AG_COMPOSITES = {o for r in ag_records for o in r["objects"] if " " in o}   # node ghép 'động từ vật'


def val_score(parse_fn, tag, seed=0):
    """Huấn luyện 1 seed với cách liên kết cho trước, chỉ đo trên VAL (không chạm test)."""
    with use_dataset(AG_OUT / "semantic_tune" / tag, parse_query=parse_fn, make_split=ag_make_split):
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(ag_records, ag_split, ag_img, ag_txt, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), _ = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        cand, sc, sg, sp_ = score_queries(model, eng, parts["val"]["q_idx"], parts["val"]["img_idx"])
        m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), parts["val"]["gold_pos"])
        clip_m = retrieval_metrics(cand, sc, parts["val"]["gold_pos"])
        vq = parts["val"]["q_idx"]
        grp = Counter(parts["val"]["groups"])
    return dict(method=tag, val_MRR=m["MRR"], val_R1=m["R@1"], val_MRR_CLIP=clip_m["MRR"],
                alpha=a, lam=l,
                coverage=float(np.mean([len(eng.q_concepts[q]) > 0 for q in vq])),
                rel_coverage=float(np.mean([any(h[1] in _AG_COMPOSITES for h in eng.q_concepts[q])
                                            for q in vq])),
                concepts_per_q=float(np.mean([len(eng.q_concepts[q]) for q in vq])),
                n_direct=grp.get("direct", 0), n_indirect=grp.get("indirect", 0))

### 1) Chọn τ trên VALIDATION

## Evaluation

In [44]:
TAU_GRID = [0.80, 0.84, 0.88, 0.92, 0.96]
rows = [val_score(ag_parse_query, "dictionary"),
        val_score(make_semantic_parse(0.999), "exact")]          # khớp chuỗi thuần, không danh sách
for tau in TAU_GRID:
    rows.append(val_score(make_semantic_parse(tau), f"semantic@{tau:.2f}"))
link_df = pd.DataFrame(rows)
(AG_OUT / "results").mkdir(parents=True, exist_ok=True)
link_df.to_csv(AG_OUT / "results" / "linking_tau_val.csv", index=False)
display(link_df.round(4))

sem_rows = link_df[link_df.method.str.startswith("semantic")]
BEST_TAU = float(sem_rows.loc[sem_rows.val_MRR.idxmax(), "method"].split("@")[1])
best_parse = make_semantic_parse(BEST_TAU)
print(f"τ tốt nhất trên val: {BEST_TAU}")

WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 346, 'indirect': 457}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8167, 'other': 1142, 'indirect': 278}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8552, 'other': 350, 'indirect': 685}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8378, 'other': 538, 'indirect': 671}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8305, 'other': 910, 'indirect': 372}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8206, 'other': 1112, 'indirect': 269}
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8167, 'other': 1142, 'indirect': 278}


,method,val_MRR,val_R1,val_MRR_CLIP,alpha,lam,coverage,rel_coverage,concepts_per_q,n_direct,n_indirect
0,dictionary,0.4567,0.3559,0.1728,0.2,0.5,0.9828,0.4580,4.0780,1077,54
1,exact,0.4471,0.3388,0.1728,0.2,0.5,0.9254,0.0334,2.1304,997,39
2,semantic@0.80,0.4580,0.3533,0.1728,0.2,0.5,0.9880,0.5643,4.9554,1036,87
3,semantic@0.84,0.4426,0.3405,0.1728,0.2,0.5,0.9743,0.3902,3.6518,1018,91
4,semantic@0.88,0.4462,0.3388,0.1728,0.2,0.5,0.9451,0.1852,2.6038,1008,58
5,semantic@0.92,0.4505,0.3465,0.1728,0.2,0.5,0.9280,0.0849,2.1973,1000,39
6,semantic@0.96,0.4586,0.3559,0.1728,0.2,0.5,0.9254,0.0360,2.1329,997,39


τ tốt nhất trên val: 0.96


In [45]:
ART_AG = dict(ART_BY_ABL)
ag_gdf = explanation_grounding(ART_AG, ag_records, ag_split)
ag_gtab = grounding_table(ag_gdf)
ag_gdf.to_csv(AG_OUT / "results" / "explanation_grounding_per_query.csv", index=False)
ag_gtab.to_csv(AG_OUT / "results" / "explanation_grounding.csv", index=False)
display(ag_gtab)

Nhãn tham chiếu: 'objects' của 2976 ảnh VAL+TEST | top_m=5


,ablation,group,n_queries,rel_paths_per_q,edge_ok,path_ok,rand_path_ok,selection_lift,grounded,mean_npmi
0,none,all,1364,28.3,"0.999 [0.997, 1.000]","0.999 [0.997, 1.000]","0.989 [0.986, 0.992]","0.010 [0.008, 0.012]","0.844 [0.827, 0.860]",0.743
1,random_edges,all,1115,3.0,"0.154 [0.141, 0.169]","0.093 [0.081, 0.106]","0.091 [0.079, 0.105]","0.002 [-0.001, 0.004]","0.600 [0.579, 0.622]",-0.467
2,no_scene,all,1335,24.5,"0.998 [0.996, 0.999]","0.998 [0.996, 0.999]","0.987 [0.984, 0.990]","0.010 [0.008, 0.013]","0.845 [0.829, 0.860]",0.701
3,no_related,all,1310,7.2,"1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","0.000 [0.000, 0.000]","0.784 [0.769, 0.799]",0.766
4,no_concept,all,230,3.4,"0.981 [0.962, 0.996]","0.981 [0.962, 0.996]","0.981 [0.962, 0.996]","-0.000 [-0.000, 0.000]","1.000 [1.000, 1.000]",0.361
5,none,direct,1264,29.0,"0.999 [0.997, 1.000]","0.999 [0.997, 1.000]","0.988 [0.985, 0.991]","0.010 [0.008, 0.013]","0.850 [0.833, 0.867]",0.748
6,random_edges,direct,1033,3.1,"0.152 [0.137, 0.167]","0.091 [0.078, 0.105]","0.089 [0.076, 0.102]","0.002 [-0.000, 0.004]","0.599 [0.577, 0.621]",-0.475
7,no_scene,direct,1239,25.0,"0.998 [0.996, 0.999]","0.997 [0.995, 0.999]","0.986 [0.983, 0.989]","0.011 [0.009, 0.014]","0.850 [0.834, 0.865]",0.703
8,no_related,direct,1219,7.4,"1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","0.000 [0.000, 0.000]","0.787 [0.771, 0.802]",0.767
9,no_concept,direct,209,3.4,"0.979 [0.959, 0.995]","0.979 [0.959, 0.995]","0.979 [0.959, 0.995]","-0.000 [-0.000, 0.000]","1.000 [1.000, 1.000]",0.362


In [46]:
# =====================================================================================
# ABLATION ĐÚNG NGHĨA CHO AG: bỏ NODE QUAN HỆ người–vật ('hold cup', 'sit sofa'...).
# Trên AG, 'no_relations' KHÔNG bỏ quan hệ: các bộ ba (người, holding, cup) đã được "biên dịch"
# thành node ghép và nối vào video bằng cạnh contains (P0). 'no_relations' chỉ bỏ cạnh
# vật–vật (afford/related/isa). Muốn hỏi "tri thức quan hệ có giúp không" thì phải bỏ chính
# các node ghép này: đồ thị chỉ còn video –chứa→ lớp đồ vật.
# Chạy NGAY sau cell 54 (cần ag_records, ag_split, ag_img, ag_txt, PQ_AG).
# =====================================================================================
import copy
ag_records_nc = copy.deepcopy(ag_records)
for r in ag_records_nc:
    r["objects"] = [o for o in r["objects"] if " " not in o]     # bỏ node ghép 'động từ vật'
    r["relations"] = []
log(f"no_composite: {np.mean([len(r['objects']) for r in ag_records_nc]):.1f} node/video "
    f"(đầy đủ: {np.mean([len(r['objects']) for r in ag_records]):.1f})")

with use_dataset(AG_OUT / "no_composite", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
    nc_res, _ = run_experiment(ag_records_nc, ag_split, ag_img, ag_txt, ablations=("none",))
    nc_df = results_table(nc_res).assign(ablation="no_composite")
    PQ_NC = {("no_composite", s): v for (ab, s), v in PER_QUERY.items() if ab == "none"}
_N_ABL = len(CFG.ablations)

R_AG = AG_OUT / "results"
ag_full = pd.concat([ag_df, nc_df])
ag_full.to_csv(R_AG / "main_results_with_no_composite.csv", index=False)
display(ag_full[ag_full.model.str.startswith("CLIP+Graph")][["ablation", "model", "R@1", "R@5", "MRR"]])

PQ_ALL = {**PQ_AG, **PQ_NC}           # nhóm direct/indirect lấy theo đồ thị ĐẦY ĐỦ ('none')
sig_ag = pd.concat([paired_test(PQ_ALL, "none", b)
                    for b in ("no_composite", "no_relations", "random_edges", "no_scene",
                              "no_related", "no_concept") if any(k[0] == b for k in PQ_ALL)])
sig_ag.to_csv(R_AG / "paired_ablation_test.csv", index=False)
display(sig_ag.round(4))

[129:47 | +1822.6s] no_composite: 3.4 node/video (đầy đủ: 11.3)

ABLATION = none
[129:47 | +  0.0s] ▶ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn
WordNet(+fallback): 31/35 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 639, 'indirect': 164}
[129:54 | +  6.4s] ✔ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn (6.4s)
[129:54 | +  0.0s]     nodes=9666 | scene=0 related=104 isa=54 | đường đi P1..P5=234426
Đồ thị: {'nodes': 9666, 'images': 9587, 'object_classes': 35, 'concepts': 44, 'edges': 32973, 'contains': 32815, 'scene': 0, 'related': 104, 'isa': 54, 'paths': {'P0_contains': 32815, 'P1_scene': 0, 'P2_isa': 50771, 'P3_related': 66878, 'P4_scene_isa': 0, 'P5_related_isa': 116777}, 'ablation': 'none'}
[129:54 | +  0.0s] ▶   huấn luyện GNN seed=0 (20 epoch, 6611 truy vấn)
  cạnh dùng khi huấn luyện: 43198/65946 (đã bỏ 22748 cạnh chạm ảnh val/test)
  epoch  0 | loss 44.4449 | w_path 1.000 0.169 0.178 0.124 0.092 0.080 | gate L0[scene,rel,is

,ablation,model,R@1,R@5,MRR
1,none,CLIP+Graph,0.2827 ± 0.0032,0.4967 ± 0.0055,0.3775 ± 0.0032
3,none,CLIP+Graph [direct] (n=1626),0.2985 ± 0.0020,0.5168 ± 0.0054,0.3945 ± 0.0029
5,none,CLIP+Graph [indirect] (n=100),0.1267 ± 0.0205,0.2733 ± 0.0047,0.2042 ± 0.0099
6,none,CLIP+Graph (adaptive),0.2891 ± 0.0019,0.4994 ± 0.0054,0.3839 ± 0.0022
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3040 ± 0.0015,0.5166 ± 0.0053,0.3987 ± 0.0025
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1467 ± 0.0094,0.3133 ± 0.0047,0.2295 ± 0.0027
10,random_edges,CLIP+Graph,0.2781 ± 0.0076,0.4913 ± 0.0021,0.3737 ± 0.0051
12,random_edges,CLIP+Graph [direct] (n=1626),0.2983 ± 0.0091,0.5148 ± 0.0018,0.3945 ± 0.0061
14,random_edges,CLIP+Graph [indirect] (n=100),0.0733 ± 0.0125,0.2433 ± 0.0125,0.1550 ± 0.0097
15,random_edges,CLIP+Graph (adaptive),0.2888 ± 0.0063,0.5000 ± 0.0016,0.3829 ± 0.0042


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1810,0.3775,0.3665,0.0110,0.0037,0.0185,0.0011,472,387,none,no_composite
1,direct,1626,0.3945,0.3851,0.0094,0.0016,0.0173,0.0097,398,350,none,no_composite
2,indirect,100,0.2042,0.1788,0.0254,-0.0068,0.0597,0.0604,39,19,none,no_composite
0,all,1810,0.3775,0.3814,-0.0039,-0.0100,0.0021,0.8961,416,411,none,no_relations
1,direct,1626,0.3945,0.3986,-0.0041,-0.0103,0.0021,0.9018,354,366,none,no_relations
2,indirect,100,0.2042,0.2200,-0.0157,-0.0510,0.0187,0.8091,32,24,none,no_relations
0,all,1810,0.3775,0.3737,0.0038,-0.0020,0.0096,0.0994,443,391,none,random_edges
1,direct,1626,0.3945,0.3945,0.0001,-0.0060,0.0061,0.4899,367,354,none,random_edges
2,indirect,100,0.2042,0.1550,0.0493,0.0186,0.0817,0.0007,42,17,none,random_edges
0,all,1810,0.3775,0.3779,-0.0003,-0.0047,0.0040,0.5661,406,372,none,no_scene


### 2) Ví dụ liên kết (đọc tay để kiểm tra chất lượng)

In [ ]:
_g = MultimodalGraph(ag_records, ag_split["train"])
names, ids, E = _node_index(_g)
for iid in ag_split["val"][:8]:
    text = ag_records[iid]["caption"]
    print(f"\n{text[:90]}")
    for nid, node, gram in best_parse(text, _g):
        cos = float(LINKER.embed([gram])[0] @ E[names.index(node)])
        print(f"   '{gram}' → {node}   (cos {cos:.3f})")

### 3) Chạy đầy đủ với τ tốt nhất: 3 ablation × 3 seed

In [ ]:
with use_dataset(AG_OUT / "semantic", parse_query=best_parse, make_split=ag_make_split):
    ag_sem_all_res, ag_sem_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                                ablations=("none", "random_edges", "no_relations"))
    ag_sem_df = results_table(ag_sem_all_res)
    ag_sem_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
    ag_sem_fdf, ag_sem_summ = fidelity_study(ag_sem_art)
    with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
        json.dump(ag_sem_summ, f, indent=2)
    if "final_rank_fidelity" in globals():
        ag_sem_rdf, ag_sem_rsumm = final_rank_fidelity(ag_sem_art)
        ag_sem_rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
    demo_query("a person drinking from a glass while sitting on the couch", ag_sem_art, k=3)

### 4) So sánh dictionary vs semantic trên TEST (cùng split)

In [49]:
cmp_ = pd.concat([ag_df.assign(linking="dictionary"),
                  ag_sem_df.assign(linking=f"semantic@{BEST_TAU}")])
cmp_.to_csv(AG_OUT / "results" / "linking_comparison_test.csv", index=False)
display(cmp_[cmp_.ablation == "none"][["linking", "model", "R@1", "R@5", "MRR", "nDCG@10"]])

,linking,model,R@1,R@5,MRR,nDCG@10
0,dictionary,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000
1,dictionary,CLIP+Graph,0.2827 ± 0.0032,0.4967 ± 0.0055,0.3775 ± 0.0032,0.4208 ± 0.0040
2,dictionary,CLIP [direct] (n=1626),0.1064 ± 0.0000,0.2380 ± 0.0000,0.1730 ± 0.0000,0.1977 ± 0.0000
3,dictionary,CLIP+Graph [direct] (n=1626),0.2985 ± 0.0020,0.5168 ± 0.0054,0.3945 ± 0.0029,0.4385 ± 0.0037
4,dictionary,CLIP [indirect] (n=100),0.0800 ± 0.0000,0.1900 ± 0.0000,0.1433 ± 0.0000,0.1570 ± 0.0000
5,dictionary,CLIP+Graph [indirect] (n=100),0.1267 ± 0.0205,0.2733 ± 0.0047,0.2042 ± 0.0099,0.2375 ± 0.0106
6,dictionary,CLIP+Graph (adaptive),0.2891 ± 0.0019,0.4994 ± 0.0054,0.3839 ± 0.0022,0.4265 ± 0.0024
7,dictionary,CLIP+Graph (adaptive) [direct] (n=1626),0.3040 ± 0.0015,0.5166 ± 0.0053,0.3987 ± 0.0025,0.4419 ± 0.0035
8,dictionary,CLIP+Graph (adaptive) [indirect] (n=100),0.1467 ± 0.0094,0.3133 ± 0.0047,0.2295 ± 0.0027,0.2580 ± 0.0077
0,semantic@0.96,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000


In [ ]:
# Nhiệm vụ 2 — GNN của SCENIR trên dataset thứ hai (Action Genome)
with use_dataset(AG_OUT / "scenir", parse_query=ag_parse_query, make_split=ag_make_split), use_gnn("scenir"):
    ag_sc_res, ag_sc_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                          ablations=("none", "random_edges", "no_relations"))
    ag_sc_df = results_table(ag_sc_res)
    ag_sc_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)

ag_cmp_gnn = pd.concat([ag_df.assign(gnn=CFG.gnn_type), ag_sc_df.assign(gnn="scenir")])
ag_cmp_gnn.to_csv(AG_OUT / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(ag_cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

In [ ]:
!mkdir -p /content/drive/MyDrive/DLDT/results
!cp -R /kaggle/working/results /kaggle/working/output /kaggle/working/split.json /content/drive/MyDrive/DLDT/results/
!ls -R /content/drive/MyDrive/DLDT/results | head -40

In [ ]:
# from google.colab import runtime
# runtime.unassign()

In [52]:
from google.colab import runtime
runtime.unassign()